<!-- dd:dd-lesson-np-1 -->

# Arrays from the ground up

*Numpy · `np-1`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "np-1"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrtPV1z3Dhyf2XKLxYrM1P4JnmVy2Pylqd7k1lbWnt2V1l55EjyntdX+e/BNwGSAEkApOTktmplaYYz3Wg0Gv3d/3iHEHn3"
    "l8M/3v1y5f+8e358+OPy7nh49/Hu+fLMX7n9x7vny8vXLz99fPx0EU/cf/7y+PRyeHl8+vjb4e758PLh+u3w18PtLTwe0PGA"
    "u+PhlhwP9HhgXae+6eFBfPDm/vn++vxyd/14uZFgbr5Vx8PL+W+X6/PjE//VvHi+frr/7Pz58vhw//xyU1Xi2y7fvlw+vlw+"
    "/cR/eZJf+7enrxcJehqF6t3/HA+LF1EfDyf+FYB/enfsPdjr0JZLPkG58N3x9oGvQ/zU7o6uALkOSb4jQKzO/CvYbHe0p5Hg"
    "C+n4Ut5RjPJOsbsahZ1kyEaAao8HCASsKRxHjy0i7RQ4c3qDgOwDySAcIkJNxMjCpp+2FMfFKV4HUKlTF3zCgW884dSvRGca"
    "+E7+jqUNKc+NkuH5LuDjgYsZI+77V6aREicoiyODIGkI5L/fPTxf8k+Be6FttTgP0Bwp9brkDpMG7aM1GEk5LR/5R9k6Wd6u"
    "/Hq47us9OvJ/t18QkBoDUKJpDRzS3x31Nqd1rIqFcMFVOhsLEIrmVgOZBEMWb+YUmOgKxGJ7ajb/pGZBarblqTkh1yVC/Bep"
    "y4QwS74z9fYJaNPfDAtdUYGLOfnb+0siTBK1TwS8KtczxTM7cD2pcm9c/h0R7eGGqpOlyQr3IevxoATHhnrGABwWO7aJAqU2"
    "Uy8JRpekdZmCHpD+LH64/ilf9UXLxBV9PPwZNf/EzwwtpEfkzLUEdpYmTS1+b84gCSG5P+sxknLEIDO2oWeBS7BJ5OAmiw/Y"
    "0iYFAbN8fULLG96GWNNovJefPd9fXxh5n3MjnekskF8eHu9eMEoGA5atQVMS70zJHLE2R8AsGUbOYIENRgnZmWD8JAGxbv4D"
    "cxSTueJ4OIU0oVtwlm9nfL3al6Cbon/f0pFuQscYe2hJmq6pz6qURlql6yt1kAs19o6awjbjREdt2OYUGyCC7yJgsg60uIGE"
    "VhLnTE8nATtFZR7uPv/86e7w9JfDzROX19YjfTw8SVXq5c8vlxvxa3W459/79cvDRbyFuqqyru4Al8RVTXWTJ2sTiqLjW3zj"
    "BSkBqGCL34Wg0ktCdknrVZRw3GnT5Xhg+SqgkBcZGxOJQ226Dh+usi5gzkLAedd9kOAU/eGQi7Q8gEXkgXDbHA/1mqUtwN98"
    "61p3KSiMB1iNwUmYoYWxUN+5FhMo7gZzQQi5QgujNQHAucJJg4swmJH4y13Rni2wWCVywCk9azVAY9usBQkGO7MS2Er+JFI6"
    "i9AqTKcp6XeZllEr5l2oy5HF+vpUN2izTlA7Hsp1EKGUWUINwWshOt5CuXIV6RE/WMrC9V21Mu4j+JB/UFwcSSsXtEY9X7At"
    "XGDJDDs6NSlKorZVu5zTOnohzRk31Cb2xwJssRva+CsfPtQKhTzabSROxE/ADT1WVWn/lD4j/J8qy0KPhbjk4a96KjZbmdBB"
    "e/Mmg69W+46GTLXiRE9tXNg3EVqUJvMbiifeuNdeJvEH+RzW2o6B1vcfywbdRuCokyQuODjYClo+ZEhCJ46gbbj6Px+vl40y"
    "qdK+NIqnpjt8K8lZqwNcy0C34tJQCYlt15WKJhXJoRrGkHZIErsZexMo3SBycG6FoiMvnHrhHeAcrl3unHx4J2lFg9kr1UI8"
    "yaTRnvB055BNSc16cte1kr2DKrtGf1unwVK2V5ZFKEOwVLx9sYBstHbCNpfNvnzs1sTW6esmJnJrucpKczOpKGHdqMpOo5Nm"
    "UhOGQQwMTdLy+i+NG1Fb2hRYQsg7H2E1zF+ApB+Dr5vnhvPy3NbInrxE0jaW3glL5NDNpaZqfkevK0J0qNOku5XIUIyIK2ln"
    "3UCVyZcKKy5PtOvQ5NbpY8Ha/yNJzzFJ8zZPw6zJgRxHL4PwNTdKPEq6XeSXeFhW6ZTfNP7OpjsmUbfZSfRV6wlu1ZaJAyKI"
    "qTBn3cYEUBRQQEkGMBGSvlXFifH6RekpMGekftU7XhF4nzOCZC46+xHPCFKSREu1173llalkMuh30c40yEy1op37fqhv+S3v"
    "p556qFcoIH7VHQ2JvQKa3JpdvrW3mJaFg2KN8vsu39tp44difsQE9eae6q0qePWi4CaVu3TTeuD50+EmobP6dUWvOZ97WcX2"
    "FGZZHFETSyosWxsF6si5lkGNtw8LAbmwmJVStMhMVZKGCgZZ3v5JlVFVXgdg8Kfq0mwp1wnCpYO030/yRm/QnD49Be+a8Qc3"
    "u031I1v6J4a+dv8jRqlib0BNzoxmtDsGk7eOysw4MseBWwbrf+7gD76DzT938IfeQYpfV4oWS1xZrPOWg7gmRQW/lXOyNz/n"
    "wlsQFe7pPaA6yq4CeXr8+/OKzk7i8SUkCamTS7OgPbRiGeqrEHIUytWIeCmiTvu8DIyo3HaZ+8akhaLa9q1EDCjDyd87Ud6V"
    "hRvQ2wd7bxbT+1D3KipCdS4DXvkiiFc69fzypKujrtX5kyjUsx0Zr7MtGEepQ6pC3fuxmMYCN7AlbqswgRtTaRUy7b5bFvyx"
    "Cmm8G9K9EVe+I5bSgeozCEbIBBLmh20fY19IjfQoKQPONNiA0yRbpkJQ+UEnGF6aeE8VDzv/poLT6fzhBSFdg2mbAajKIC37"
    "ICpx+R4PHx8fhCQX2xrgz/65IaO67yzj2J8fHx8Eu94OlQqxyNFrq64juxQRjdh7KYmoktdAVVN69G8ytelrMk4BhhFButdZ"
    "gs/1g796KV6+Xx4OVsXr9qA2DVplRacaGDBWfV8EQsi9fVsKAJ0hVZBgqli2SCuGl/OLbA4uuleIK8pRj/mf0m0sufOvLwI8"
    "VnGMCXb+NuRip6v48fBtISNLEO+PfT9Z02pc9pWVXd4nkVxThNqvWBZRnGzi7WarmlCmJPoO+BT0R7b6xsswUsbNkdf/9PLF"
    "R2fNsu6e7q6/cuWFeBzHSHV+ujz/dvfFKIhkcxZUJTNeT+OeDcd/dcfD8mc7883Ttq3IE1N9FoDstaCiKCKEL3RIcalLpQ3y"
    "hyG3saEwrqWhjUQBjPhOofmpRqkpPCW7bTlnXrIuIzucD010cNYElYj091TxFAxHDNid8CWdkDTB5ipQySHZ7McTXqmXgbMH"
    "A08d54dwXM+rMSx0Lw0kpJRQEQPppjeQRiVOBaihNRbjIQwQQj1l4UtZ1XtzmhLenIEVYwzq++ef5Grvr7/+9OWRU/5mhbmt"
    "hKbuEuFeDvJ3OHzBvNrlOFhKI74OGbQDFV0CrsOO7rXHkd2eeGv8fjd2J8HwW2nEILsSY5oO0ySYXr17XxRPB0PxZnwjrnOy"
    "CNN8RrPwFssKlS2anqgJY6jkp2mSZSstxiQcX9lAve86lHs7/OHpUFjGPJAMgojeSF314frL/cMDf+g0JYr/OB7E267m9Mfc"
    "MTJ2EFL/UK2JTMBeeN79Jbg4g1Ioq+FFiRgR2WLWItWWQopoBUp9/yrUtMFSO9tLS6Elmg9Ttb1UZnqeTJyuNx1UA3/VKjaF"
    "pMYm6fFHBdFHA+u9V8bomw+tQStGzI+3E9CCP26cD+27Z0t/rFpE/apr6Dt4ZPdlvvJzrOIE6Dwl4sX7cV1PPrBQ3UPhmxyF"
    "b3IUvsnRKmVWrVVclDuslSagJq/zM90cN9U45qS622vK6r/sO85r9p3VC5IR1e1pPcU3q3GVvWjR9sQXQKIW4MQDPuv7D0wc"
    "C/tAyAgUDzh5LhAXkCP3n4QTUXgsyTQN+QNzVJSPLKNjwLYtkEphl3I8gD1WAlOQo/shV4aa9HX5Yj3C+DjtMtsEYTjKrCk/"
    "gEb1pQgY2YFwztTLY/9qasINjKNTChIOV175Cx+uePR3IYQiVVO3Iq58G/Jhs/JTdgYRh1FGcqzEB/aziOWvnYxfFY4+2NbS"
    "kWby+oxL/MvHHupx/DtMFV3qVzgaNBFWDaEwkfmEQ9xEd/SPuvOrS8WocBScA9FzQ5SDD+N8UAQGmVujv0x/pcPY8yRaSgnE"
    "uGwmiTeeS82mqSbu03WxYjUZCduvhiaJYgJWSuSblsGSKrToajy0N5HpHNFsLCQtBgSbcKF7pJP2cSL9To4ik42/sTttLuw6"
    "1dPHDNmJI7AMdmb60iiH1vfR1NnDDL57nOEnwnjrkHN+br5YzA+/PD4dvhzur1ov/R6+tlAvK5SnGdtEK6JrJLTLebld7eHt"
    "J/CgUngbpOWUG4mx0WCAbnixBteRJtSVwhPLL4SjHslp2I1KsEtQcjhIBNsXYOKeQ9Rv+ihlK5+YeutrqXRSkw0IVdaU7Cor"
    "1+FlUo3yrJwcLMetygpfhd6s5erD9fdhIokd9/p7L4SC7djUTKMUISipouHDNPhtGmTbO/X1Fm/jZAoFkIaCLZDAcGt9SV7E"
    "akoyx/n55e7p5Xh4fnn8cjz8cffw9aJ8VGTkY7WLGX9klsAT3hOLRIomwz8cwh3oiHUx5JWKMP0zCXkSRl5V19TFcB/rYokn"
    "PMIsStZhP8CQgbMaW5um3QZQxFIyo63I6v0wJ7mmu2loN+rWMzfcWT5m/YbfZ12GypWl1RyqXS9E32N4A9UsG2FkEIa9XoYK"
    "62UlqAqrTJ1nqJ0VRhE6dERb6GPZGBLVXFqqZ7VRtpA8zxD0hRqtZgKiWwIYlasuXMcy6cz8cP34+BDUffh7M43d9JCslJT6"
    "KD5gLT762Gfgo1syJFOk6daMJBveAB4jonQs+o6UYj6CtblJUYkOkbyuLl+cKP2E/+C7uJnkQ64j4fusI+EEjQ1ygiab66Qm"
    "zjXqt954mcj7Mg/2Rk6ScKDeGgOxz9QlDkvZ/HWkIVx7CIvZfAURbiV+rcSv9eRWctqdf02QbsRTqCRPITV+Ou2eAB5uwqEF"
    "SnID/zafjvKFnpHnGN3eGrBpN1XdJgxl/5qU9uLgpvRsyD31uGnswQbYI3PVG5+gLlOSk9sL4o62xN34OXVzXFmXV0R/VrjT"
    "bXDPV6ZDyuCEY6YQ2lYzbJS3QpeqURM9VocfGfdd7TnjEMo+49psvFw/KUnbBLI3++eGaZzuOysLSIfya/FueViL0dtoT7QF"
    "wJO8lZQKYk56EvY03MNhE+RpIpFrseZdqSwgMpXXryiehjkI9/nYBG+gi0rUCSWoeGHl7dPj1+snUXbAVGTgjz4y4ORUdyGv"
    "j8p/lD/qxU6qUdZ+LharIJOi6xcrl55hJn40XWrzq1xEhOhQ2GCLErV41RI58aPtUqtHczHE8j+Jkviv7jsgkfyLRzgqhY4s"
    "M2OVR50uRb7/9IJVAE1YnQtFrZuT6ngCFr8RFVmg2qlPxW9Mufep7aHFf2vEb434rRW/tfKbwXKftL9whRzacOHecVfx9xRM"
    "sfYWb4Vokn/fw1CfG7QhKXvJkYSh8a5vhaHjuKco+4T+dv/rb8fDw+PftZ079j/ZJ+ZCDdqe0IH+haRz4ePjVFh0MXycAhMM"
    "/Q4rgQLdAAOlABeKf50B3MT5RSjf9kMuoYpMBMiuiyJNOZJodLGtAbwSDk5f4PpFQZgODQL1HyQtgJBBKZ2QfhG1DUAYISmt"
    "sH4Qtgi0qGmlnCT6SdoyQBjp94bpb8ANgQ2tW/laK/9rGlC3oJaKS+N9qXyqUU/VDNIGEtFsZyU5WAZzTS18ao1TyxutBfaS"
    "FGGwScZLYi8q8dw3/cC6skl7uSYl3w5XAfbEH5TCWhiStbLY0Z4LOBHbiFXJJJS7IhOaKNJ9LnVTplJk09o9jQ4JkNcn2JXN"
    "OLizBnsOdhVkpN2w09a5lTk5reoWVYVzvaTjqcsunkC6prUOA0NFAAnBU+WpZ7GvPyn1oo4tBDjKEIK0QF2n5K+fOK8pFfnD"
    "O8nyH96FSuCdT4yq4b33VjaDTAsZTi1Ad5rbbQlOZ7uQPElcDtT7seNieh9gKtLE7sGObBSV6akrwZr8kO1Jfsic49AXjNLy"
    "TXz57WDut2D6ha4BLFLvJqr3zGFZUOw3OF4ZhajvdV1cuKKuUOUnUSTlWzhD0iFV5Se0jQvLBL2/X54en29uZNIIqKoxst+n"
    "cXzPLY7Dz3++XJ7frwo8+vDi3UXjCDRpCJhYc0ydjANGJAXuRBFntwZoBrFNs9xhN93l0Lm5aOFrQbNBF1YH3SokAUUIlBTo"
    "ZmpM0wCYJg+MWolvskBWhQeN8rNQsjw4uC7ozFdt6jIipK/Rhta1QqmK51Qfrj/fX58D7TS/Hw/iXX0ljrspOBkG6rmZ69GM"
    "9lGhaqSLXdNys7S3pXXXgPZdQzrqiDpYkx2xduqPU3EfBgudlcAdV0LS+UbyPpQ/G3UO3O2AYGcuctMubQm4G7uHbVlBAA//"
    "ckD/xcUr/5f8V6niRNh79VQPsTUm02CPOGKgGGJ9IXUiOqKrlsDoTDnR0OF0wPxfgSMshuOpjwZbdIU/BK/tjONjLtGWyJ74"
    "jpdClg7L008oGUMoySi8SBxFWJYZT27jgZNb780o2r5J/VxR+tAJ3nsgtmgNr6ZXhFEyj4TReaVmLXrMQYRIpZu3zOwciG6a"
    "37yl/EzSeePUm1iBMvoxzZr5npOGA2tyfApxJwZQtnayzk/NcmIwgnEE0m8o2cyuMjb+LOG5ZZ1p+biwYsyEMiCNxOHsDkCU"
    "uay6inuJaruLdMvIibnJ+e1d0yXXwGl8DxSRacrQiOtFxYHKZLRTG46yuI9sAP9kR5zEe9aElq85hG3JIbWqqa/ltJb35p4P"
    "+hoHo2vKKAXSL71AKjryvTyDGv/4nHfXaapmG6aVcPV68ceuWnDnTepIdj4BK55kvTD5ZKrKRw11VjWkqdnVi8F3W6dQGRVA"
    "NQ/qUifFrKSnjmTqgfde9Zsq5BQpFGbml3xV5F7NzPwSzssuOe18/QpmJpcJrP25ZSY4jbP5+c4vZ1JNJUR5En9dT7+NdF+4"
    "Ox5ULdOgpKl/fUk1lqlsGumrt343NdmJiNn8mEY9Iy8rlRcO5VCSxZaut/Zmaulo+6Ujf+nWO4PNgu3yhbiwA5rTFsn8RcLt"
    "9xeu2d+0RcHRomxX3O0WBU23CzVmLw1zREZMF28pUYTpdCxruB/WOeg1TbIDwhtVwzA4a2qmovxbucLkh6FmVyi/AMrTCuUX"
    "wFazuEpDV7BVtjs2dFQ3dVuoM5StjZXtEZ6DbbP8IcHLL/G05j09NigPG3vh6iskDR/S40OyqbPm+p/o7+GMa7RIwXSk3JZ8"
    "Mixh7k4GN0jPs/vrSNaSfDQDM9ykO4tbIlCFJIFGZpXgiRlYJNjw/PZWyRy3BaMVJmCPxjie2H64+/zzp7vD018ON09OuunT"
    "+frp/nNV2dzUiFnpt8iQC086VbrdkWlF2m2HajqSdWGc6gx6hVuDliKVRk7JIQT2bkcfPK+0QOhAmLOqNwEEElZACsJ81+Xw"
    "wtAdKZjuaNp0En5g1i4rM114griBsQTQbnjx3B8WGc4wbAJjGqQcdd9L2Twl1T/DIuMwAoCREuK651RGDiBcFGBx/0oF1qwg"
    "r9cVVy1UTR9k8pG6j3EitPG5XzL42nuy7bpyhyJyEsxDoAQ85UeT4OoQOPlMXWp1lq7a6yRhn+ACApuPnGCvhiL8RqaRKOet"
    "Th8sLpvnwELtDcmHXM+DKrE+26Jylpyw6udD7NYlddRyMljq4t8EpZufLsdDNZDCXVZXUyXuEhBwPl+6ofw6DLriXUmXbwGR"
    "3nPdY9TcWlDxiGpHai0qVKAq6unx78K6Z8fJgIN6f0ufTI+C8K/jbBzmXfnrMSPHyejCUsQmTOKVCKDJjg5rKOPY4XhvO9wY"
    "ZMut7DUXYMDUdi69JFy61ffhqIP5KshtThtaR/iTod99Bq53bNSne/8v2YBVHI14DXlaL9Do/pnIJkY8rEbDCBX7e56LYz0Z"
    "LAG6XGYJaApR8M5xd//sTSqyn0m16BijxV6oeV1aGlXdAtcSLgJU9eUN5hhrl2Sp9Wl+tP3jSbeUxNjzpdH9fWnRkaDT9qJu"
    "P0q0P57qJ1jXVYVNVI0gWoZg5/ZFRVpQlvIIGHTC4YKbqXCNRGn0ajkDTivshpEiZLIDvUbDjhhim8WSIJrjsUBoR2ZLOEpo"
    "nxPTdUbPybO4TerEHI8NUoNkVoOTAGXRQUXQYYrHZtAZmgy9zoXzUYhHqiQKhrHVvDUj+TqjCLm6EMw3tkJTkk54OMdv1CIl"
    "MuOoUcUldeqcHZAIHKQDVE1MaCJYiNTn+b/Js5DS1qvzNCTF0wl+gvabYCrxve8AoO9/xTDYMifXazVha7HoHLpIteyUPTpp"
    "gfYx0B+I7tSAzWFCLN1O6FygUY7tkycDeqaGqmynu+iWAEUAnc89C5n2rEJKV0yKglcsOuHyw4nTvG+2pGTNpLPvS+TZ94Ly"
    "bFD5qLoy2ZS6FOAn3A95Q6mItOdWdqICMAmH1iT8p1HBVBToEecpRPCPtZVocOM6FL6BiGqZNs8l6vJRp76IAJHlL7Nga9oV"
    "qvpQNY5ze2ELREqIKPfKrtbe8IoLCvd5OCHbBBQHpih/XzkZyPtC0UVwCKKINEtGLru8W9+ZU/pqFsX6OukJIEnl0pOSOA1H"
    "N1Ha+d5ECQmAbAwBhhI6j37A9JYlNWlrQqG0P0+T74Q+YI1ojDbWH6mjPzpTpHzVcpGisK7/+CKF0keqH9teLZ/Y2pWqsnJf"
    "WCI35VMFPDI6x1xmaxiy+PioPHRkntEXxTyG3mnqv9voo029gYV90lySZm0qUWTawaYmVE8UW3RJ6PQ9Q0wdSnMG3Q9i9xuV"
    "AtEMk1dpJJpfmrL8cvKqkodo3kz0rf224P6gVlAhqi+5IZQkrgKGNbsSeKqvI6acxPvyFPTouQj5qCwk1d+WyHSo75mbj5DR"
    "/Guq71j32zVf4jL1gJO3plZAPlx/9ocw+g0Z6umzfXc8/LzIQUK9b1pVVeWJlyGaIB0vkIqIvpnpFD6nqP91CVr62/Fa7Lyh"
    "8hN7ie2G9qpJxobiccOOJGIia0AOsdbzeWgqmvbzVrq3haU7tuaYHi3/8Hg8/HYvZ9Eonp8scFJPLZ8pQfP1hDFyws2XgZ3z"
    "1Rku6xMa0E1dZXmoWcWZpqFGBjjpwVA5KBHX+VTW7YDVoDCk2CTFSYat0UzTPaTtvh7SYYvqRMfw2nlTAySMi1w2nkdpLkrY"
    "+yzqZDet8NOm7UDbuqy5Yc46Ngw2NtLlMIxqQZxBlSPnW6TNEA1Z2TzvIqjLuGdVCfaEY6BZ5i8GawbOxau6bK+FoT2OtEkI"
    "1e0xb0aa2nDq2N8Mk60d7cb0men3dEKOxYz8yEhVKGQXbzsFnFJ8uK5PZ9TPFO0veHIcxwWASQfLqTZWc7hK032ud82oV7yG"
    "kHjDxK6ht3xRNNP3uRfoATmKby8LbaNiKPSRfeWvRybc1Oko73yytxlsZvqjSt9cNmJEB9qXxdjt4BZWOIEY64Ad0Je46a4q"
    "B3ueGnGo1+TTyk/IT1fJOvsqgGZ3E8E5irgQFDjgA5pZdJ+nsAqLvrU9qlaAMypTUq6yaJYzV18VBUwlYMWMzVYZXM70Gwc7"
    "NSvU4ChGhk73jzzTRBdbGjxh6CYZgWgyrrYQKlgJ1SRegkG21kJwJJGovT/H+IwSgIuP2/ljGzgDJ12AsfwT6YCJVUiKn1WS"
    "R2geITWscX+0pnxUa6mjWyrKf9YhooQltjiYv9/W5tA8eug8Mlq6Smg+YOY1ATke7p/vr88vd9ePF/HXy/lv8quq+WYgyqBf"
    "tb/fhn7PbTCTOmUGYo4muQ2CuZS77ZXUrTbXJaHmVbY/r8a1FJUbW6U7P5eDUv6bnHij46JeqfPiVC7u3QCrgKredn1tSNGa"
    "fwgG7QH//HKxOTwVZ9iDO+4k3pyA0MSxFV05HBJROOHxZOQcNECVMBmMxCaC7YmONwwXiR94NBA3Bx0GQC/GtkiVoSbzh2sL"
    "L78Jo+vx4ZMKqUxGU+wzUbRR4khhR3VxkaGJyGjNJdXwnurC2Q1xA1mESrwD8JhGOJ9Gms2ardw3ys1osqhXeS+s80dsAqJV"
    "GeM5DpTa8wFSfTfK+Q1DiWiBxZoPSR9D8s1tclZMot6aa9T7rJ3c2ZQeu0fkvDE9cK9EbqzJcYWkBRAyKKexEf0ibRkgjKjx"
    "Zq38r2aQNpAgqHTSqScnv3L08eSk5UL5yjpmkYiHl3hSMoPaxlOGENLT0WnvqymUJ+/W4QCn53cAXgrqaDsCuxEzNIresA0r"
    "A21qp86Nm82fc5Lo9KDLAhEv7LfSBrFA43TbbWz6WZQIwMGZYWLditnj8XxxoENQOlc9PD+v5xSqS8GBO0SRouwevde7z5fn"
    "gT4s1GGhDQs95fnj49PgAS38Ze0ajHgCVCMl5YP9Q+hLctTdH2LU3ZMo4rYOAYkD16olqCA5ENBttbQXQsInBomFHZrGq5Ws"
    "HFjnZP7zlms0FfYu6NSV1YFljcXYhguq195u0zsk7dbAJgErn/bcKdUIAnqbZa8ZLcC3LIR0DunxsFqwu4cpP3FBjY+RrW+O"
    "hxTxLjpaqK+oC2QF+FhEhnbdwq5Exx8yrP4xrvpQrpXql6glOCxssvnSWSbjm23AyXUs5s633Yhhl580m4pIKuh+VEcicHXx"
    "5K48oFgnFhipWctOz+Y+P4xtWBrYM1XrVzFEesi0jk8GLW8Et0qPjXW41vorKqXCEe9gVZFuQ15Ov9ahcTGlVadjx0jfP6UZ"
    "pVchISvtGqBl7SW3qoLa+zbTIH0r5rtbQg0KlmkPEo9GUJJdDbjgzvpfaCQiHlrCFNa7ea+Wtx2W92Fpl81C8EbqpwWiUhuu"
    "vFYxgbnYLDuU93IPRqy13ry8RE1leN3XthdVm1zYc5upLmTWFd3CueDuAgTsF+TU0UK3ZwdNRucE3XYW/ZdZVms3kTytFT/S"
    "SP99ONJs2kzVx+V4+L1fVxdKtG7Vt5c4sgpBXBRBXwamyrG+TvT34cCzbAyTy7UG7twtthf1fae0FYk2EoleHxVvzKYaNCei"
    "qKE5c0u6SozsIXjOse+W4gzX4jz/3YlNrXv5lUxP6oit5DulddbW5NEKOoNmkXPfaat0y9Et0W1Skeso1sB5HPpWeolBOO1w"
    "Di8eJGnNsV9IQynSRqZP7ovUqhhzp8QQIidgR0zdXhi4O9OX2sctW5Eto2VewCM4qSzXlTjyqYRgoRIOjNCXg0LtmXRsNg5J"
    "bx59S54q29ay29E/JSOt3b6OqdJtuZb4opy8GpO1lK272FnQWLrVcDAIpMZCP9lB0E/n++effnl4vHu5v/7605fH+6v0XPz8"
    "+Phwc/N0+Dd+GVdn/j031eHu+unAX/pXvn/qFfPc0/n69fNFPPJXflceuDb3dP58943//W/it/srf9TGlhTg0Jxpg74qIXB/"
    "LnXYxAd6/ziU6IeFZ1MCOc3zf0BKGPTL8AT9sXmCluOJ0SSVH4snJPqTlNDtV4vPVohMU1DvHQ/v5efPklAYvV+cSzwGBsNx"
    "nBstGMoBI+HAiXqvJDAcnK17o94LANPbWpfeVp0HEMSJHqv1R21y2SRYf3lzw8pAOamT0YTANGXAKIc0Dm9jNX0mm9Kbt3TX"
    "RjyVfC7Dh3IjgAAEQQKwEdClOzsFUG91W3qrSVhCVWUGNsf3NiPzsQ59cT1AvZ8omAoKhW+oahj5Zk3xBNvgqOOwwBe3S/hj"
    "aVS4vW3DMPX1qZ9JBQD74UQoDAs56/MeTwWrU810CoWZJxVVSrwRZObzDhPALRQlcanO4DUhRmQ93PscnSmiyUzYU0XAEnVq"
    "IwoU2QIsVmlo4esBjzXkIWDNAcXTpMw01Oj+qznBhW4sceFHdQFlPJWC1o8dmzNQsDxwpeBKSzjC4ETRlYQv5ga/gl3EJB1K"
    "aUTzppGSBIXgLbCOGkn1UspXxEyQ7/HFNYGzo/a4ZRsYSTjotiUtQ5TRpm2aliKGZcfQmosVBOuWUv4GamX8ADQNqWuCUY3q"
    "BuIapXpcSXhwIMen4ZBZyxFBrKawVllOkAKAOTaYEYRq3GVYPssJkQqlDg67FA51TFqEKAGwqRtMJbUhBwkwbBiBjOPQNuJF"
    "RlsEIWspbAkihKjXWNMCTgDQEljXNqeItcUNaxIbP7lsl8RruEENJyVt25ZQwBqionQ5H880fVF4UYuOgq7XXPhoDrJ1NBt4"
    "yEg6/Dl+NQcFiHCUYqhlEHIGhZhvUkNV3SqFnE8RajDfTdBi3Vlw2ZOex6HdICj6H5fr5emOP3NTnT/fXb/ePfz0fLl8ugEz"
    "+ccEtC0DNWE1oxCrqXOoxQRxBqgbWjOmB8lXxa6SMLIslq19gueGi+m24UwIOYKQYYlt22LGDxzHXhQQY5VhBRu+IoQp52XY"
    "UBEeV16CzZcAYrnuY3rrafU7IFbXMU4Qswa5EMb8ImxJAxGAUB59BkCNBCVZDVra0ggvaN5uNrjlfxzZRuO3CwacT5saUX4f"
    "ICCvAoi4AAEthJi1GGF1P7SCQ1p+KXI2Brhp9O2S8fGcRcn59LEWAZhf3IxLOoow4H8pbKdezpLZUT6ouajCUFyoXOMATPWH"
    "r1v+O7+TOX0oQrqEc+GTvsRut3Dwl2LsdStK1m0J3lJ1IpypmXiLQn75IKQUSEZaiAgklAKunMsXufoIuAYA+KawlmuL+dpI"
    "Hd2IRZrttJYSenRFu7dJ37tujBQ7k1zF5iQXWjajGJvq5Rbz2xMzTlLCRXw9YPMagDcrvEdWmqQt5lIQkIYzA6CEZQs6CKMc"
    "LhqWcPsEcs2OcHzUGEDCOZZijgnffG7CKLZvKGOc9RmjXAnUxIec1QEUdId8ZwhlUlNscEu4MsMY4Ie3ld/IBTk/HPwh0vAN"
    "gwVsA7hiC7RIKXstMhLlAyCODUS10DM4ixJpKPKjwjU7wjmW8S2m+ojxAw+x4G6uWfN9QErYMczPF9cOOTVr/tqIr+H/a6Wk"
    "iXI1betWnCqEIf9NIdAQKgwbIliQ1Fiyaku4CAMIczJzuhuxnfHp3Zl66uUsZY9GuUDcAAhwnZpzK7/DpGXIKcRvBv4avxG4"
    "KDBug2VP+jyNNuBpnXYQcvMQ2QwAL65xm86zkF5CFK7CpxluMJ1qEHILnlTe1im5wwCCmkggsgQRLsSd3Si8wUaRINupJAgV"
    "uFRR6CKBnTYiQ/o2BJ6RnQcwesKHZnNmiK4OR8oEz6v+PrqviArcTMassgsgn66/yhqQkA1PUPXheg1XfbFR3RL/xuPhuqAw"
    "TWnvVmcnVlMnS8tX5pAHGndYHHcp1+tSeNYaT1ocT2ke4N4osO5wZJ3gUpvhSi5DpZYDEdYLQsUXJJ2fqPd4lsIZAUQ00mSD"
    "XeAqvNRJGgwRk7pmW0N+18qdgS0G0AYgmnpfV224JGCJp1bWdcCaG59Ce+B8Rbmh15V3JcbqCISZxLjqClpu7WAEiSqobbg+"
    "yDV8xk0brtTCbk/H65TftTh8IkKu4fgYxoxxC48xznGQ1i1TzjsuBjidaiyMRwgbdfwJYdwQohQ0gBMSqxcBxvx/bvNw+4gR"
    "0nNosy+Hbq4iRkRZZbRHHNEeMxTI+MpPqhCEBitolBqkHjyh8ki0lYGAwj0Z+smGTbsvZ5Boowi8uDHHGrgij4NGu4Go9EW0"
    "r8DZgvlh5How3b+svxq8kQjj2/PxhRdCq02DN7BtETcG+QXEZbhqBN0iERchmH+gFnFNpYAwRiiCrCVcJ8F0s+WC6g04BiPC"
    "ro3z1Si8pPQ4AAVipKZMqHm6JSnhFykQ0ZAWUIxV1TDXkkBTEwII3y80jtHDt6H4vUYiU/wGDqc4LYpD7SuHy2U/xS2smOq3"
    "JEVElSNDVgNuiQgThBJlFDYNpVydBpiLHlbXvYRH/+TPCcMxyp/cBGHcMOGGUU3EmVc+Dgi4Gk65wAZ8oxj+P8igTZQ9Uz34"
    "kmVFdJFfzwxjEflte/bE+7LnyhKPW/W2rlewHxs0895AeMZqQOR7mc3MZ2k0Xx4yfMwp0CltMkukRPHHeCdiyCFTM+JtnNMA"
    "X80pxW3ZoZGhIWXDoYB9GxSnvdxwNpuqZXYmBdqqZve1mVaNTiWP7KKBNHJEDZ7XM2HWjDx8vF6eZcI6ruy61GvC/V1tsQZb"
    "+YTXoxscb2tw1w9g9wE8qrYvthlqIWYsPFm/oMufolpuxFGt05yo7nvybMlRIZApAzR1Z8LRsvBWq7A8ZZCWEgGCurBEQE53"
    "KhCWCMzQslrab85f5ky3vm41i+HgmNmJfqDZeCb0s0mVvaqtENxsCYnYTyNbniPSMUTDFm14EmVsun2VxZtrnAJ6338fNG/3"
    "7p5aV6wHIDTd7hRd1zRZdS5lKZj9McijZikrMANDyQVWozUn3UND0hbhB9PRgbYV7QRSJDZcdhFm8htOxO1HC9ofgluDtlCI"
    "S0tqlgHNZvHg3+gK4OQ9LNEPfP/6IeTeAeyP5EpMp8+dwHT6nSpDnLtfFjijcCj2YQr5p6GJRU0NVOucFvJgo4Nju9cSO3GO"
    "hVYW5yriIE8dPmJqiaqgvUqWVOtR6lTyk7TG8lSp9aBNj/CjauBWpYiA1TvgMhbsoUuqazaCe8nfiem6gWkB/MB8i05YUCNX"
    "RDVVzj6OiFNNzGhV3LoSSeduyUASeWMVAySMzSGIoajvLJEKmCLRsZLoPkKtHd+8Dhdgh12qLRGpkD2DordkeM5LdGcozppO"
    "frGbHuqZxXQtLv0HU5CZvOm80zJh4MD1FHPspwwkp3cRrUcHuOTSPIhzefDz3Ys/5S8soEM6wVgAWPE6sUIBMHLk+s9yGJbV"
    "3eth0R7YZUmlTs8urcKT1JPw1DeJ+UdrDnA9kr0yOoUlnbTPl2EZUk0TqanEwPnr9fm/v14u3y8yAdjHto6Ipzlsa3e/oVUH"
    "/ud/AXjqU+k="
)
print("Delta Drills checker ready — 176 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-numpy-ndarray-model -->

## What a tensor is — data + shape + dtype

`numpy.ndarray-model`


<!-- dd:dd-seg-numpy-ndarray-model-0 -->

### a tensor is one block of one type


PyTorch's core object is the **tensor** (n-dimensional array). Every other
thing this course teaches — selecting parts of one, doing arithmetic on whole
ones at a time, rearranging them — is a way of manipulating this one object, so
it pays to know exactly what it is.

A Python list is a bag of pointers: each element can be a different type, live
anywhere in memory, and even be another list of a different length. A tensor is
the opposite: **one block of memory holding elements that are all the same
type**, plus a small amount of metadata describing how to interpret that block.

By convention PyTorch is imported once per file as `import torch as t`. That
short alias is what the ARENA exercises use, so every `t.` below is the same
library you would import as `torch`.


In [ ]:
import torch as t

# A list can hold three different types at once.
print([type(item).__name__ for item in [1, "two", [3]]])

# A tensor holds one, for every element, and says which one.
a = t.tensor([[1, 2, 3], [4, 5, 6]])
print(a)
print("dtype of the whole block:", a.dtype)




That one block is also why some operations are nearly free. Re-describing how
the block should be read costs almost nothing, while producing a block of your
own costs a write of every element. Which operations are which — and how to
ask a tensor the question — is the concept after next, and this page does not
assume any of it.

Why this design? Because when every element is the same type and sits at a
predictable memory address, PyTorch can hand whole-tensor operations to fast
compiled kernels instead of interpreting Python code element by element. That is
the entire performance story — and the reason the idiomatic style you will learn
here avoids writing Python `for` loops over elements.

The general procedure for turning existing Python data into a tensor is
`t.tensor(data)`: it walks the (possibly nested) sequence, finds a common
element type, and copies the values into one block. Nesting of any depth goes
through the same procedure.


In [ ]:
cube = t.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print("shape:", cube.shape)

# A cell ending in a bare expression prints its value, the way a notebook does.
cube * 10




Two ways of reading a tensor back out come up constantly from here on, so name
them now. **`t.equal(x, y)`** answers "same shape AND same values?" as one
bool — the whole-tensor comparison, which is what a check wants. **`x.item()`**
pulls a single element out as a plain Python number, and it refuses unless the
tensor holds exactly one; that refusal is the point, because a silent "first
element" would be a guess.


In [ ]:
same = t.tensor([[1, 2], [3, 4]])
also = t.tensor([[1, 2], [3, 4]])
print("t.equal ->", t.equal(same, also), "  one answer for the whole tensor")

one = t.tensor([7])
print("one.item() ->", one.item(), "as a", type(one.item()).__name__)
try:
    same.item()
except RuntimeError as exc:
    print("four elements ->", type(exc).__name__, "- item() wants exactly one")


The problem below asks you to turn a nested Python list into a 2-D tensor. This
example shows that move once. Start with the list — three inner lists of two
integers each — and hand it to `t.tensor`:


In [ ]:
import torch as t

rows = [[7, 8], [9, 10], [11, 12]]
a = t.tensor(rows)
print(a)




`t.tensor` walked the nesting and copied the values into one block. Nothing was
rounded, reordered, or dropped, and `.tolist()` reads them straight back out —
which is the check worth running whenever you are not sure a conversion did what
you meant:


In [ ]:
print("back to a list:", a.tolist())
assert a.tolist() == [[7, 8], [9, 10], [11, 12]]




You never told `t.tensor` how big the result should be, and you never told it
what type to use. It read both off the data. The next two segments are about
those two answers, because they are where almost every tensor bug lives.


<!-- dd:dd-q224 -->

### Problem 224 · faded — your turn

Turn a nested Python list of equal-length integer rows into a 2-D tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 2, 3],
        [4, 5, 6]])
```


In [ ]:
import torch as t

def solve(rows):
    """Return a 2-D tensor whose i-th row holds rows[i]."""
    return t._____(rows)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(224)


In [ ]:
#@title 💡 Solution — Problem 224
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    return t.tensor(rows)


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


The problem below asks you to turn a nested Python list into a 2-D tensor. This
example shows that move once. Start with the list — three inner lists of two
integers each — and hand it to `t.tensor`:


In [ ]:
import torch as t

rows = [[7, 8], [9, 10], [11, 12]]
a = t.tensor(rows)
print(a)




`t.tensor` walked the nesting and copied the values into one block. Nothing was
rounded, reordered, or dropped, and `.tolist()` reads them straight back out —
which is the check worth running whenever you are not sure a conversion did what
you meant:


In [ ]:
print("back to a list:", a.tolist())
assert a.tolist() == [[7, 8], [9, 10], [11, 12]]




You never told `t.tensor` how big the result should be, and you never told it
what type to use. It read both off the data. The next two segments are about
those two answers, because they are where almost every tensor bug lives.


<!-- dd:dd-q532 -->

### Problem 532 · faded — your turn

Read the tensor's numbers back out as a plain nested Python list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[7, 8], [9, 10]]
```


In [ ]:
import torch as t

def solve(rows):
    """Return the tensor's contents as a plain nested Python list."""
    a = t._____(rows)
    return a._____()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[7, 8], [9, 10]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(532)


In [ ]:
#@title 💡 Solution — Problem 532
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return the tensor's contents as a plain nested Python list."""
    a = t.tensor(rows)
    return a.tolist()


example = [[7, 8], [9, 10]]
print(solve(example))


The problem below asks you to turn a nested Python list into a 2-D tensor. This
example shows that move once. Start with the list — three inner lists of two
integers each — and hand it to `t.tensor`:


In [ ]:
import torch as t

rows = [[7, 8], [9, 10], [11, 12]]
a = t.tensor(rows)
print(a)




`t.tensor` walked the nesting and copied the values into one block. Nothing was
rounded, reordered, or dropped, and `.tolist()` reads them straight back out —
which is the check worth running whenever you are not sure a conversion did what
you meant:


In [ ]:
print("back to a list:", a.tolist())
assert a.tolist() == [[7, 8], [9, 10], [11, 12]]




You never told `t.tensor` how big the result should be, and you never told it
what type to use. It read both off the data. The next two segments are about
those two answers, because they are where almost every tensor bug lives.


<!-- dd:dd-q533 -->

### Problem 533 · faded — your turn

The tensor holds exactly one number. Hand that number back as a plain Python value.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
7
```


In [ ]:
import torch as t

def solve(values):
    """Return the tensor's only element as a plain Python number."""
    a = t._____(values)
    return a._____()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [7]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(533)


In [ ]:
#@title 💡 Solution — Problem 533
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return the tensor's only element as a plain Python number."""
    a = t.tensor(values)
    return a.item()


example = [7]
print(solve(example))


The problem below asks you to turn a nested Python list into a 2-D tensor. This
example shows that move once. Start with the list — three inner lists of two
integers each — and hand it to `t.tensor`:


In [ ]:
import torch as t

rows = [[7, 8], [9, 10], [11, 12]]
a = t.tensor(rows)
print(a)




`t.tensor` walked the nesting and copied the values into one block. Nothing was
rounded, reordered, or dropped, and `.tolist()` reads them straight back out —
which is the check worth running whenever you are not sure a conversion did what
you meant:


In [ ]:
print("back to a list:", a.tolist())
assert a.tolist() == [[7, 8], [9, 10], [11, 12]]




You never told `t.tensor` how big the result should be, and you never told it
what type to use. It read both off the data. The next two segments are about
those two answers, because they are where almost every tensor bug lives.


<!-- dd:dd-q534 -->

### Problem 534 · faded — your turn

One bool for the whole pair: same layout AND same numbers?

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
True
```


In [ ]:
import torch as t

def solve(rows_a, rows_b):
    """Return True when the two tensors match on shape AND values."""
    a = t._____(rows_a)
    b = t._____(rows_b)
    return t._____(a, b)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2], [3, 4]], [[1, 2], [3, 4]])
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(534)


In [ ]:
#@title 💡 Solution — Problem 534
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows_a, rows_b):
    """Return True when the two tensors match on shape AND values."""
    a = t.tensor(rows_a)
    b = t.tensor(rows_b)
    return t.equal(a, b)


example = ([[1, 2], [3, 4]], [[1, 2], [3, 4]])
print(solve(*example))


<!-- dd:dd-seg-numpy-ndarray-model-1 -->

### nesting becomes axes — shape, ndim, numel


The **shape** is a tuple giving the length along each dimension (axis). A 3×4
matrix has `shape == (3, 4)`: axis 0 has length 3 (rows), axis 1 has length 4
(columns). Axis 0 is always the outermost nesting level, so a flat list becomes
1-D, a list of equal-length lists becomes 2-D, and a list of those becomes 3-D.


In [ ]:
import torch as t

grid = t.tensor([[1, 2, 3], [4, 5, 6]])
print("shape:", grid.shape)
print("axis 0 (rows)   :", grid.shape[0])
print("axis 1 (columns):", grid.shape[1])




Two smaller readings come off the same metadata:

- **`ndim`** — how many axes there are. This is the nesting depth, and it is an
  attribute, not a call.
- **`numel()`** — the total element count, which is the *product* of the shape.
  This is a method, so it needs the parentheses. For a 2×3 tensor `ndim` is 2
  and `numel()` is 6 — six numbers arranged as two rows, not two of anything.

Getting those two confused is the classic first-week error, and it is worth
fixing now: `ndim` counts axes, `numel()` counts numbers.


In [ ]:
# Four numbers, three nestings, three different answers for ndim — and the same
# answer for numel every time.
for data in ([1, 2, 3, 4], [[1, 2], [3, 4]], [[[1], [2]], [[3], [4]]]):
    x = t.tensor(data)
    print(tuple(x.shape), "ndim", x.ndim, "numel", x.numel())


The problem below asks for a tensor's number of axes and its element count,
in that order: `(ndim, numel)`. Build the tensor, then read those two pieces of
metadata directly:


In [ ]:
import torch as t

rows = [[1, 2, 3], [4, 5, 6]]
a = t.tensor(rows)

print("ndim:", a.ndim)
print("numel:", a.numel())
assert (a.ndim, a.numel()) == (2, 6)




Two nesting levels give `ndim == 2`. Six numbers give `numel() == 6`.
Tuple position matters: `(2, 6)` is correct; `(6, 2)` reverses the requested
order.


<!-- dd:dd-q482 -->

### Problem 482 · faded — your turn

Count the axes and the elements. Return them in this exact order:
`(ndim, numel)`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2, 6)
```


In [ ]:
import torch as t

def solve(rows):
    """Return (number of axes, total element count) for the tensor from rows."""
    a = t._____(rows)
    return (a._____, a._____())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(482)


In [ ]:
#@title 💡 Solution — Problem 482
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (number of axes, total element count) for the tensor from rows."""
    a = t.tensor(rows)
    return (a.ndim, a.numel())


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


The problem below asks for a tensor's number of axes and its element count,
in that order: `(ndim, numel)`. Build the tensor, then read those two pieces of
metadata directly:


In [ ]:
import torch as t

rows = [[1, 2, 3], [4, 5, 6]]
a = t.tensor(rows)

print("ndim:", a.ndim)
print("numel:", a.numel())
assert (a.ndim, a.numel()) == (2, 6)




Two nesting levels give `ndim == 2`. Six numbers give `numel() == 6`.
Tuple position matters: `(2, 6)` is correct; `(6, 2)` reverses the requested
order.


<!-- dd:dd-q537 -->

### Problem 537 · faded — your turn

Return the tensor's shape as a plain Python tuple of ints.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2, 3)
```


In [ ]:
import torch as t

def solve(rows):
    """Return the tensor's shape as a plain tuple of ints."""
    a = t._____(rows)
    return tuple(a._____)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(537)


In [ ]:
#@title 💡 Solution — Problem 537
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return the tensor's shape as a plain tuple of ints."""
    a = t.tensor(rows)
    return tuple(a.shape)


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


The problem below asks for a tensor's number of axes and its element count,
in that order: `(ndim, numel)`. Build the tensor, then read those two pieces of
metadata directly:


In [ ]:
import torch as t

rows = [[1, 2, 3], [4, 5, 6]]
a = t.tensor(rows)

print("ndim:", a.ndim)
print("numel:", a.numel())
assert (a.ndim, a.numel()) == (2, 6)




Two nesting levels give `ndim == 2`. Six numbers give `numel() == 6`.
Tuple position matters: `(2, 6)` is correct; `(6, 2)` reverses the requested
order.


<!-- dd:dd-q538 -->

### Problem 538 · faded — your turn

Return the length along axis 0 and the length along axis 1.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2, 3)
```


In [ ]:
import torch as t

def solve(rows):
    """Return (length along axis 0, length along axis 1)."""
    a = t._____(rows)
    return (a._____[0], a._____[1])


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(538)


In [ ]:
#@title 💡 Solution — Problem 538
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (length along axis 0, length along axis 1)."""
    a = t.tensor(rows)
    return (a.shape[0], a.shape[1])


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


The problem below asks for a tensor's number of axes and its element count,
in that order: `(ndim, numel)`. Build the tensor, then read those two pieces of
metadata directly:


In [ ]:
import torch as t

rows = [[1, 2, 3], [4, 5, 6]]
a = t.tensor(rows)

print("ndim:", a.ndim)
print("numel:", a.numel())
assert (a.ndim, a.numel()) == (2, 6)




Two nesting levels give `ndim == 2`. Six numbers give `numel() == 6`.
Tuple position matters: `(2, 6)` is correct; `(6, 2)` reverses the requested
order.


<!-- dd:dd-q539 -->

### Problem 539 · faded — your turn

Return how many axes the tensor has, as a plain int.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
3
```


In [ ]:
import torch as t

def solve(data):
    """Return the tensor's number of axes."""
    a = t._____(data)
    return a._____


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(539)


In [ ]:
#@title 💡 Solution — Problem 539
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(data):
    """Return the tensor's number of axes."""
    a = t.tensor(data)
    return a.ndim


example = [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
print(solve(example))


The problem below asks for a tensor's number of axes and its element count,
in that order: `(ndim, numel)`. Build the tensor, then read those two pieces of
metadata directly:


In [ ]:
import torch as t

rows = [[1, 2, 3], [4, 5, 6]]
a = t.tensor(rows)

print("ndim:", a.ndim)
print("numel:", a.numel())
assert (a.ndim, a.numel()) == (2, 6)




Two nesting levels give `ndim == 2`. Six numbers give `numel() == 6`.
Tuple position matters: `(2, 6)` is correct; `(6, 2)` reverses the requested
order.


<!-- dd:dd-q540 -->

### Problem 540 · faded — your turn

Two counts that are not the same question: every number in the tensor, and the rows in the input.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(6, 2)
```


In [ ]:
import torch as t

def solve(rows):
    """Return (total element count, number of outer rows)."""
    a = t._____(rows)
    return (a._____(), len(rows))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(540)


In [ ]:
#@title 💡 Solution — Problem 540
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (total element count, number of outer rows)."""
    a = t.tensor(rows)
    return (a.numel(), len(rows))


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


The problem below asks for a tensor's number of axes and its element count,
in that order: `(ndim, numel)`. Build the tensor, then read those two pieces of
metadata directly:


In [ ]:
import torch as t

rows = [[1, 2, 3], [4, 5, 6]]
a = t.tensor(rows)

print("ndim:", a.ndim)
print("numel:", a.numel())
assert (a.ndim, a.numel()) == (2, 6)




Two nesting levels give `ndim == 2`. Six numbers give `numel() == 6`.
Tuple position matters: `(2, 6)` is correct; `(6, 2)` reverses the requested
order.


<!-- dd:dd-q541 -->

### Problem 541 · faded — your turn

One bool: does the tensor's shape equal the tuple you were handed?

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
True
```


In [ ]:
import torch as t

def solve(rows, wanted):
    """Return True when the tensor's shape equals `wanted`."""
    a = t._____(rows)
    return tuple(a._____) == tuple(wanted)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]], (2, 3))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(541)


In [ ]:
#@title 💡 Solution — Problem 541
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows, wanted):
    """Return True when the tensor's shape equals `wanted`."""
    a = t.tensor(rows)
    return tuple(a.shape) == tuple(wanted)


example = ([[1, 2, 3], [4, 5, 6]], (2, 3))
print(solve(*example))


<!-- dd:dd-seg-numpy-ndarray-model-2 -->

### dtype is a property of the whole block


The **dtype** is the single element type shared by every entry — `torch.int64`,
`torch.float32`, `torch.bool`, and so on. There is exactly one per tensor,
because there is exactly one block of memory and every slot in it is the same
size.

That has a consequence people meet by accident: when you build a tensor from
mixed Python numbers, PyTorch cannot keep some entries as ints and some as
floats. It picks ONE type that can hold everything, so a single float anywhere
in the input turns the whole tensor into `torch.float32`. All-integer input
gives you `torch.int64` instead.


In [ ]:
import torch as t

print(t.tensor([1, 2, 3]).dtype)
print(t.tensor([1, 2.5, 3]).dtype)




All-integer input becomes `torch.int64`. One float makes the whole tensor
`torch.float32`. You can also choose the dtype explicitly:


In [ ]:
print(t.tensor([1, 2, 3], dtype=t.float32).dtype)




Division shows why one dtype for the whole block matters. `a / 2` halves every
number and creates a new float tensor. It does not change `a`:


In [ ]:
a = t.tensor([2, 4, 6])
halved = a / 2

print(a.tolist())
print(halved.tolist())
print(halved.dtype)




`a = a / 2` would make that new float tensor, then rebind the name `a` to it.
`a /= 2` instead tries to write float results back into the original integer
tensor. PyTorch rejects that in-place mutation rather than silently rounding:


In [ ]:
a = t.tensor([2, 4, 6])

try:
    a /= 2
except RuntimeError as exc:
    print(type(exc).__name__)




Two tensors can hold the same numbers in the same layout and still disagree on
dtype. Shape and dtype are independent, and code that checks only one of them
is checking half the question.


The problem below asks you to compare two tensors on shape and dtype. Those are
two separate checks.

Start with two tensors built from the same numbers in the same layout:


In [ ]:
import torch as t

a = t.tensor([[20, 21, 22], [23, 24, 25]])
b = t.tensor([[30, 31, 32], [33, 34, 35]])

print(a.shape == b.shape)
print(a.dtype == b.dtype)
assert (a.shape == b.shape, a.dtype == b.dtype) == (True, True)




Both checks are `True`. Now change one number to a float. Layout stays same, so
shapes still match. Dtypes do not:


In [ ]:
c = t.tensor([[30, 31.5, 32], [33, 34, 35]])

print(a.shape == c.shape)
print(a.dtype == c.dtype)
assert (a.shape == c.shape, a.dtype == c.dtype) == (True, False)




Last, keep integer dtype but change layout. Dtypes match; shapes do not:


In [ ]:
d = t.tensor([[40, 41], [42, 43], [44, 45]])

print(a.shape == d.shape)
print(a.dtype == d.dtype)
assert (a.shape == d.shape, a.dtype == d.dtype) == (False, True)




`.shape` and `.dtype` are attributes; neither takes parentheses.


<!-- dd:dd-q484 -->

### Problem 484 · faded — your turn

Two tensors, two independent questions. Compare each piece of metadata on its
own.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, True)
```


In [ ]:
import torch as t

def solve(rows_a, rows_b):
    """Return (do the shapes match?, do the dtypes match?)."""
    a = t._____(rows_a)
    b = t._____(rows_b)
    return (a._____ == b._____, a._____ == b._____)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2], [3, 4]], [[5, 6], [7, 8]])
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(484)


In [ ]:
#@title 💡 Solution — Problem 484
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows_a, rows_b):
    """Return (do the shapes match?, do the dtypes match?)."""
    a = t.tensor(rows_a)
    b = t.tensor(rows_b)
    return (a.shape == b.shape, a.dtype == b.dtype)


example = ([[1, 2], [3, 4]], [[5, 6], [7, 8]])
print(solve(*example))


The problem below asks you to compare two tensors on shape and dtype. Those are
two separate checks.

Start with two tensors built from the same numbers in the same layout:


In [ ]:
import torch as t

a = t.tensor([[20, 21, 22], [23, 24, 25]])
b = t.tensor([[30, 31, 32], [33, 34, 35]])

print(a.shape == b.shape)
print(a.dtype == b.dtype)
assert (a.shape == b.shape, a.dtype == b.dtype) == (True, True)




Both checks are `True`. Now change one number to a float. Layout stays same, so
shapes still match. Dtypes do not:


In [ ]:
c = t.tensor([[30, 31.5, 32], [33, 34, 35]])

print(a.shape == c.shape)
print(a.dtype == c.dtype)
assert (a.shape == c.shape, a.dtype == c.dtype) == (True, False)




Last, keep integer dtype but change layout. Dtypes match; shapes do not:


In [ ]:
d = t.tensor([[40, 41], [42, 43], [44, 45]])

print(a.shape == d.shape)
print(a.dtype == d.dtype)
assert (a.shape == d.shape, a.dtype == d.dtype) == (False, True)




`.shape` and `.dtype` are attributes; neither takes parentheses.


<!-- dd:dd-q542 -->

### Problem 542 · faded — your turn

Return the name of the tensor's element type as a string.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
torch.int64
```


In [ ]:
import torch as t

def solve(values):
    """Return str() of the tensor's dtype."""
    a = t._____(values)
    return str(a._____)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [1, 2, 3]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(542)


In [ ]:
#@title 💡 Solution — Problem 542
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return str() of the tensor's dtype."""
    a = t.tensor(values)
    return str(a.dtype)


example = [1, 2, 3]
print(solve(example))


The problem below asks you to compare two tensors on shape and dtype. Those are
two separate checks.

Start with two tensors built from the same numbers in the same layout:


In [ ]:
import torch as t

a = t.tensor([[20, 21, 22], [23, 24, 25]])
b = t.tensor([[30, 31, 32], [33, 34, 35]])

print(a.shape == b.shape)
print(a.dtype == b.dtype)
assert (a.shape == b.shape, a.dtype == b.dtype) == (True, True)




Both checks are `True`. Now change one number to a float. Layout stays same, so
shapes still match. Dtypes do not:


In [ ]:
c = t.tensor([[30, 31.5, 32], [33, 34, 35]])

print(a.shape == c.shape)
print(a.dtype == c.dtype)
assert (a.shape == c.shape, a.dtype == c.dtype) == (True, False)




Last, keep integer dtype but change layout. Dtypes match; shapes do not:


In [ ]:
d = t.tensor([[40, 41], [42, 43], [44, 45]])

print(a.shape == d.shape)
print(a.dtype == d.dtype)
assert (a.shape == d.shape, a.dtype == d.dtype) == (False, True)




`.shape` and `.dtype` are attributes; neither takes parentheses.


<!-- dd:dd-q543 -->

### Problem 543 · faded — your turn

One bool: is the whole block held as 64-bit integers?

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
True
```


In [ ]:
import torch as t

def solve(values):
    """Return True when the tensor's dtype is the 64-bit integer one."""
    a = t._____(values)
    return a._____ == t._____


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [1, 2, 3]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(543)


In [ ]:
#@title 💡 Solution — Problem 543
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return True when the tensor's dtype is the 64-bit integer one."""
    a = t.tensor(values)
    return a.dtype == t.int64


example = [1, 2, 3]
print(solve(example))


The problem below asks you to compare two tensors on shape and dtype. Those are
two separate checks.

Start with two tensors built from the same numbers in the same layout:


In [ ]:
import torch as t

a = t.tensor([[20, 21, 22], [23, 24, 25]])
b = t.tensor([[30, 31, 32], [33, 34, 35]])

print(a.shape == b.shape)
print(a.dtype == b.dtype)
assert (a.shape == b.shape, a.dtype == b.dtype) == (True, True)




Both checks are `True`. Now change one number to a float. Layout stays same, so
shapes still match. Dtypes do not:


In [ ]:
c = t.tensor([[30, 31.5, 32], [33, 34, 35]])

print(a.shape == c.shape)
print(a.dtype == c.dtype)
assert (a.shape == c.shape, a.dtype == c.dtype) == (True, False)




Last, keep integer dtype but change layout. Dtypes match; shapes do not:


In [ ]:
d = t.tensor([[40, 41], [42, 43], [44, 45]])

print(a.shape == d.shape)
print(a.dtype == d.dtype)
assert (a.shape == d.shape, a.dtype == d.dtype) == (False, True)




`.shape` and `.dtype` are attributes; neither takes parentheses.


<!-- dd:dd-q544 -->

### Problem 544 · faded — your turn

Build the tensor as 32-bit floats rather than letting the input decide, then read it back as a list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[1.0, 2.0, 3.0]
```


In [ ]:
import torch as t

def solve(values):
    """Return the values as a float32 tensor, read back as a list."""
    a = t._____(values, _____=t._____)
    return a._____()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [1, 2, 3]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(544)


In [ ]:
#@title 💡 Solution — Problem 544
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return the values as a float32 tensor, read back as a list."""
    a = t.tensor(values, dtype=t.float32)
    return a.tolist()


example = [1, 2, 3]
print(solve(example))


The problem below asks you to compare two tensors on shape and dtype. Those are
two separate checks.

Start with two tensors built from the same numbers in the same layout:


In [ ]:
import torch as t

a = t.tensor([[20, 21, 22], [23, 24, 25]])
b = t.tensor([[30, 31, 32], [33, 34, 35]])

print(a.shape == b.shape)
print(a.dtype == b.dtype)
assert (a.shape == b.shape, a.dtype == b.dtype) == (True, True)




Both checks are `True`. Now change one number to a float. Layout stays same, so
shapes still match. Dtypes do not:


In [ ]:
c = t.tensor([[30, 31.5, 32], [33, 34, 35]])

print(a.shape == c.shape)
print(a.dtype == c.dtype)
assert (a.shape == c.shape, a.dtype == c.dtype) == (True, False)




Last, keep integer dtype but change layout. Dtypes match; shapes do not:


In [ ]:
d = t.tensor([[40, 41], [42, 43], [44, 45]])

print(a.shape == d.shape)
print(a.dtype == d.dtype)
assert (a.shape == d.shape, a.dtype == d.dtype) == (False, True)




`.shape` and `.dtype` are attributes; neither takes parentheses.


<!-- dd:dd-q545 -->

### Problem 545 · faded — your turn

Two answers: is the block float32, and how many numbers are in it?

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, 3)
```


In [ ]:
import torch as t

def solve(values):
    """Return (is the block float32?, how many elements)."""
    a = t._____(values)
    return (a._____ == t._____, a._____())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [1, 2.5, 3]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(545)


In [ ]:
#@title 💡 Solution — Problem 545
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (is the block float32?, how many elements)."""
    a = t.tensor(values)
    return (a.dtype == t.float32, a.numel())


example = [1, 2.5, 3]
print(solve(example))


The problem below asks you to compare two tensors on shape and dtype. Those are
two separate checks.

Start with two tensors built from the same numbers in the same layout:


In [ ]:
import torch as t

a = t.tensor([[20, 21, 22], [23, 24, 25]])
b = t.tensor([[30, 31, 32], [33, 34, 35]])

print(a.shape == b.shape)
print(a.dtype == b.dtype)
assert (a.shape == b.shape, a.dtype == b.dtype) == (True, True)




Both checks are `True`. Now change one number to a float. Layout stays same, so
shapes still match. Dtypes do not:


In [ ]:
c = t.tensor([[30, 31.5, 32], [33, 34, 35]])

print(a.shape == c.shape)
print(a.dtype == c.dtype)
assert (a.shape == c.shape, a.dtype == c.dtype) == (True, False)




Last, keep integer dtype but change layout. Dtypes match; shapes do not:


In [ ]:
d = t.tensor([[40, 41], [42, 43], [44, 45]])

print(a.shape == d.shape)
print(a.dtype == d.dtype)
assert (a.shape == d.shape, a.dtype == d.dtype) == (False, True)




`.shape` and `.dtype` are attributes; neither takes parentheses.


<!-- dd:dd-q546 -->

### Problem 546 · faded — your turn

One bool: do the two tensors agree on element type?

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
True
```


In [ ]:
import torch as t

def solve(values_a, values_b):
    """Return True when the two tensors share a dtype."""
    a = t._____(values_a)
    b = t._____(values_b)
    return a._____ == b._____


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([1, 2, 3], [4, 5, 6])
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(546)


In [ ]:
#@title 💡 Solution — Problem 546
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values_a, values_b):
    """Return True when the two tensors share a dtype."""
    a = t.tensor(values_a)
    b = t.tensor(values_b)
    return a.dtype == b.dtype


example = ([1, 2, 3], [4, 5, 6])
print(solve(*example))


<!-- dd:dd-q480 -->

### Problem 480 · independent

Write a function `solve(rows)` that takes a nested Python list of equal-length inner lists of numbers and returns a tuple `(a, shape, is_float)`, where `a` is the 2-D tensor built from rows.

Two of the three are easy to get subtly wrong, so check them:

- `shape` must be a plain Python tuple of ints. Returning it straight off the tensor gives you a `torch.Size`, which prints similarly and is NOT what this asks for.
- `is_float` asks whether the tensor's dtype is the 32-bit float type. Remember a tensor holds ONE type for every element.

For `[[1, 2, 3], [4, 5, 6]]` a WRONG return looks like this — note the second entry:

```text
(tensor([[1, 2, 3],
        [4, 5, 6]]), torch.Size([2, 3]), False)
```

and the correct one has `(2, 3)` there instead.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([[1, 2, 3],
        [4, 5, 6]]), (2, 3), False)
```


In [ ]:
import torch
import torch as t

def solve(rows):
    """Return (tensor, shape-as-plain-tuple, True if dtype is float32)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(480)


In [ ]:
#@title 💡 Solution — Problem 480
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    a = t.tensor(rows)
    return (a, tuple(a.shape), a.dtype == t.float32)


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


<!-- dd:dd-q481 -->

### Problem 481 · independent

Write a function solve(values) that takes a FLAT Python list of numbers and returns a tuple (a, ndim). `a` is the tensor built from values, and `ndim` is its number of axes as a plain int. A flat list has one level of nesting, so think about what that makes ndim before you run it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([4, 1, 7]), 1)
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return (tensor built from values, its number of axes)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [4, 1, 7]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(481)


In [ ]:
#@title 💡 Solution — Problem 481
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (tensor built from values, its number of axes)."""
    a = t.tensor(values)
    return (a, a.ndim)


example = [4, 1, 7]
print(solve(example))


<!-- dd:dd-q483 -->

### Problem 483 · independent

Write a function solve(values) that takes a FLAT Python list of numbers and returns a tuple (dtype_name, numel). `dtype_name` is str() of the tensor's dtype — the string 'torch.int64' or 'torch.float32'. `numel` is the element count as a plain int. A tensor holds ONE type for every element, so a single float anywhere in the list decides the type of the whole block.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
('torch.int64', 3)
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return (str of the tensor's dtype, its element count)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [1, 2, 3]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(483)


In [ ]:
#@title 💡 Solution — Problem 483
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (str of the tensor's dtype, its element count)."""
    a = t.tensor(values)
    return (str(a.dtype), a.numel())


example = [1, 2, 3]
print(solve(example))


<!-- dd:dd-q485 -->

### Problem 485 · independent

Write a function solve(cube) that takes a TRIPLY nested Python list — a list of equal-length lists of equal-length lists of numbers — and returns a tuple (ndim, shape, numel). `shape` must be a plain Python tuple of ints, not a torch.Size. Nesting depth becomes the number of axes: the outermost list is axis 0.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(3, (2, 2, 2), 8)
```


In [ ]:
import torch
import torch as t

def solve(cube):
    """Return (number of axes, shape as a plain tuple, element count)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(485)


In [ ]:
#@title 💡 Solution — Problem 485
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(cube):
    """Return (number of axes, shape as a plain tuple, element count)."""
    a = t.tensor(cube)
    return (a.ndim, tuple(a.shape), a.numel())


example = [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
print(solve(example))


<!-- dd:dd-q486 -->

### Problem 486 · independent

Write a function solve(rows) that returns a tuple (inferred_name, forced_name, unchanged). Build one tensor from rows and let PyTorch infer the dtype; build a second from the same rows but force it to torch.float32 by passing dtype=. The first two entries are str() of each tensor's dtype; `unchanged` is True when the two dtypes are equal. Forcing is how you stop an all-integer input from silently giving you an integer tensor where the rest of your code expects floats.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
('torch.int64', 'torch.float32', False)
```


In [ ]:
import torch
import torch as t

def solve(rows):
    """Return (inferred dtype name, forced float32 dtype name, are they equal?)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2], [3, 4]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(486)


In [ ]:
#@title 💡 Solution — Problem 486
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (inferred dtype name, forced float32 dtype name, are they equal?)."""
    inferred = t.tensor(rows)
    forced = t.tensor(rows, dtype=t.float32)
    return (str(inferred.dtype), str(forced.dtype), inferred.dtype == forced.dtype)


example = [[1, 2], [3, 4]]
print(solve(example))


<!-- dd:dd-q547 -->

### Problem 547 · independent

Write a function solve(values) that takes a FLAT Python list of numbers and returns a tuple (ndim, numel, shape), where shape is a plain tuple of ints. A flat list is one level of nesting however many numbers are in it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(1, 5, (5,))
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return (number of axes, element count, shape as a tuple)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [4, 1, 7, 2, 9]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(547)


In [ ]:
#@title 💡 Solution — Problem 547
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (number of axes, element count, shape as a tuple)."""
    a = t.tensor(values)
    return (a.ndim, a.numel(), tuple(a.shape))


example = [4, 1, 7, 2, 9]
print(solve(example))


<!-- dd:dd-q548 -->

### Problem 548 · independent

Write a function solve(values) that takes a FLAT list and returns a tuple (dtype_name, is_float) — str() of the tensor's dtype, and whether that dtype is torch.float32. A single float in the input decides the type of the whole block.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
('torch.float32', True)
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return (dtype name as a string, is it float32?)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [1, 2, 3.5]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(548)


In [ ]:
#@title 💡 Solution — Problem 548
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (dtype name as a string, is it float32?)."""
    a = t.tensor(values)
    return (str(a.dtype), a.dtype == t.float32)


example = [1, 2, 3.5]
print(solve(example))


<!-- dd:dd-q549 -->

### Problem 549 · independent

Write a function solve(cube) that takes a TRIPLY nested Python list and returns a tuple (shape, numel, product_matches). shape is a plain tuple of ints; product_matches is a bool saying whether multiplying the shape together gives the element count. Compute the product with a plain Python loop over the shape.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((2, 2, 2), 8, True)
```


In [ ]:
import torch
import torch as t

def solve(cube):
    """Return (shape, element count, does the shape multiply out?)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(549)


In [ ]:
#@title 💡 Solution — Problem 549
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(cube):
    """Return (shape, element count, does the shape multiply out?)."""
    a = t.tensor(cube)
    shape = tuple(a.shape)
    product = 1
    for length in shape:
        product = product * length
    return (shape, a.numel(), product == a.numel())


example = [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
print(solve(example))


<!-- dd:dd-q550 -->

### Problem 550 · independent

Write a function solve(values) that takes a FLAT list and returns the tensor's only element as a plain Python number when the tensor holds exactly one, and None otherwise. Check the element count yourself rather than letting the call raise.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
42
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return the single element as a Python number, or None."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [42]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(550)


In [ ]:
#@title 💡 Solution — Problem 550
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return the single element as a Python number, or None."""
    a = t.tensor(values)
    if a.numel() != 1:
        return None
    return a.item()


example = [42]
print(solve(example))


<!-- dd:dd-q551 -->

### Problem 551 · independent

Write a function solve(rows_a, rows_b) that returns a tuple (equal, same_shape): the whole-tensor comparison of the two tensors, and whether they merely agree on shape. Code that checks only the shape is checking half the question — two tensors can be the same size and hold completely different numbers.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(False, True)
```


In [ ]:
import torch
import torch as t

def solve(rows_a, rows_b):
    """Return (whole-tensor equal?, same shape?)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2], [3, 4]], [[9, 9], [9, 9]])
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(551)


In [ ]:
#@title 💡 Solution — Problem 551
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows_a, rows_b):
    """Return (whole-tensor equal?, same shape?)."""
    a = t.tensor(rows_a)
    b = t.tensor(rows_b)
    return (t.equal(a, b), tuple(a.shape) == tuple(b.shape))


example = ([[1, 2], [3, 4]], [[9, 9], [9, 9]])
print(solve(*example))


<!-- dd:dd-q554 -->

### Problem 554 · independent

Write a function solve(values) that takes a FLAT list which may contain floats, forces the tensor to 64-bit integers, and returns a tuple (dtype_name, contents) — str() of the dtype and the values read back as a plain list. Forcing an integer type truncates toward zero, which is the point: say what you want rather than relying on the input.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
('torch.int64', [1, 2, 3])
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return (forced dtype name, the values as a plain list)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [1.9, 2.2, 3.7]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(554)


In [ ]:
#@title 💡 Solution — Problem 554
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (forced dtype name, the values as a plain list)."""
    a = t.tensor(values, dtype=t.int64)
    return (str(a.dtype), a.tolist())


example = [1.9, 2.2, 3.7]
print(solve(example))


<!-- dd:dd-q555 -->

### Problem 555 · independent

Write a function solve(values) that returns a tuple (inferred_name, forced_name, same) — the dtype name PyTorch infers from the FLAT list, the dtype name you get by forcing torch.float32 on the same list, and whether the two names are equal.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
('torch.int64', 'torch.float32', False)
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return (inferred dtype name, forced float32 name, equal?)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [1, 2, 3]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(555)


In [ ]:
#@title 💡 Solution — Problem 555
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (inferred dtype name, forced float32 name, equal?)."""
    inferred = t.tensor(values)
    forced = t.tensor(values, dtype=t.float32)
    return (str(inferred.dtype), str(forced.dtype),
            inferred.dtype == forced.dtype)


example = [1, 2, 3]
print(solve(example))


<!-- dd:dd-q556 -->

### Problem 556 · independent

Write a function solve(rows_a, rows_b) that returns a tuple (same_numel, same_shape) for the two tensors built from the nested lists. Two tensors can hold the same COUNT of numbers arranged differently, so these are not the same question.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, False)
```


In [ ]:
import torch
import torch as t

def solve(rows_a, rows_b):
    """Return (same element count?, same shape?)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]], [[1, 2], [3, 4], [5, 6]])
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(556)


In [ ]:
#@title 💡 Solution — Problem 556
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows_a, rows_b):
    """Return (same element count?, same shape?)."""
    a = t.tensor(rows_a)
    b = t.tensor(rows_b)
    return (a.numel() == b.numel(), a.shape == b.shape)


example = ([[1, 2, 3], [4, 5, 6]], [[1, 2], [3, 4], [5, 6]])
print(solve(*example))


<!-- dd:dd-q557 -->

### Problem 557 · independent

Write a function solve(rows) that builds a 2-D tensor and returns a tuple (row_shape, row_ndim) describing its FIRST row, indexed as rows[0] would be. Indexing away the outermost axis leaves one fewer.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((3,), 1)
```


In [ ]:
import torch
import torch as t

def solve(rows):
    """Return (shape of the first row as a tuple, its number of axes)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(557)


In [ ]:
#@title 💡 Solution — Problem 557
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (shape of the first row as a tuple, its number of axes)."""
    a = t.tensor(rows)
    first = a[0]
    return (tuple(first.shape), first.ndim)


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


<!-- dd:dd-q559 -->

### Problem 559 · independent

Write a function solve(values) that takes a FLAT list and returns a tuple (numel, ndim, is_scalar_sized) — the element count, the number of axes, and a bool saying whether the tensor holds exactly one element. A one-element tensor is still 1-D.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(1, 1, True)
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return (element count, number of axes, holds exactly one?)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [5]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(559)


In [ ]:
#@title 💡 Solution — Problem 559
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (element count, number of axes, holds exactly one?)."""
    a = t.tensor(values)
    return (a.numel(), a.ndim, a.numel() == 1)


example = [5]
print(solve(example))


#### Common mistakes

- **"A tensor is just a faster list."** — A list stores anything, a tensor
  stores exactly one dtype in one memory block. That's why `t.tensor([1, 2.5])`
  changes your integer to a float: PyTorch must pick ONE type for the block.
- **"I need to tell PyTorch the shape when converting data."** — `t.tensor`
  infers shape from the nesting. You only specify shapes with from-scratch
  constructors (`t.zeros((2, 3))`), covered next.
- **"shape is (columns, rows)."** — It is (axis 0, axis 1) = (rows, columns)
  for a matrix. Axis 0 is always the outermost nesting level.
- **"`ndim` and `numel()` are two names for the size."** — `ndim` counts axes,
  `numel()` counts elements. A 2×3 tensor has `ndim == 2` and `numel() == 6`.


<!-- dd:dd-kp-numpy-transpose-axes -->

## Transpose — swapping which axis is which

`numpy.transpose-axes`


<!-- dd:dd-seg-numpy-transpose-axes-0 -->

### transpose turns the rows into the columns


You already know a 2-D tensor as a grid: `t.tensor([[1, 2, 3], [4, 5, 6]])` has
two rows and three columns, and its shape says so — `(2, 3)`.

**Transposing** that grid means turning it on its side. Row 0 of the result is
column 0 of the original, row 1 of the result is column 1, and so on. Three
columns going in means three rows coming out, so the shape `(2, 3)` comes back
as `(3, 2)`. Nothing is added, dropped or rounded — the same six numbers are
being written down in a different arrangement.

The spelling is `a.T`, and it is worth saying out loud that **`.T` has no
parentheses**. It is an attribute, like `.shape` and `.ndim`, not a method like
`.tolist()`. Writing `a.T()` is an error, and writing `a.T` where you meant
`a.T.tolist()` hands back a tensor when you asked for a list.


In [ ]:
import torch as t

a = t.tensor([[1, 2, 3], [4, 5, 6]])
print(a)
print("shape:", tuple(a.shape))

print(a.T)
print("shape after transposing:", tuple(a.T.shape))




Read those two grids against each other. The first row of `a` is `1 2 3`, and
those three numbers came out as the first *column* of `a.T`. That is the whole
operation.


The problem below asks for the transpose's shape. Do it once here, on a tensor
whose two axis lengths are different — which is the only way to see that the
shape really did reverse.


In [ ]:
import torch as t

a = t.tensor([[1, 2], [3, 4], [5, 6]])
print("three rows of two:", tuple(a.shape))
print("becomes two rows of three:", tuple(a.T.shape))




`tuple(...)` is there because `a.shape` is a `torch.Size`, which prints like a
tuple but is not one; a checker comparing against `(2, 3)` wants the plain
tuple. And notice the shape reversed without you telling PyTorch anything about
lengths — it read both off the tensor.


In [ ]:
square = t.tensor([[1, 2], [3, 4]])
print("a square tensor keeps its shape:", tuple(square.T.shape))
print("...but not its contents:", square.T.tolist())




That second line is the case worth remembering. A square tensor transposes to
the same *shape*, so shape alone cannot tell you whether a transpose happened.


<!-- dd:dd-q610 -->

### Problem 610 · faded — your turn

The transpose's shape, as a plain tuple.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(3, 2)
```


In [ ]:
import torch as t

def solve(rows):
    """Return the transpose's shape as a plain tuple."""
    a = t.tensor(rows)
    return tuple(a._____.shape)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(610)


In [ ]:
#@title 💡 Solution — Problem 610
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return the transpose's shape as a plain tuple."""
    a = t.tensor(rows)
    return tuple(a.T.shape)


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


The problem below asks for the transpose's shape. Do it once here, on a tensor
whose two axis lengths are different — which is the only way to see that the
shape really did reverse.


In [ ]:
import torch as t

a = t.tensor([[1, 2], [3, 4], [5, 6]])
print("three rows of two:", tuple(a.shape))
print("becomes two rows of three:", tuple(a.T.shape))




`tuple(...)` is there because `a.shape` is a `torch.Size`, which prints like a
tuple but is not one; a checker comparing against `(2, 3)` wants the plain
tuple. And notice the shape reversed without you telling PyTorch anything about
lengths — it read both off the tensor.


In [ ]:
square = t.tensor([[1, 2], [3, 4]])
print("a square tensor keeps its shape:", tuple(square.T.shape))
print("...but not its contents:", square.T.tolist())




That second line is the case worth remembering. A square tensor transposes to
the same *shape*, so shape alone cannot tell you whether a transpose happened.


<!-- dd:dd-q552 -->

### Problem 552 · faded — your turn

Both shapes at once: the tensor's own, then the transpose's.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((2, 3), (3, 2))
```


In [ ]:
import torch as t

def solve(rows):
    """Return (a's shape, a.T's shape), both plain tuples."""
    a = t.tensor(rows)
    return (tuple(a.shape), tuple(a._____.shape))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(552)


In [ ]:
#@title 💡 Solution — Problem 552
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (a's shape, a.T's shape), both plain tuples."""
    a = t.tensor(rows)
    return (tuple(a.shape), tuple(a.T.shape))


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


The problem below asks for the transpose's shape. Do it once here, on a tensor
whose two axis lengths are different — which is the only way to see that the
shape really did reverse.


In [ ]:
import torch as t

a = t.tensor([[1, 2], [3, 4], [5, 6]])
print("three rows of two:", tuple(a.shape))
print("becomes two rows of three:", tuple(a.T.shape))




`tuple(...)` is there because `a.shape` is a `torch.Size`, which prints like a
tuple but is not one; a checker comparing against `(2, 3)` wants the plain
tuple. And notice the shape reversed without you telling PyTorch anything about
lengths — it read both off the tensor.


In [ ]:
square = t.tensor([[1, 2], [3, 4]])
print("a square tensor keeps its shape:", tuple(square.T.shape))
print("...but not its contents:", square.T.tolist())




That second line is the case worth remembering. A square tensor transposes to
the same *shape*, so shape alone cannot tell you whether a transpose happened.


<!-- dd:dd-q669 -->

### Problem 669 · faded — your turn

Rows before and rows after — the second is the old column count.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2, 3)
```


In [ ]:
import torch as t

def solve(rows):
    """Return (how many rows a has, how many rows a.T has)."""
    a = t.tensor(rows)
    return (len(a), len(a._____))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(669)


In [ ]:
#@title 💡 Solution — Problem 669
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return (how many rows a has, how many rows a.T has)."""
    a = t.tensor(rows)
    return (len(a), len(a.T))


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


<!-- dd:dd-seg-numpy-transpose-axes-1 -->

### reading the transposed numbers back out


The shape tells you the arrangement. To see the numbers themselves, read them
out the same way you read any tensor — with `.tolist()`, which you met on the
previous concept.


In [ ]:
import torch as t

a = t.tensor([[1, 2, 3], [4, 5, 6]])
print("a          :", a.tolist())
print("a.T        :", a.T.tolist())
print("a.T row 0  :", a.T.tolist()[0], "  <- column 0 of a")




`[1, 4]` is the answer to "what was the first number of each row?" — and that
is what a column *is*. Every row of the transpose answers that question for one
column of the original.

Transposing twice puts every axis back where it started, so `a.T.T` is `a`
again — same shape, same numbers:


In [ ]:
back = a.T.T
print("shape is back:", tuple(back.shape) == tuple(a.shape))
print("values are back:", t.equal(back, a))


Pulling one row out of a transpose is two steps, and it is worth keeping them
separate the first time: transpose, then read.


In [ ]:
import torch as t

a = t.tensor([[10, 20], [30, 40], [50, 60]])

view = a.T                 # step 1: the tensor, turned on its side
rows = view.tolist()       # step 2: the same numbers as plain Python lists

print("every row's first number:", rows[0])
print("every row's second number:", rows[1])
assert rows[0] == [10, 30, 50]


<!-- dd:dd-q611 -->

### Problem 611 · faded — your turn

The transpose's first row — which is the first number of every original row.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[1, 4]
```


In [ ]:
import torch as t

def solve(rows):
    """Return the transpose's first row as a plain list."""
    a = t.tensor(rows)
    return a._____.tolist()[0]


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(611)


In [ ]:
#@title 💡 Solution — Problem 611
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return the transpose's first row as a plain list."""
    a = t.tensor(rows)
    return a.T.tolist()[0]


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


Pulling one row out of a transpose is two steps, and it is worth keeping them
separate the first time: transpose, then read.


In [ ]:
import torch as t

a = t.tensor([[10, 20], [30, 40], [50, 60]])

view = a.T                 # step 1: the tensor, turned on its side
rows = view.tolist()       # step 2: the same numbers as plain Python lists

print("every row's first number:", rows[0])
print("every row's second number:", rows[1])
assert rows[0] == [10, 30, 50]


<!-- dd:dd-q558 -->

### Problem 558 · faded — your turn

The whole transpose, as a plain nested Python list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[1, 4], [2, 5], [3, 6]]
```


In [ ]:
import torch as t

def solve(rows):
    """Return a.T's contents as a plain nested list."""
    a = t.tensor(rows)
    return a._____.tolist()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(558)


In [ ]:
#@title 💡 Solution — Problem 558
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return a.T's contents as a plain nested list."""
    a = t.tensor(rows)
    return a.T.tolist()


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


Pulling one row out of a transpose is two steps, and it is worth keeping them
separate the first time: transpose, then read.


In [ ]:
import torch as t

a = t.tensor([[10, 20], [30, 40], [50, 60]])

view = a.T                 # step 1: the tensor, turned on its side
rows = view.tolist()       # step 2: the same numbers as plain Python lists

print("every row's first number:", rows[0])
print("every row's second number:", rows[1])
assert rows[0] == [10, 30, 50]


<!-- dd:dd-q670 -->

### Problem 670 · faded — your turn

The last row of the transpose — the last column of the original.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[3, 6]
```


In [ ]:
import torch as t

def solve(rows):
    """Return the transpose's LAST row as a plain list."""
    a = t.tensor(rows)
    return a._____.tolist()[-1]


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(670)


In [ ]:
#@title 💡 Solution — Problem 670
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return the transpose's LAST row as a plain list."""
    a = t.tensor(rows)
    return a.T.tolist()[-1]


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


<!-- dd:dd-q612 -->

### Problem 612 · independent

Write a function solve(rows) that builds a tensor a and returns a tuple (same_values, shape_after) — whether transposing TWICE gives back a tensor equal to a, and the shape after those two transposes, as a plain tuple. Each transpose swaps the axes, so doing it twice puts them back.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, (2, 3))
```


In [ ]:
import torch as t


def solve(rows):
    """Return (is a.T.T equal to a?, its shape)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(612)


In [ ]:
#@title 💡 Solution — Problem 612
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return (is a.T.T equal to a?, its shape)."""
    a = t.tensor(rows)
    back = a.T.T
    return (t.equal(back, a), tuple(back.shape))


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


<!-- dd:dd-q613 -->

### Problem 613 · independent

Write a function solve(rows) that builds a tensor a and returns a tuple (transposed_values, transposed_shape): the transpose's contents as a plain nested Python list, and its shape as a plain tuple of ints.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[1, 4], [2, 5], [3, 6]], (3, 2))
```


In [ ]:
import torch as t


def solve(rows):
    """Return (the transpose's contents, the transpose's shape)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(613)


In [ ]:
#@title 💡 Solution — Problem 613
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return (the transpose's contents, the transpose's shape)."""
    a = t.tensor(rows)
    view = a.T
    return (view.tolist(), tuple(view.shape))


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


<!-- dd:dd-q671 -->

### Problem 671 · independent

Write a function solve(rows) that builds a tensor a from the nested list and returns one bool: whether a is equal to its own transpose — same shape and the same numbers in every position. Only a square tensor can be, and not every square one is.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
False
```


In [ ]:
import torch as t


def solve(rows):
    """Return whether a equals its own transpose."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2], [3, 4]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(671)


In [ ]:
#@title 💡 Solution — Problem 671
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return whether a equals its own transpose."""
    a = t.tensor(rows)
    return t.equal(a, a.T)


example = ([[1, 2], [3, 4]],)
print(solve(*example))


<!-- dd:dd-q672 -->

### Problem 672 · independent

Write a function solve(rows) that builds a tensor from the nested list (always at least two columns) and returns a tuple (row, count): the SECOND row of its transpose as a plain list, and how many numbers that row holds — which is the original's row count.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([2, 5], 2)
```


In [ ]:
import torch as t


def solve(rows):
    """Return (the transpose's second row, how many numbers it has)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(672)


In [ ]:
#@title 💡 Solution — Problem 672
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return (the transpose's second row, how many numbers it has)."""
    a = t.tensor(rows)
    row = a.T.tolist()[1]
    return (row, len(row))


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


<!-- dd:dd-q673 -->

### Problem 673 · independent

Write a function solve(rows, i, j) that builds a tensor a from the nested list and returns a tuple of two plain numbers: the entry at row i, column j of a, and the entry at row j, column i of a's transpose. Read both out of nested lists.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2, 2)
```


In [ ]:
import torch as t


def solve(rows, i, j):
    """Return (a's entry at row i col j, a.T's entry at row j col i)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2], [3, 4]], 0, 1,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(673)


In [ ]:
#@title 💡 Solution — Problem 673
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, i, j):
    """Return (a's entry at row i col j, a.T's entry at row j col i)."""
    a = t.tensor(rows)
    return (a.tolist()[i][j], a.T.tolist()[j][i])


example = ([[1, 2], [3, 4]], 0, 1,)
print(solve(*example))


<!-- dd:dd-q674 -->

### Problem 674 · independent

Write a function solve(rows) that builds a tensor a from the nested list and returns a tuple of two plain nested lists: the transpose's contents, and the contents after transposing a second time.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[1, 4], [2, 5], [3, 6]], [[1, 2, 3], [4, 5, 6]])
```


In [ ]:
import torch as t


def solve(rows):
    """Return (a.T's values, a.T.T's values), both as nested lists."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(674)


In [ ]:
#@title 💡 Solution — Problem 674
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return (a.T's values, a.T.T's values), both as nested lists."""
    a = t.tensor(rows)
    return (a.T.tolist(), a.T.T.tolist())


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


#### Common mistakes

- **"`.T` is a method, so it needs parentheses."** — It does not. `a.T` is an
  attribute like `a.shape`; `a.T()` raises. The habit to build is that anything
  *describing* a tensor tends to be an attribute, and anything *doing work* on
  one tends to be a call.
- **"Transposing a square tensor changes nothing."** — It changes the numbers,
  it just cannot change the shape. `[[1, 2], [3, 4]]` transposes to
  `[[1, 3], [2, 4]]` while the shape stays `(2, 2)`, so a shape check will not
  notice a transpose you did or did not mean to do.
- **"`a.T` gives me a list of columns."** — It gives you a *tensor*. It reads
  like a list of the original's columns once you call `.tolist()` on it, but
  until then it is a tensor and behaves like one.
- **"Transposing rearranges the numbers, so it must be slow on a big tensor."**
  — It does not touch them at all. What that costs, and how to tell, is the
  next concept.


<!-- dd:dd-kp-numpy-views-and-copies -->

## Views and copies — the same numbers, read a different way

`numpy.views-and-copies`


<!-- dd:dd-seg-numpy-views-and-copies-0 -->

### one strip of numbers, and a note saying how to read it


Your computer does not store a grid. It stores **one long strip of numbers, one
after another**. For `[[1, 2, 3], [4, 5, 6]]` the strip in memory is literally

    1 2 3 4 5 6

and that is all of it. There is nothing two-dimensional anywhere in memory.

The grid comes from a small **note** attached to that strip, saying how to read
it: *"treat this as 2 rows by 3 columns; to step one place right, move forward
1 number; to step one place down, move forward 3."* Follow the note and the
grid comes back. Shape is the first half of that note.

Two words get used constantly from here on, and both of them are about the
strip rather than the grid:

- **"the block"** (or "the buffer", or "the storage") is the strip itself — the
  actual run of numbers in memory. Two tensors *share a block* when both of
  their notes point at the same strip.
- **"in reading order"** — the technical word is **contiguous** — means the
  numbers sit on the strip in exactly the order you would read them off the
  grid, left to right and top to bottom, with no hopping. A tensor you have
  just built is always in reading order: `t.tensor` writes the strip by reading
  the grid.

PyTorch will answer both questions for you. `a.data_ptr()` is *"what address
does my strip start at?"* — a big integer that is meaningless on its own and
tells you everything when you compare two of them. `a.is_contiguous()` is
*"are my numbers in reading order?"*


In [ ]:
import torch as t

a = t.tensor([[1, 2, 3], [4, 5, 6]])

print("a starts at:", type(a.data_ptr()).__name__, "- an address, not a value")
print("a is in reading order:", a.is_contiguous())
print("a compared with itself:", a.data_ptr() == a.data_ptr())




Now the point of all this. Transposing does **not** touch the strip. It writes
a *new note*: *"treat this as 3 rows by 2 columns; to step one place right,
move forward 3; to step one place down, move forward 1."* Same strip, new note.


In [ ]:
view = a.T   # same cell as above — `a` is still the 2x3 tensor

print("a.T starts at the same address:", view.data_ptr() == a.data_ptr())
print("a.T is in reading order:", view.is_contiguous())
print("a.T reads as:", view.tolist())




Both answers together are the whole idea. `a.T` **shares a's block** — it is
reading the very same `1 2 3 4 5 6`. And it is **not in reading order**,
because reading the transposed grid gives `1 4 2 5 3 6`, which is not the order
the strip is written in. It only produces that order by hopping.

A tensor that borrows someone else's block like this is called a **view**.


The two questions are asked with two different calls, and it is worth running
them side by side once on a tensor whose transpose is genuinely out of order.


In [ ]:
import torch as t

a = t.tensor([[1, 2], [3, 4], [5, 6]])
view = a.T

print("strip order of a  :", a.tolist(), "-> reads 1 2 3 4 5 6")
print("grid order of a.T :", view.tolist(), "-> reads 1 3 5 2 4 6")

print("same strip?      ", view.data_ptr() == a.data_ptr())
print("in reading order?", view.is_contiguous())




The freshly built tensor is always in reading order, so the interesting answer
is always the second one:


In [ ]:
print("a itself:", a.is_contiguous())


<!-- dd:dd-q616 -->

### Problem 616 · faded — your turn

Two bools: is `a` in reading order, and is its transpose?

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, False)
```


In [ ]:
import torch as t

def solve(rows):
    """Return (is a in reading order?, is a.T?)."""
    a = t.tensor(rows)
    return (a._____(), a.T._____())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(616)


In [ ]:
#@title 💡 Solution — Problem 616
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return (is a in reading order?, is a.T?)."""
    a = t.tensor(rows)
    return (a.is_contiguous(), a.T.is_contiguous())


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


<!-- dd:dd-seg-numpy-views-and-copies-1 -->

### sharing a strip and being in order are two different questions


It is easy to hear "shares memory" and "is contiguous" as one idea. They are
not, and a transpose is the standard case where they disagree: it shares the
strip *and* is out of order.

Every combination is reachable, so neither answer implies the other:


In [ ]:
import torch as t

a = t.tensor([[1, 2, 3], [4, 5, 6]])

print("a.T      shares:", a.T.data_ptr() == a.data_ptr(),
      " in order:", a.T.is_contiguous())          # shares, out of order

fresh = t.tensor([[1, 2, 3], [4, 5, 6]])
print("a rebuild shares:", fresh.data_ptr() == a.data_ptr(),
      " in order:", fresh.is_contiguous())        # own strip, in order




There is one shape where the transpose *is* already in reading order, and it
catches people out: when the tensor has only one row, or only one column. A
`(1, 4)` tensor transposes to `(4, 1)`, and reading a single column top to
bottom walks the strip straight through — no hopping, so nothing is out of
order.


In [ ]:
strip_like = t.tensor([[1, 2, 3, 4]])
print("one row, transposed:", strip_like.T.tolist())
print("still in reading order:", strip_like.T.is_contiguous())




🔴 **`data_ptr()` is a STARTING address, not the identity of the strip.** For a
transpose the two are the same question, because a transpose starts on the same
number `a` does. A tensor that starts PART-WAY along the strip — a slice — is
still reading `a`'s strip while answering a different starting address:


In [ ]:
later = a[1:]
print("a[1:] starts where a does:", later.data_ptr() == a.data_ptr())
print("...but it is the same strip:",
      later.untyped_storage().data_ptr() == a.untyped_storage().data_ptr())




So `x.data_ptr() == y.data_ptr()` reads as "do these two start at the same
place", and that is exactly the question every drill on this page asks. When
you want "is this the same strip at all", regardless of where each tensor
begins, the question to ask is `x.untyped_storage().data_ptr()`.


Both questions, on the same tensor, in the order a checker would ask them.


In [ ]:
import torch as t

a = t.tensor([[7, 8, 9], [1, 2, 3]])
view = a.T

shares = view.data_ptr() == a.data_ptr()
in_order = view.is_contiguous()

print("(shares, in_order) =", (shares, in_order))
assert shares is True
assert in_order is False




Getting these two the wrong way round is the single most common mistake on this
concept, because both answers are bools and the tuple still looks plausible.


<!-- dd:dd-q617 -->

### Problem 617 · faded — your turn

Two bools about the same transpose: does it start where `a` does, and is it in reading order?

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, False)
```


In [ ]:
import torch as t

def solve(rows):
    """Return (does a.T start where a does?, is a.T in reading order?)."""
    a = t.tensor(rows)
    view = a.T
    return (view._____() == a._____(), view._____())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(617)


In [ ]:
#@title 💡 Solution — Problem 617
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return (does a.T start where a does?, is a.T in reading order?)."""
    a = t.tensor(rows)
    view = a.T
    return (view.data_ptr() == a.data_ptr(), view.is_contiguous())


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


<!-- dd:dd-seg-numpy-views-and-copies-2 -->

### asking for a strip of your own — .contiguous()


Sometimes hopping is not good enough and you want the numbers actually written
out in the order you read them. **`.contiguous()`** is how you ask. The name is
the technical one; what it does is plain enough:

> *"Give me a tensor holding these numbers in reading order."*

For `a.T`, whose grid reads `1 4 2 5 3 6` while its strip says `1 2 3 4 5 6`,
the only way to do that is to write `1 4 2 5 3 6` somewhere else in memory. A
new strip means a new starting address — the result does **not** share `a`'s
block any more. That is the copy.


In [ ]:
import torch as t

a = t.tensor([[1, 2, 3], [4, 5, 6]])
view = a.T
packed = view.contiguous()

print("same numbers as the view:", t.equal(packed, view))
print("in reading order now    :", packed.is_contiguous())
print("still a's block         :", packed.data_ptr() == a.data_ptr())




🔴 **`.contiguous()` does not always copy, and this is the part that surprises
people.** If the tensor is already in reading order there is nothing to do, so
it hands the *same tensor* straight back — same strip, same address. So "a
packed copy never shares the original's block" is false; it is only true when
the thing you packed was actually out of order.


In [ ]:
print("a is already in order:", a.is_contiguous())
print("so a.contiguous() is a itself:", a.contiguous().data_ptr() == a.data_ptr())

one_row = t.tensor([[1, 2, 3, 4]])
print("a one-row transpose is in order too:", one_row.T.is_contiguous())
print("so ITS packed copy shares:",
      one_row.T.contiguous().data_ptr() == one_row.data_ptr())




That is why a drill on this can answer `(True, False)` for a 2×3 input and
`(True, True)` for a `[[9]]` or `[[1, 2]]` input without contradicting itself.


The three-step move — transpose, pack, compare — written out once.


In [ ]:
import torch as t

a = t.tensor([[1, 2], [3, 4], [5, 6]])

view = a.T                  # borrows a's strip, out of order
packed = view.contiguous()  # writes its own strip, in order

print("view   shares:", view.data_ptr() == a.data_ptr(),
      " in order:", view.is_contiguous())
print("packed shares:", packed.data_ptr() == a.data_ptr(),
      " in order:", packed.is_contiguous())
print("same values either way:", t.equal(packed, view))


<!-- dd:dd-q618 -->

### Problem 618 · faded — your turn

Pack two different tensors and ask where each copy starts: `a`'s, then its transpose's.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, False)
```


In [ ]:
import torch as t

def solve(rows):
    """Return (does a's packed copy start where a does?, does a.T's?)."""
    a = t.tensor(rows)
    return (a._____()._____() == a._____(),
            a.T._____()._____() == a._____())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(618)


In [ ]:
#@title 💡 Solution — Problem 618
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Return (does a's packed copy start where a does?, does a.T's?)."""
    a = t.tensor(rows)
    return (a.contiguous().data_ptr() == a.data_ptr(),
            a.T.contiguous().data_ptr() == a.data_ptr())


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


The three-step move — transpose, pack, compare — written out once.


In [ ]:
import torch as t

a = t.tensor([[1, 2], [3, 4], [5, 6]])

view = a.T                  # borrows a's strip, out of order
packed = view.contiguous()  # writes its own strip, in order

print("view   shares:", view.data_ptr() == a.data_ptr(),
      " in order:", view.is_contiguous())
print("packed shares:", packed.data_ptr() == a.data_ptr(),
      " in order:", packed.is_contiguous())
print("same values either way:", t.equal(packed, view))


<!-- dd:dd-q536 -->

### Problem 536 · faded — your turn

Two bools: is the transpose in reading order, and is its packed copy?

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(False, True)
```


In [ ]:
import torch as t

def solve(rows):
    """Return (is a.T in reading order?, is its packed copy?)."""
    a = t.tensor(rows)
    view = a.T
    return (view._____(), view._____()._____())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(536)


In [ ]:
#@title 💡 Solution — Problem 536
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (is a.T in reading order?, is its packed copy?)."""
    a = t.tensor(rows)
    view = a.T
    return (view.is_contiguous(), view.contiguous().is_contiguous())


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


<!-- dd:dd-q553 -->

### Problem 553 · independent

Write a function solve(rows) that builds a tensor a and returns a tuple of three bools: is a itself in reading order, is a.T, and is the packed copy of a.T. A freshly built tensor is laid out contiguously; a transpose is the same block read in a different order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, False, True)
```


In [ ]:
import torch
import torch as t

def solve(rows):
    """Return (a in order?, a.T in order?, packed copy in order?)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(553)


In [ ]:
#@title 💡 Solution — Problem 553
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (a in order?, a.T in order?, packed copy in order?)."""
    a = t.tensor(rows)
    view = a.T
    packed = view.contiguous()
    return (a.is_contiguous(), view.is_contiguous(), packed.is_contiguous())


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


<!-- dd:dd-q523 -->

### Problem 523 · independent

Write a function solve(rows) that takes a nested Python list of at least two equal-length inner lists of at least two integers each, builds the tensor a from it, and returns a tuple (view_shares, copy_shares, values). Take the transpose a.T and then make a contiguous copy of that transpose. `view_shares` is True when the transpose reads the same memory block as a, `copy_shares` is True when the contiguous copy does, and `values` is the transposed data as a nested Python list. Transposing only re-describes the existing block; .contiguous() is how you ask for the copy. Compare buffers with .data_ptr().

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, False, [[1, 4], [2, 5], [3, 6]])
```


In [ ]:
import torch
import torch as t

def solve(rows):
    """Return (does a.T share a's buffer?, does its copy?, the transposed values)."""
    a = t.tensor(rows)
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(523)


In [ ]:
#@title 💡 Solution — Problem 523
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (does a.T share a's buffer?, does its copy?, the transposed values)."""
    a = t.tensor(rows)
    view = a.T
    packed = view.contiguous()
    return (view.data_ptr() == a.data_ptr(),
            packed.data_ptr() == a.data_ptr(),
            packed.tolist())


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


#### Common mistakes

- **"Shares memory" and "is contiguous" are the same question.** — They are
  independent. A transpose shares the block *and* is out of reading order,
  which is exactly the case every drill here is built on.
- **"A packed copy never shares the original's block."** — Only when the thing
  being packed was out of order. `.contiguous()` on a tensor that is already in
  reading order returns that tensor unchanged, address and all, which is why a
  single-row or single-column input answers `True` where a 2×3 answers `False`.
- **"`data_ptr()` tells me something about the values."** — It is an address.
  On its own it means nothing; it is only ever useful compared against another
  tensor's.
- **"`data_ptr()` equal means same block, unequal means different block."** —
  The second half is wrong. It is the address a tensor STARTS at, so a slice
  that begins part-way along someone else's strip answers a different address
  while sharing every byte. `x.untyped_storage().data_ptr()` is the question
  that ignores where each tensor begins.
- **"Transposing is expensive because it moves the numbers."** — It moves
  nothing and costs almost nothing. `.contiguous()` is the call that pays, and
  it pays because it writes a whole new strip.
- **"A view is a copy that happens to look the same."** — The opposite. A view
  is a second note on someone else's strip; write through one and the other
  sees it.


<!-- dd:dd-kp-numpy-constructors -->

## Tensor constructors — zeros, ones, full, eye, *_like

`numpy.constructors`


<!-- dd:dd-seg-numpy-constructors-0 -->

### constructors and the shape argument


`t.tensor` converts data you already have. Just as often you need a tensor
**built from scratch** — a canvas of zeros to fill in, a mask of ones. PyTorch
has one constructor per pattern, and they all share the same calling
convention:

> **constructor(shape, dtype=...)** — say how big, optionally say what type.

- **`t.zeros(shape)`** — all entries `0.0`. The default "empty canvas".
- **`t.ones(shape)`** — all entries `1.0`.
- **`t.full(shape, v)`** — all entries equal to your value `v`.
- **`t.empty(shape)`** — allocates *without initializing* (contents are
  whatever bytes were in memory). Only worth it when you will overwrite
  every entry immediately.

Three of them side by side — same call shape, different fill:


In [ ]:
import torch as t

print(t.zeros((2, 3)))
print(t.ones((2, 3)))
print(t.full((2, 3), 7.0))




PyTorch accepts the shape **either way**: `t.zeros(2, 3)` and `t.zeros((2, 3))`
both give a 2×3 tensor. That is worth noticing precisely because NumPy does
*not* allow it — `np.zeros(2, 3)` is a `TypeError`, since NumPy reads the
second positional argument as the dtype. Code translated from NumPy will use
the tuple form, and it keeps working.


In [ ]:
loose = t.zeros(3, 4)
tupled = t.zeros((3, 4))
assert loose.shape == tupled.shape == (3, 4)
print(loose.shape, "==", tupled.shape)


Build a 3×4 canvas of zeros:


In [ ]:
import torch as t

# Both spellings mean the same thing in PyTorch.
board = t.zeros((3, 4))
same = t.zeros(3, 4)
assert board.shape == (3, 4) == same.shape
print(board)
print("both spellings give", tuple(board.shape), "and", tuple(same.shape))




Why: shape comes first and everything else is keyword-only, so the constructor
never has to guess whether you meant a dimension or a dtype.


<!-- dd:dd-q227 -->

### Problem 227 · faded — your turn

All-zeros float vector of a given length (must also work for length 0).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0., 0., 0., 0.])
```


In [ ]:
import torch as t

def solve(n):
    """Return a 1-D float tensor of n zeros."""
    return t._____(n)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = 4
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(227)


In [ ]:
#@title 💡 Solution — Problem 227
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.zeros(n)


example = 4
print(solve(example))


Build a 3×4 canvas of zeros:


In [ ]:
import torch as t

# Both spellings mean the same thing in PyTorch.
board = t.zeros((3, 4))
same = t.zeros(3, 4)
assert board.shape == (3, 4) == same.shape
print(board)
print("both spellings give", tuple(board.shape), "and", tuple(same.shape))




Why: shape comes first and everything else is keyword-only, so the constructor
never has to guess whether you meant a dimension or a dtype.


<!-- dd:dd-q639 -->

### Problem 639 · faded — your turn

A grid of one value — say how big, then say the value.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[7.0, 7.0, 7.0], [7.0, 7.0, 7.0]]
```


In [ ]:
import torch as t

def solve(rows, cols, v):
    """Return a rows x cols tensor of v, as a nested list."""
    return t._____((rows, cols), v).tolist()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3, 7.0,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(639)


In [ ]:
#@title 💡 Solution — Problem 639
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols, v):
    """Return a rows x cols tensor of v, as a nested list."""
    return t.full((rows, cols), v).tolist()


example = (2, 3, 7.0,)
print(solve(*example))


<!-- dd:dd-seg-numpy-constructors-1 -->

### the dtype is float32 unless you say otherwise


**The default floating dtype is `torch.float32`,** even though the values print
like integers. This is a real difference from NumPy, whose default is
`float64` — the same line of code gives you half the precision here, which is
deliberate, because neural networks are trained in 32-bit.


In [ ]:
import torch as t

default = t.ones(3)
print(default, default.dtype)
assert default.dtype == t.float32




The values printed as `1.` with a trailing dot, and that dot is the whole
warning: they are floats, not the integers they look like.

Pass `dtype=` to override: `t.ones((2, 2), dtype=t.bool)` is a matrix of `True`
(1 as a boolean is `True`); `t.zeros(4, dtype=t.int64)` is integer zeros.
Checking `dtype` right after construction is the habit that catches the
float-by-default surprise before it propagates.


In [ ]:
flags = t.ones((2, 2), dtype=t.bool)
counts = t.zeros(4, dtype=t.int64)
print(flags)
print(counts, counts.dtype)
assert bool(flags.all()) and flags.dtype == t.bool


An all-`True` boolean mask — ones, with the dtype said out loud:


In [ ]:
import torch as t

# ones gives every entry the value 1 — and 1 as a boolean is True.
mask = t.ones((3, 4), dtype=t.bool)
assert bool(mask.all()) and mask.dtype == t.bool
print(mask)

# The default, for contrast — float32, not float64 and not int.
assert t.ones(3).dtype == t.float32
print("asked for bool:", mask.dtype, "| default:", t.ones(3).dtype)




Why: without `dtype=t.bool` this would be a float tensor of 1.0s that merely
*prints* like what you wanted — say the type when it matters.


<!-- dd:dd-q212 -->

### Problem 212 · faded — your turn

A rows×cols tensor where every entry is the boolean `True`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[True, True, True],
        [True, True, True]])
```


In [ ]:
import torch as t

def solve(rows, cols):
    """All-True boolean matrix of shape (rows, cols)."""
    return t._____((rows, cols), _____=_____)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(2, 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(212)


In [ ]:
#@title 💡 Solution — Problem 212
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols):
    return t.ones((rows, cols), dtype=t.bool)


print(solve(2, 3))


An all-`True` boolean mask — ones, with the dtype said out loud:


In [ ]:
import torch as t

# ones gives every entry the value 1 — and 1 as a boolean is True.
mask = t.ones((3, 4), dtype=t.bool)
assert bool(mask.all()) and mask.dtype == t.bool
print(mask)

# The default, for contrast — float32, not float64 and not int.
assert t.ones(3).dtype == t.float32
print("asked for bool:", mask.dtype, "| default:", t.ones(3).dtype)




Why: without `dtype=t.bool` this would be a float tensor of 1.0s that merely
*prints* like what you wanted — say the type when it matters.


<!-- dd:dd-q640 -->

### Problem 640 · faded — your turn

Zeros that are actually integers.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0, 0, 0], 'torch.int64')
```


In [ ]:
import torch as t

def solve(n):
    """Return (n integer zeros as a list, their dtype name)."""
    z = t._____(n, _____=t.int64)
    return (z.tolist(), str(z.dtype))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(640)


In [ ]:
#@title 💡 Solution — Problem 640
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    """Return (n integer zeros as a list, their dtype name)."""
    z = t.zeros(n, dtype=t.int64)
    return (z.tolist(), str(z.dtype))


example = (3,)
print(solve(*example))


<!-- dd:dd-seg-numpy-constructors-2 -->

### *_like — copy shape AND dtype from an existing tensor


The **`*_like` variants** (`t.zeros_like(x)`, `t.ones_like(x)`,
`t.full_like(x, v)`) copy both the shape *and the dtype* from an existing
tensor — the right tool whenever the question is "give me a blank tensor
shaped like this one". Reaching for `*_like` is both shorter and safer than
reading off `.shape` and `.dtype` yourself, and in real model code it also
carries across the device the original lives on.


In [ ]:
import torch as t

x = t.tensor([[3, -1, 4], [1, 5, -9]], dtype=t.int32)
print("zeros_like:", t.zeros_like(x).dtype)
print("zeros(x.shape):", t.zeros(x.shape).dtype)




Same shape from both, but only one of them still knows the tensor was
integer. Every `*_like` behaves that way:


In [ ]:
print(t.ones_like(x))
print(t.full_like(x, 7))
assert t.full_like(x, 7).dtype == x.dtype == t.int32


In [ ]:
import torch as t

# "Blank tensor shaped like x" — zeros_like copies shape AND dtype,
# so an int32 input yields an int32 result, not the float default.
x = t.tensor([[3, -1, 4], [1, 5, -9]], dtype=t.int32)
blank = t.zeros_like(x)
assert blank.shape == x.shape
assert blank.dtype == t.int32
print(blank)
print("copied dtype:", blank.dtype, "| t.zeros(x.shape) would give:",
      t.zeros(x.shape).dtype)




Why: `t.zeros(x.shape)` would lose the dtype (float default) — `_like`
keeps both properties in one call.


<!-- dd:dd-q41 -->

### Problem 41 · faded — your turn

Blank tensor matching BOTH the shape and dtype of an existing tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 0, 0],
        [0, 0, 0]], dtype=torch.int32)
```


In [ ]:
import torch as t

def solve(x):
    """Return an all-zeros tensor with x's shape and x's dtype."""
    return t._____(x)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[3, -1, 4], [1, 5, -9]], dtype=t.int32)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(41)


In [ ]:
#@title 💡 Solution — Problem 41
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.zeros_like(x)


example = t.tensor([[3, -1, 4], [1, 5, -9]], dtype=t.int32)
print(solve(example))


In [ ]:
import torch as t

# "Blank tensor shaped like x" — zeros_like copies shape AND dtype,
# so an int32 input yields an int32 result, not the float default.
x = t.tensor([[3, -1, 4], [1, 5, -9]], dtype=t.int32)
blank = t.zeros_like(x)
assert blank.shape == x.shape
assert blank.dtype == t.int32
print(blank)
print("copied dtype:", blank.dtype, "| t.zeros(x.shape) would give:",
      t.zeros(x.shape).dtype)




Why: `t.zeros(x.shape)` would lose the dtype (float default) — `_like`
keeps both properties in one call.


<!-- dd:dd-q641 -->

### Problem 641 · faded — your turn

Shaped and typed like x, filled with v.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([-1, -1, -1], 'torch.int32')
```


In [ ]:
import torch as t

def solve(x, v):
    """Return (a tensor of v shaped AND typed like x, its dtype name)."""
    filled = t._____(x, v)
    return (filled.tolist(), str(filled.dtype))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1, 2, 3], dtype=t.int32), -1,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(641)


In [ ]:
#@title 💡 Solution — Problem 641
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, v):
    """Return (a tensor of v shaped AND typed like x, its dtype name)."""
    filled = t.full_like(x, v)
    return (filled.tolist(), str(filled.dtype))


example = (t.tensor([1, 2, 3], dtype=t.int32), -1,)
print(solve(*example))


<!-- dd:dd-seg-numpy-constructors-3 -->

### t.eye — the identity matrix


**`t.eye(n)`** is the n×n identity matrix: `1.0` on the main diagonal,
`0.0` elsewhere. It's the seed for anything diagonal-shaped: `v * t.eye(n)`
puts a constant v on the diagonal, and indexing its rows with a permutation
turns it into a permutation matrix (a trick you will use in the random-number
KP).


In [ ]:
import torch as t

print(t.eye(3))
print(5.0 * t.eye(3))




The permutation trick, since it is the least obvious one — reordering the
identity's ROWS builds the matrix that reorders a vector the same way:


In [ ]:
order = t.tensor([2, 0, 1])
P = t.eye(3)[order]
v = t.tensor([10.0, 20.0, 30.0])
print(P)
print(P @ v)
assert (P @ v).tolist() == [30.0, 10.0, 20.0]


In [ ]:
import torch as t

I = t.eye(3)
assert I.tolist() == [[1.0, 0.0, 0.0],
                      [0.0, 1.0, 0.0],
                      [0.0, 0.0, 1.0]]
print(I)




Why: the identity is a constructor, not something you assemble by loop —
and scaling it (`v * t.eye(n)`) is the one-liner for "v on the diagonal".


<!-- dd:dd-q228 -->

### Problem 228 · faded — your turn

The n×n identity matrix.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
```


In [ ]:
import torch as t

def solve(n):
    """n-by-n identity matrix."""
    return t._____(n)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = 3
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(228)


In [ ]:
#@title 💡 Solution — Problem 228
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.eye(n)


example = 3
print(solve(example))


In [ ]:
import torch as t

I = t.eye(3)
assert I.tolist() == [[1.0, 0.0, 0.0],
                      [0.0, 1.0, 0.0],
                      [0.0, 0.0, 1.0]]
print(I)




Why: the identity is a constructor, not something you assemble by loop —
and scaling it (`v * t.eye(n)`) is the one-liner for "v on the diagonal".


<!-- dd:dd-q642 -->

### Problem 642 · faded — your turn

The identity, and its shape.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[1.0, 0.0], [0.0, 1.0]], (2, 2))
```


In [ ]:
import torch as t

def solve(n):
    """Return (the n x n identity as a nested list, its shape)."""
    I = t._____(n)
    return (I.tolist(), tuple(I.shape))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(642)


In [ ]:
#@title 💡 Solution — Problem 642
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    """Return (the n x n identity as a nested list, its shape)."""
    I = t.eye(n)
    return (I.tolist(), tuple(I.shape))


example = (2,)
print(solve(*example))


<!-- dd:dd-q48 -->

### Problem 48 · guided

Write a function solve(v, fill) that takes a 1-D PyTorch integer tensor v and an integer fill value. It should return a tensor of the same length in which every element at an odd index (indices 1, 3, 5, ...) has been replaced by fill, while every element at an even index keeps its original value. Do not use Python loops, and leave the caller's tensor v unmodified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 3, -1,  2, -1,  5, -1])
```


<details>
<summary>Hints</summary>

1. Nothing is being constructed from scratch here — you need a COPY of v
   that you are allowed to write into.
2. Odd indices are a slice with a step, and assigning a scalar into a
   slice broadcasts it across every selected position.
3. `out = v.clone()` then `out[1::2] = fill`. Skipping the clone mutates
   the caller's tensor, which the drill checks for.

</details>


In [ ]:
import torch as t

def solve(v, fill):
    """Return a copy of v with every odd index replaced by fill."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([3, 8, 2, 7, 5, 1])
print(solve(example, -1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(48)


In [ ]:
#@title 💡 Solution — Problem 48
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v, fill):
    out = v.clone()
    out[1::2] = fill
    return out


example = t.tensor([3, 8, 2, 7, 5, 1])
print(solve(example, -1))


<!-- dd:dd-q225 -->

### Problem 225 · independent

Write a function solve(n) that takes a non-negative integer n and returns a 1-D PyTorch tensor of length n in which every entry is 1.0. The result must use torch's default floating-point dtype, not integers. Note that torch's default float is float32, where numpy's is float64 — the grader checks the dtype as well as the values.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 1., 1., 1.])
```


In [ ]:
import torch as t

def solve(n):
    """Return a 1-D float tensor of length n where every entry is 1.0."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = 4
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(225)


In [ ]:
#@title 💡 Solution — Problem 225
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.ones(n)


example = 4
print(solve(example))


<!-- dd:dd-q50 -->

### Problem 50 · independent

Write a function solve(n, v) that takes an integer n >= 1 and a number v, and returns an n x n floating-point PyTorch tensor with v at every position on the main diagonal and 0.0 everywhere else.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2., 0., 0., 0.],
        [0., 2., 0., 0.],
        [0., 0., 2., 0.],
        [0., 0., 0., 2.]])
```


In [ ]:
import torch as t

def solve(n, v):
    """Return an n x n float tensor with v on the main diagonal, 0.0 elsewhere."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(4, 2.0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(50)


In [ ]:
#@title 💡 Solution — Problem 50
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, v):
    return t.eye(n) * v


print(solve(4, 2.0))


<!-- dd:dd-q213 -->

### Problem 213 · independent

Write a function solve(n, idx) that takes a length n and a valid index idx, and returns a 1-D float PyTorch tensor of n zeros except for a 1.0 at position idx — a standard basis (one-hot) vector.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])
```


In [ ]:
import torch as t

def solve(n, idx):
    """Return a length-n float tensor that is 1.0 at idx and 0.0 elsewhere."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(10, 4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(213)


In [ ]:
#@title 💡 Solution — Problem 213
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, idx):
    z = t.zeros(n)
    z[idx] = 1.0
    return z


print(solve(10, 4))


<!-- dd:dd-q643 -->

### Problem 643 · independent

Write a function solve(rows, cols) that returns a tuple (values, dtype_name): a rows x cols tensor of boolean False, as a nested list, and str() of its dtype. Zeros of the right type, not floats that print like them.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[False, False, False], [False, False, False]], 'torch.bool')
```


In [ ]:
import torch as t


def solve(rows, cols):
    """Return (an all-False rows x cols mask as a list, its dtype name)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(643)


In [ ]:
#@title 💡 Solution — Problem 643
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols):
    """Return (an all-False rows x cols mask as a list, its dtype name)."""
    mask = t.zeros((rows, cols), dtype=t.bool)
    return (mask.tolist(), str(mask.dtype))


example = (2, 3,)
print(solve(*example))


<!-- dd:dd-q644 -->

### Problem 644 · independent

Write a function solve(x) that returns a tuple (values, count, dtype_name): a tensor of ones with x's shape and x's dtype as a nested list, how many elements it has, and str() of its dtype.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[1, 1, 1], [1, 1, 1]], 6, 'torch.int64')
```


In [ ]:
import torch as t


def solve(x):
    """Return (ones shaped like x, their count, their dtype name)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1, 2, 3], [4, 5, 6]]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(644)


In [ ]:
#@title 💡 Solution — Problem 644
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (ones shaped like x, their count, their dtype name)."""
    o = t.ones_like(x)
    return (o.tolist(), o.numel(), str(o.dtype))


example = (t.tensor([[1, 2, 3], [4, 5, 6]]),)
print(solve(*example))


<!-- dd:dd-q645 -->

### Problem 645 · independent

Write a function solve(n) that returns a tuple (values, dtype_name): the n-by-n identity matrix stored as 64-bit INTEGERS (1 and 0, not 1.0 and 0.0), as a nested list, and str() of its dtype.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[1, 0], [0, 1]], 'torch.int64')
```


In [ ]:
import torch as t


def solve(n):
    """Return (the n x n identity as int64, its dtype name)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(645)


In [ ]:
#@title 💡 Solution — Problem 645
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    """Return (the n x n identity as int64, its dtype name)."""
    I = t.eye(n, dtype=t.int64)
    return (I.tolist(), str(I.dtype))


example = (2,)
print(solve(*example))


#### Common mistakes

- **"Constructors give me integers if I write `t.ones(5)`."** — The default
  dtype is `torch.float32` regardless of how the values look. If the grader (or
  your model) needs ints or bools, say so with `dtype=`.
- **"float is float — precision doesn't change when I port NumPy code."** —
  `np.ones(4)` is float64 and `t.ones(4)` is float32. Exact equality checks
  written against NumPy output can fail here for no reason other than the
  narrower dtype.
- **"`dtype=bool` works, like in NumPy."** — PyTorch wants its own dtype
  objects: `t.bool`, `t.int64`, `t.float32`. Python's builtin `bool` and `int`
  are accepted in some places but `t.*` is the spelling to learn.
- **"`t.empty` means a tensor with no elements."** — It means *uninitialized
  memory* of the full requested shape: garbage values, not zeros, not empty.
  Use `t.zeros` unless you will overwrite everything.


<!-- dd:dd-kp-numpy-slicing-views -->

## Slicing, views, and slice assignment

`numpy.slicing-views`


Slicing is how you name a rectangular piece of a tensor. The syntax
generalizes Python's list slicing in two ways, and adds one semantic twist
that trips everyone at least once.

**Syntax.** A slice is `start:stop:step` (stop exclusive, any part omittable),
and a multi-dimensional tensor takes **one slice per axis, separated by
commas** inside a single pair of brackets:

- `x[2:5]` — elements 2, 3, 4 of a vector.
- `z[0, :]` — row 0, all columns. `z[:, -1]` — every row, last column.
- `x[::2]` — every second element.

Negative *indices* count from the end (`-1` is the last element), exactly as in
Python.


In [ ]:
import torch as t

x = t.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
print("x        ", x)
print("x[2:5]   ", x[2:5])
print("x[::2]   ", x[::2])
print("x[-3:]   ", x[-3:])




One slice per axis, comma-separated — and note what an *int* in a slot does
that a slice does not: it removes that axis.


In [ ]:
z = t.tensor([[0, 1, 2, 3],
              [4, 5, 6, 7],
              [8, 9, 10, 11]])
print(z)
print("z[0, :]  ", z[0, :],  " shape", z[0, :].shape)     # int -> 1-D
print("z[:, -1] ", z[:, -1], " shape", z[:, -1].shape)
print("z[0:1, :]", z[0:1, :]," shape", z[0:1, :].shape)   # slice -> stays 2-D




**Negative *steps* are the exception.** NumPy reverses an axis with `x[::-1]`;
PyTorch refuses — it raises `ValueError: step must be greater than zero`. This
is probably the single most common surprise when moving NumPy habits to torch.
Reversal has its own function:

- **`t.flip(x, [0])`** — reverse along axis 0. `t.flip(z, [1])` mirrors each
  row left-right; `t.flip(z, [0])` reverses the row order (mirror top-bottom).
- **`t.rot90(z)`** — rotate 90° counterclockwise, the composition of a
  transpose and a flip.


In [ ]:
try:
    x[::-1]
except ValueError as err:
    print("ValueError:", err)

print("flip axis 0", t.flip(x, [0]))
print("mirror rows left-right")
print(t.flip(z, [1]))
print("rot90")
print(t.rot90(z))
assert t.equal(t.rot90(z), t.flip(z.T, [0]))




**The twist: slices are *views*, not copies.** A slice doesn't copy data — it
is a new window onto the *same* memory block. Two consequences:

1. **Writing through a slice writes the original.** That enables the single
   most useful idiom in this KP, **slice assignment**:
   `x[start:stop] = value` sets a whole range at once (the scalar is
   broadcast to every selected position — no loop).
2. **"Return a new tensor" tasks need an explicit `.clone()`** if you would
   otherwise be returning or mutating a view of the caller's data. Rule of
   thumb: mutate → `.clone()` first, unless the task says to modify in place.

`t.flip` is not in that category: it always returns a **copy**, so writing into
its result never touches the input.


In [ ]:
data = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
window = data[1:3]
window[0] = 99.0                 # window[0] IS data[1]
print("data after writing through the view:", data)
assert data[1].item() == 99.0

safe = data[1:3].clone()
safe[0] = -1.0
print("data after writing through the clone:", data)
assert data[1].item() == 99.0




Slice assignment is the same fact used deliberately — a whole range set at
once, the scalar broadcast across every selected position, no loop:


In [ ]:
y = t.zeros(6)
y[1:4] = 5.0
y[::2] = -1.0
print(y)
assert y.tolist() == [-1.0, 5.0, -1.0, 5.0, -1.0, 0.0]


Task: given a vector, produce a reversed copy; then blank out the middle of
another vector in place.


In [ ]:
import torch as t

x = t.tensor([1.0, 2.0, 3.0, 4.0])

# The NumPy reflex does not work here.
try:
    x[::-1]
    raised = False
except ValueError:
    raised = True
assert raised, "torch rejects negative slice steps"

print("x[::-1] raised ValueError:", raised)

# Reverse with flip instead — and flip hands back a COPY.
rev = t.flip(x, [0])
assert rev.tolist() == [4.0, 3.0, 2.0, 1.0]
rev[0] = 99.0                    # writes only into rev
assert x.tolist() == [1.0, 2.0, 3.0, 4.0]
print("wrote 99 into the flipped COPY:", rev, "-> x is still", x)

# A plain slice, by contrast, IS a view: writing through it writes x.
window = x[1:3]
window[0] = 99.0                 # window[0] is x[1]!
assert x[1] == 99.0
print("wrote 99 through a VIEW:       ", window, "-> x is now  ", x)
x[1] = 2.0                       # undo

# Slice ASSIGNMENT: set positions 1..3 (stop 4 exclusive) to 0 — in place,
# no loop. The scalar 0.0 is broadcast across the selected range.
y = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
y[1:4] = 0.0
assert y.tolist() == [1.0, 0.0, 0.0, 0.0, 5.0, 6.0]
print("after y[1:4] = 0:              ", y)




Why each step:

1. The failed `x[::-1]` is worth writing once deliberately. It is the fastest
   way to stop reaching for it by reflex later.
2. `t.flip(x, dims)` names the axes to reverse as a list — you pick *which*
   axis by what you put in that list, the same choice you would have made by
   which comma slot got the `::-1`.
3. The view demonstration is the mental model to keep: a slice is a window,
   not a photocopy. Cheap to make, dangerous to mutate casually.
4. Slice assignment replaces the `for i in range(start, stop)` loop entirely —
   and it is the building block for border/checkerboard/striping patterns in
   the next lesson.


<!-- dd:dd-q233 -->

### Problem 233 · faded — your turn

Reversed copy of a 1-D tensor (input unmodified).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([4., 3., 2., 1.])
```


In [ ]:
import torch as t

def solve(x):
    """Return a new tensor with x's elements in reverse order."""
    return t._____(x, [0])


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.0, 2.0, 3.0, 4.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(233)


In [ ]:
#@title 💡 Solution — Problem 233
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.flip(x, [0])


example = t.tensor([1.0, 2.0, 3.0, 4.0])
print(solve(example))


<!-- dd:dd-q76 -->

### Problem 76 · guided

Write a function solve(z) that takes a 2-D PyTorch tensor and returns a tuple of two new tensors: first, z mirrored left-right (each row reversed); second, z mirrored top-bottom (the order of the rows reversed). PyTorch does not support negative-step slicing, so reach for the flip operation and pick the right axis for each. The input must not be modified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([[2, 1, 0],
        [5, 4, 3]]), tensor([[3, 4, 5],
        [0, 1, 2]]))
```


<details>
<summary>Hints</summary>

1. Two mirrors of a 2-D tensor: left-right (reverse within each row) and
   top-bottom (reverse the order of rows). Both are single `t.flip` calls.
2. `t.flip` takes the axes to reverse as a list. Which axis number reverses
   each row? Which reverses the row order?
3. `t.flip(z, [1])` and `t.flip(z, [0])` — and because flip copies, the
   "input must not be modified" requirement is already satisfied.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return (z mirrored left-right, z mirrored top-bottom)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(6).reshape(2, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(76)


In [ ]:
#@title 💡 Solution — Problem 76
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.flip(z, [1]), t.flip(z, [0])


example = t.arange(6).reshape(2, 3)
print(solve(example))


<!-- dd:dd-q506 -->

### Problem 506 · guided

Write a function solve(x, k) that takes a 1-D tensor and an int, and returns the first k elements. A slice with no start means 'from the beginning'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 2, 3])
```


<details>
<summary>Hints</summary>

1. A slice, not an index — you want a run of elements, not one element.
2. Leaving the start empty means 'from the beginning'.
3. `x[:k]`.

</details>


In [ ]:
import torch as t

def solve(x, k):
    """Return the first k elements."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1, 2, 3, 4, 5]), 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(506)


In [ ]:
#@title 💡 Solution — Problem 506
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, k):
    """Return the first k elements."""
    return x[:k]


example = (t.tensor([1, 2, 3, 4, 5]), 3)
print(solve(*example))


<!-- dd:dd-q231 -->

### Problem 231 · independent

Write a function solve(x, start, stop, value) that takes a 1-D PyTorch tensor of floats x, two integer indices start and stop, and a float value. Set every entry of x from index start up to (but not including) index stop to value, leave every entry outside that range unchanged, and return the tensor. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 0., 0., 0., 5., 6.])
```


In [ ]:
import torch as t

def solve(x, start, stop, value):
    """Set x[start:stop] to value in place and return x."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
print(solve(example, 1, 4, 0.0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(231)


In [ ]:
#@title 💡 Solution — Problem 231
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, start, stop, value):
    x[start:stop] = value
    return x


example = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
print(solve(example, 1, 4, 0.0))


<!-- dd:dd-q75 -->

### Problem 75 · independent

Write a function solve(z) that takes a 2-D PyTorch tensor and returns it rotated 90 degrees counterclockwise: the last column of the input becomes the first row of the output. An input of shape (r, c) produces output of shape (c, r).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2, 5],
        [1, 4],
        [0, 3]])
```


In [ ]:
import torch as t

def solve(z):
    """Rotate the 2-D tensor z by 90 degrees counterclockwise."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(6).reshape(2, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(75)


In [ ]:
#@title 💡 Solution — Problem 75
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.rot90(z)


example = t.arange(6).reshape(2, 3)
print(solve(example))


<!-- dd:dd-q507 -->

### Problem 507 · independent

Write a function solve(x, col) that returns a tuple (values, shape) for column `col` of the 2-D tensor x: the column's values as a plain list, and its shape as a plain tuple. Indexing an axis with a plain int REMOVES that axis, so the result is 1-D, not a column-shaped 2-D tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([2, 5], (2,))
```


In [ ]:
import torch as t

def solve(x, col):
    """Return one column of a 2-D tensor, and its shape."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1, 2, 3], [4, 5, 6]]), 1)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(507)


In [ ]:
#@title 💡 Solution — Problem 507
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, col):
    """Return one column of a 2-D tensor, and its shape."""
    c = x[:, col]
    return (c.tolist(), tuple(c.shape))


example = (t.tensor([[1, 2, 3], [4, 5, 6]]), 1)
print(solve(*example))


<!-- dd:dd-q74 -->

### Problem 74 · independent

Write a function solve(z, step, v) that takes a 1-D PyTorch tensor z, a positive integer step, and a value v. Return a new tensor equal to z except that every step-th element — indices 0, step, 2*step, ... — has been overwritten with v. The input tensor must not be modified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([-1,  1,  2, -1,  4,  5, -1,  7,  8, -1, 10, 11])
```


In [ ]:
import torch as t

def solve(z, step, v):
    """Return a copy of z with every step-th element replaced by v."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(12)
print(solve(example, 3, -1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(74)


In [ ]:
#@title 💡 Solution — Problem 74
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, step, v):
    out = z.clone()
    out[::step] = v
    return out


example = t.arange(12)
print(solve(example, 3, -1))


<!-- dd:dd-q189 -->

### Problem 189 · independent

Write a function solve(z, k) that takes a 2-D PyTorch tensor and an integer k (possibly 0 or larger than 3), and returns z rotated counterclockwise by k quarter-turns (k=1 is 90 degrees CCW; k is taken modulo 4). Do not modify the input.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2, 5],
        [1, 4],
        [0, 3]])
```


In [ ]:
import torch as t

def solve(z, k):
    """Rotate z counterclockwise by k quarter-turns."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.arange(6).reshape(2, 3), 1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(189)


In [ ]:
#@title 💡 Solution — Problem 189
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, k):
    return t.rot90(z, k)


print(solve(t.arange(6).reshape(2, 3), 1))


#### Common mistakes

- **"`x[::-1]` reverses a tensor."** — It raises `ValueError: step must be
  greater than zero`. PyTorch supports no negative slice steps at all;
  `t.flip(x, [0])` is the operation.
- **"A slice is a copy."** — It's a view of the same memory. Mutating a slice
  mutates the original. When a task says "return a new tensor" or "do not
  modify the input" and you plan to write into the result, `.clone()` first.
- **"`.copy()` makes the copy."** — That's the NumPy name. Tensors clone with
  `.clone()`.
- **"2-D indexing is `z[i][j]`."** — That works but chains two operations;
  the idiom is one bracket, comma-separated: `z[i, j]`, `z[i, :]`, `z[:, j]`.
  The chained form also breaks down for slice-then-assign patterns.


<!-- dd:dd-kp-numpy-ranges -->

## Numeric ranges — arange and linspace

`numpy.ranges`


<!-- dd:dd-seg-numpy-ranges-0 -->

### t.arange — the stop is exclusive


When you know the **step**, use **`t.arange(start, stop, step)`** (step
defaults to 1). It counts from `start` in increments of `step` and — exactly
like Python's `range` — **stops BEFORE `stop`**. The endpoint is never
included, even with a step: `t.arange(0, 10, 2)` is `[0, 2, 4, 6, 8]` — 10
is left out.


In [ ]:
import torch as t

print(t.arange(5))
print(t.arange(0, 10, 2))          # 10 is the stop, so 10 is missing
print(t.arange(0, 11, 2))          # push the stop past it to get it back




That exclusive stop is the whole trick, and the source of most range bugs.
When a task wants the endpoint *included*, you have to extend the stop past
where you want to end.

`t.arange` over integers gives you an **integer** tensor (`int64`), which
matters because integer tensors are what you index with.


In [ ]:
idx = t.arange(3)
letters = t.tensor([10, 20, 30, 40])
print(idx.dtype, "->", letters[idx])
assert idx.dtype == t.int64


In [ ]:
import torch as t

# Counting by 2 up to 10 — but 10 is the stop, so it's EXCLUDED.
evens = t.arange(0, 10, 2)
assert evens.tolist() == [0, 2, 4, 6, 8]
assert evens.dtype == t.int64
print(evens, evens.dtype, "  <- no 10")




Why: notice 10 never appears. The step doesn't change the rule — `arange`
always halts one step short of `stop`.


<!-- dd:dd-q229 -->

### Problem 229 · faded — your turn

Every integer from `start` to `end`, **including both endpoints**. (Watch the
exclusive stop — how do you make `end` appear?)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([3, 4, 5, 6, 7, 8])
```


In [ ]:
import torch as t

def solve(start, end):
    """Integers start..end inclusive, in order."""
    return t._____(start, _____)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example_start, example_end = 3, 8
print(solve(example_start, example_end))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(229)


In [ ]:
#@title 💡 Solution — Problem 229
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(start, end):
    return t.arange(start, end + 1)


example_start, example_end = 3, 8
print(solve(example_start, example_end))


<!-- dd:dd-seg-numpy-ranges-1 -->

### t.linspace — you know the number of points


When you know the **number of points** instead of the step, use
**`t.linspace(start, stop, num)`**: exactly `num` evenly spaced values, and
this time **both endpoints are included**. Mind the fencepost — `num` points
make `num − 1` gaps, so `t.linspace(0.0, 1.0, 5)` has step 1/4, not 1/5.


In [ ]:
import torch as t

grid = t.linspace(0.0, 1.0, 5)
print(grid)
print("gaps:", grid[1:] - grid[:-1])   # 5 points, so 4 of them
assert grid[0].item() == 0.0 and grid[-1].item() == 1.0




Both ends present, four gaps between five points. Side by side with `arange`
the two conventions are hard to confuse again:


In [ ]:
print("arange  ", t.arange(0.0, 1.0, 0.25))     # stop excluded -> 4 values
print("linspace", t.linspace(0.0, 1.0, 5))      # stop included -> 5 values
assert len(t.arange(0.0, 1.0, 0.25)) == 4
assert len(t.linspace(0.0, 1.0, 5)) == 5


In [ ]:
import torch as t

# 5 points from 0 to 1, endpoints INCLUDED -> 4 equal gaps of 0.25.
grid = t.linspace(0.0, 1.0, 5)
assert grid.tolist() == [0.0, 0.25, 0.5, 0.75, 1.0]
print(grid, " 5 points,", len(grid) - 1, "gaps")




Why: both 0.0 and 1.0 are present — that's the opposite of `arange`. Count
the points, not the intervals.


<!-- dd:dd-q242 -->

### Problem 242 · faded — your turn

The `n` evenly spaced breakpoints **strictly inside** (0, 1) — exclude 0.0 and
1.0. (linspace includes the endpoints; how do you get `n` points *between*
them?)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.2500, 0.5000, 0.7500])
```


In [ ]:
import torch as t

def solve(n):
    """n interior breakpoints of (0, 1), endpoints excluded."""
    return t._____(0.0, 1.0, n + _____)[1:-1]


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = 3
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(242)


In [ ]:
#@title 💡 Solution — Problem 242
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.linspace(0.0, 1.0, n + 2)[1:-1]


example = 3
print(solve(example))


<!-- dd:dd-seg-numpy-ranges-2 -->

### float steps drift — count, then scale


`t.arange` with a **float** step is a trap: each element is built by repeated
addition, so rounding error accumulates and the endpoint may or may not show
up. It is a sharper trap here than in NumPy, because the default float is
32-bit and has fewer digits to lose.

The robust recipe is to turn the step-question into a count-question:
figure out how many points there are, generate exact **integers**, and scale
them once — `t.arange(n_points) * step`. One multiply per element, no drift.


In [ ]:
import torch as t

drifty = t.arange(0.0, 1.0 + 0.1, 0.1)
print(drifty)
print("last value:", drifty[-1].item(), "| point count:", len(drifty))




The last entry is not the clean `1.0` the call asked for, and whether an
eleventh point appears at all is decided by rounding error. Counting first
removes the gamble:


In [ ]:
n = int(round(1.0 / 0.1)) + 1
exact = t.arange(n) * 0.1
print(exact)
print("point count:", len(exact))
assert len(exact) == 11




The count uses the exclusive-stop insight again: an inclusive range of
`step`-spaced points from 0 to `stop` has `round(stop / step) + 1` of them.


In [ ]:
import torch as t

# 0 to 1 inclusive, spacing 0.25. Count the points, scale integers.
n = int(round(1.0 / 0.25)) + 1        # 5 points: 0, 0.25, 0.5, 0.75, 1.0
grid = t.arange(n) * 0.25
assert grid.tolist() == [0.0, 0.25, 0.5, 0.75, 1.0]
print("n =", n, "->", grid)




Why: `t.arange(0, 1.0 + 0.25, 0.25)` would gamble on the endpoint;
`t.arange(n) * step` is exact because the integers are exact.


<!-- dd:dd-q214 -->

### Problem 214 · faded — your turn

`solve(stop, step)`: the inclusive float range 0, step, 2·step, …, up to **and
including** `stop` (an exact multiple of `step`). (Why does the point count
need a `+ 1`?)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 0.0000,  0.5000,  1.0000,  1.5000,  2.0000,  2.5000,  3.0000,  3.5000,
         4.0000,  4.5000,  5.0000,  5.5000,  6.0000,  6.5000,  7.0000,  7.5000,
         8.0000,  8.5000,  9.0000,  9.5000, 10.0000])
```


In [ ]:
import torch as t

def solve(stop, step):
    """0, step, ..., stop inclusive — exact, no float drift."""
    n = int(round(stop / step)) + _____
    return t._____(n) * step


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(10.0, 0.5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(214)


In [ ]:
#@title 💡 Solution — Problem 214
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(stop, step):
    n = int(round(stop / step)) + 1
    return t.arange(n) * step


print(solve(10.0, 0.5))


<!-- dd:dd-q524 -->

### Problem 524 · guided

Write a function solve(high, low) that takes two integers with high >= low and returns a 1-D integer tensor counting DOWN from high to low with BOTH endpoints included: high, high - 1, ..., low. Use a single t.arange call with a negative step. The stop is exclusive when counting down too, so decide which side of low the stop has to sit on.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([5, 4, 3, 2, 1])
```


<details>
<summary>Hints</summary>

1. The exclusive stop does not care which way you are counting. Going down,
   "stops before the stop" means the stop has to sit one step PAST `low`.
2. A negative step reverses the direction: `t.arange(high, ?, -1)`. Ask what
   `?` makes `low` the last value actually produced.
3. `t.arange(high, low - 1, -1)` — the `- 1` is the inclusive-endpoint fix
   from q229, pointed the other way.

</details>


In [ ]:
import torch
import torch as t

def solve(high, low):
    """Integers from high down to low, both endpoints included."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (5, 1)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(524)


In [ ]:
#@title 💡 Solution — Problem 524
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(high, low):
    """Integers from high down to low, both endpoints included."""
    return t.arange(high, low - 1, -1)


example = (5, 1)
print(solve(*example))


<!-- dd:dd-q53 -->

### Problem 53 · independent

Write a function solve(n) that takes an integer n >= 2 and returns a 1-D floating-point PyTorch tensor of n evenly spaced values that starts at exactly 0.0 and ends at exactly 1.0, with equal spacing between consecutive entries.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
```


In [ ]:
import torch as t

def solve(n):
    """Return n evenly spaced floats from 0.0 to 1.0 inclusive."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(53)


In [ ]:
#@title 💡 Solution — Problem 53
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.linspace(0.0, 1.0, n)


print(solve(5))


#### Common mistakes

- **"`t.arange(3, 8)` includes 8."** — Like Python's `range`, the stop is
  exclusive. Endpoint bugs from this are the most common range mistake; when
  a task says "inclusive", plan the `+ step` (or switch to `linspace`).
- **"Float steps in arange are fine."** — Each element is built by repeated
  float addition, so the endpoint may or may not appear and interior values
  drift. Scale exact integers (`t.arange(n) * step`) or use `linspace`.
- **"`linspace(0, 1, 5)` has step 1/5."** — It has step 1/4: five points means
  FOUR gaps. `linspace` counts points, not intervals.
- **"`t.arange(5)` gives floats."** — Integer arguments give an `int64` tensor.
  That is what you want for indexing; if you need floats, ask for them
  (`t.arange(5.0)` or `dtype=t.float32`).


<!-- dd:dd-kp-numpy-dtype-astype -->

## Dtypes, .to(), and memory size

`numpy.dtype-astype`


<!-- dd:dd-seg-numpy-dtype-astype-0 -->

### .to() — converting an existing tensor


Every tensor has exactly one **dtype** — the type shared by all of its
elements. It determines what the values can be: `int64` can't hold 0.5;
`float32` holds it with less precision than `float64`; `bool` holds only
`True`/`False`. Mixed inputs get promoted to the common type (ints + one
float → all float), silently.

To convert an existing tensor: **`x.to(new_dtype)`** (this is PyTorch's
`astype`). It returns a *new* tensor when the dtype actually changes — the
original is untouched; there is no in-place dtype change. Converting
float→int **truncates toward zero** rather than rounding: `1.9 → 1`,
`-1.9 → -1`.


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])
y = x.to(t.float32)
print(x, x.dtype)
print(y, y.dtype)
assert x.dtype == t.int64          # the original never changed




Truncation is the part that surprises people — it is not rounding:


In [ ]:
print(t.tensor([1.9, -1.9, 0.5]).to(t.int64))
assert t.tensor([1.9]).to(t.int64).item() == 1
assert t.tensor([-1.9]).to(t.int64).item() == -1




One wrinkle worth knowing: if the dtype you ask for is the one it already
has, `.to()` hands back the *same* tensor rather than a copy. It promises a
tensor of that dtype, not a fresh buffer.


In [ ]:
same = x.to(t.int64)
print("same object?", same is x)
assert same is x


Make a float32 copy of an integer tensor:


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])
assert x.dtype == t.int64           # default integer dtype

# .to() returns a NEW tensor with converted values; x is untouched.
y = x.to(t.float32)
assert y.dtype == t.float32
assert x.dtype == t.int64           # original unchanged — .to() copies
print("x", x, x.dtype)
print("y", y, y.dtype)




Why: checking `x.dtype` first tells you what conversion is actually needed —
don't convert blind. And dtype is a property of the whole memory block, so
changing it means building a new block.


<!-- dd:dd-q230 -->

### Problem 230 · faded — your turn

Float32 copy of an integer tensor, original left unmodified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 2., 3.])
```


In [ ]:
import torch as t

def solve(x):
    """Return a float32 copy of integer tensor x."""
    return x._____(t.float32)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1, 2, 3])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(230)


In [ ]:
#@title 💡 Solution — Problem 230
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.to(t.float32)


example = t.tensor([1, 2, 3])
print(solve(example))


Make a float32 copy of an integer tensor:


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])
assert x.dtype == t.int64           # default integer dtype

# .to() returns a NEW tensor with converted values; x is untouched.
y = x.to(t.float32)
assert y.dtype == t.float32
assert x.dtype == t.int64           # original unchanged — .to() copies
print("x", x, x.dtype)
print("y", y, y.dtype)




Why: checking `x.dtype` first tells you what conversion is actually needed —
don't convert blind. And dtype is a property of the whole memory block, so
changing it means building a new block.


<!-- dd:dd-q649 -->

### Problem 649 · faded — your turn

Float to integer — the fraction is cut, not rounded.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[1, -1, 0]
```


In [ ]:
import torch as t

def solve(x):
    """Return x converted to int64, as a plain list."""
    return x._____(t.int64).tolist()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.9, -1.9, 0.5]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(649)


In [ ]:
#@title 💡 Solution — Problem 649
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return x converted to int64, as a plain list."""
    return x.to(t.int64).tolist()


example = (t.tensor([1.9, -1.9, 0.5]),)
print(solve(*example))


<!-- dd:dd-seg-numpy-dtype-astype-1 -->

### dtypes at creation, and dtype names as strings


Rather than build-then-convert, request the dtype at creation: every
constructor accepts `dtype=`, e.g. `t.arange(n, dtype=t.int32)` — one
step, no copy.


In [ ]:
import torch as t

built = t.arange(4, dtype=t.int32)
print(built, built.dtype)




Here PyTorch is stricter than NumPy. NumPy accepts the *string* `'float32'`
anywhere a dtype is wanted; PyTorch does not — `t.arange(3, dtype='float32')`
raises a `TypeError`. Watch it happen:


In [ ]:
try:
    t.arange(3, dtype='float32')
except TypeError as err:
    print("TypeError:", err)




When the dtype arrives as **data** (from a config, a file header, a function
argument), you have to turn the name into the dtype object first, and
`getattr(t, name)` does exactly that.


In [ ]:
for name in ('int32', 'float32', 'float64'):
    z = t.arange(3, dtype=getattr(t, name))
    print(f"{name:8} -> {z} {z.dtype}")
assert t.arange(3, dtype=getattr(t, 'float32')).dtype == t.float32


Build 0..2 already in float32 — dtype named by a string:


In [ ]:
import torch as t

name = 'float32'

# PyTorch wants the dtype OBJECT, so look it up from the name.
z = t.arange(3, dtype=getattr(t, name))
assert z.dtype == t.float32
print(repr(name), "->", getattr(t, name), "->", z)




Why: when a function receives `dtype_str` as an argument, one `getattr` turns
it into the real dtype — no lookup table from strings to torch objects, and no
`if/elif` chain over dtype names.


<!-- dd:dd-q215 -->

### Problem 215 · faded — your turn

The integers 0..n-1 stored with a dtype named by a string.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0, 1, 2, 3, 4], dtype=torch.int32)
```


In [ ]:
import torch as t

def solve(n, dtype_str):
    """0..n-1 with the dtype named by dtype_str (e.g. 'int32')."""
    return t.arange(n, _____=_____(t, dtype_str))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(5, "int32"))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(215)


In [ ]:
#@title 💡 Solution — Problem 215
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, dtype_str):
    return t.arange(n, dtype=getattr(t, dtype_str))


print(solve(5, "int32"))


Build 0..2 already in float32 — dtype named by a string:


In [ ]:
import torch as t

name = 'float32'

# PyTorch wants the dtype OBJECT, so look it up from the name.
z = t.arange(3, dtype=getattr(t, name))
assert z.dtype == t.float32
print(repr(name), "->", getattr(t, name), "->", z)




Why: when a function receives `dtype_str` as an argument, one `getattr` turns
it into the real dtype — no lookup table from strings to torch objects, and no
`if/elif` chain over dtype names.


<!-- dd:dd-q650 -->

### Problem 650 · faded — your turn

A dtype that arrives as a string.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1, 1, 1], 'torch.int32')
```


In [ ]:
import torch as t

def solve(n, name):
    """Return (n ones stored as the dtype named `name`, that dtype's name)."""
    o = t.ones(n, _____=_____(t, name))
    return (o.tolist(), str(o.dtype))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 'int32',)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(650)


In [ ]:
#@title 💡 Solution — Problem 650
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, name):
    """Return (n ones stored as the dtype named `name`, that dtype's name)."""
    o = t.ones(n, dtype=getattr(t, name))
    return (o.tolist(), str(o.dtype))


example = (3, 'int32',)
print(solve(*example))


<!-- dd:dd-seg-numpy-dtype-astype-2 -->

### dtype is a memory choice — element_size and numel


The number in a dtype's name is bits: `int32` = 4 bytes per element,
`float64` = 8. Total buffer size is `elements × bytes-per-element` —
**`x.numel()`** elements at **`x.element_size()`** bytes each. Dtype choices
are memory choices: halving precision halves the buffer, which is the whole
reason models train in float32 (or bfloat16) rather than float64.


In [ ]:
import torch as t

grid = t.zeros((10, 10))
for dtype in (t.float64, t.float32, t.int16, t.bool):
    z = grid.to(dtype)
    print(f"{str(dtype):15} {z.numel()} x {z.element_size()} = "
          f"{z.numel() * z.element_size()} bytes")




Same 100 numbers, an 8× spread in what they cost:


In [ ]:
assert grid.to(t.float64).element_size() == 2 * grid.to(t.float32).element_size()
assert grid.to(t.float32).numel() * grid.to(t.float32).element_size() == 400
print("float64 is exactly", grid.to(t.float64).element_size(), "bytes/element,",
      "float32", grid.to(t.float32).element_size())
print("100 float32 elements =",
      grid.to(t.float32).numel() * grid.to(t.float32).element_size(), "bytes")


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])             # int64: 8 bytes each
assert x.numel() * x.element_size() == 3 * 8

# float32 elements are 4 bytes, so the converted copy is half the size.
y = x.to(t.float32)
assert y.element_size() == 4
assert y.numel() * y.element_size() == 3 * 4
print(x.dtype, x.numel() * x.element_size(), "bytes")
print(y.dtype, y.numel() * y.element_size(), "bytes")




Why: `numel × element_size` makes the cost concrete — a 10×10 float32
tensor is exactly 400 bytes, where the float64 NumPy equivalent is 800.


<!-- dd:dd-q51 -->

### Problem 51 · faded — your turn

A tensor's buffer size, reported as the string "&lt;n&gt; bytes".

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
400 bytes
```


In [ ]:
import torch as t

def solve(z):
    """Return z's data-buffer size as e.g. '24 bytes'."""
    return f"{z.numel() * z._____()} bytes"


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.zeros((10, 10))
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(51)


In [ ]:
#@title 💡 Solution — Problem 51
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return f"{z.numel() * z.element_size()} bytes"


example = t.zeros((10, 10))
print(solve(example))


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])             # int64: 8 bytes each
assert x.numel() * x.element_size() == 3 * 8

# float32 elements are 4 bytes, so the converted copy is half the size.
y = x.to(t.float32)
assert y.element_size() == 4
assert y.numel() * y.element_size() == 3 * 4
print(x.dtype, x.numel() * x.element_size(), "bytes")
print(y.dtype, y.numel() * y.element_size(), "bytes")




Why: `numel × element_size` makes the cost concrete — a 10×10 float32
tensor is exactly 400 bytes, where the float64 NumPy equivalent is 800.


<!-- dd:dd-q651 -->

### Problem 651 · faded — your turn

What one element costs, and what they all cost together.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(4, 24)
```


In [ ]:
import torch as t

def solve(x):
    """Return (bytes per element, total bytes) for x."""
    return (x._____(), x.numel() * x._____())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.zeros((2, 3)),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(651)


In [ ]:
#@title 💡 Solution — Problem 651
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (bytes per element, total bytes) for x."""
    return (x.element_size(), x.numel() * x.element_size())


example = (t.zeros((2, 3)),)
print(solve(*example))


<!-- dd:dd-q87 -->

### Problem 87 · guided

Write a function solve(z, bins) that takes a 1-D PyTorch tensor of floats in [0, 1) and a positive integer bins, and returns a 1-D INTEGER tensor of length bins counting how many values fall into each of the bins equal-width intervals dividing [0, 1]. torch's histogram counter returns floats, so converting the dtype is part of the exercise.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 0, 2, 0, 1])
```


<details>
<summary>Hints</summary>

1. Torch has a histogram counter that takes the bin count and the range
   directly — no bucketing by hand.
2. It returns FLOAT counts. The drill wants integers, so the last step is
   a dtype conversion.
3. `t.histc(z, bins=bins, min=0.0, max=1.0).to(t.int64)` — this is the
   dtype lesson in miniature: the numbers were already right, only their
   type was wrong.

</details>


In [ ]:
import torch as t

def solve(z, bins):
    """Return integer counts of z's values across `bins` equal-width bins over [0, 1]."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([0.1, 0.5, 0.55, 0.9])
print(solve(example, 5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(87)


In [ ]:
#@title 💡 Solution — Problem 87
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, bins):
    counts = t.histc(z, bins=bins, min=0.0, max=1.0)
    return counts.to(t.int64)


example = t.tensor([0.1, 0.5, 0.55, 0.9])
print(solve(example, 5))


<!-- dd:dd-q19 -->

### Problem 19 · independent

Write a function solve(z) that takes a 1-D complex-valued PyTorch tensor and returns a tuple of two real-valued tensors of the same length: the first holding the real part of each entry, the second holding the imaginary part.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([1., 3.]), tensor([2., 4.]))
```


In [ ]:
import torch as t

def solve(z):
    """Split a complex tensor into its real and imaginary parts."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1 + 2j, 3 + 4j])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(19)


In [ ]:
#@title 💡 Solution — Problem 19
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.real, z.imag


example = t.tensor([1 + 2j, 3 + 4j])
print(solve(example))


<!-- dd:dd-q652 -->

### Problem 652 · independent

Write a function solve(x) that returns a tuple (values, new_dtype, old_dtype): a float64 copy of x as a plain list, str() of the copy's dtype, and str() of x's own dtype afterwards — which the conversion must not have changed.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1.0, 2.0, 3.0], 'torch.float64', 'torch.int64')
```


In [ ]:
import torch as t


def solve(x):
    """Return (x as float64, the copy's dtype name, the original's dtype name)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1, 2, 3]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(652)


In [ ]:
#@title 💡 Solution — Problem 652
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (x as float64, the copy's dtype name, the original's dtype name)."""
    y = x.to(t.float64)
    return (y.tolist(), str(y.dtype), str(x.dtype))


example = (t.tensor([1, 2, 3]),)
print(solve(*example))


<!-- dd:dd-q653 -->

### Problem 653 · independent

Write a function solve(n, name) that builds the integers 0 through n-1 already stored as the dtype named by the string `name` (no build-then-convert), and returns a tuple (values, per_element): the values as a plain list and how many bytes each element takes.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0, 1, 2, 3], 2)
```


In [ ]:
import torch as t


def solve(n, name):
    """Return (0..n-1 built as dtype `name`, bytes per element)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (4, 'int16',)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(653)


In [ ]:
#@title 💡 Solution — Problem 653
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, name):
    """Return (0..n-1 built as dtype `name`, bytes per element)."""
    z = t.arange(n, dtype=getattr(t, name))
    return (z.tolist(), z.element_size())


example = (4, 'int16',)
print(solve(*example))


<!-- dd:dd-q654 -->

### Problem 654 · independent

Write a function solve(x, name) that returns the number of bytes x's data would occupy if stored as the dtype named by the string `name`: convert, then multiply element count by bytes per element.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
800
```


In [ ]:
import torch as t


def solve(x, name):
    """Return how many bytes x occupies once stored as the dtype named `name`."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.zeros((10, 10)), 'float64',)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(654)


In [ ]:
#@title 💡 Solution — Problem 654
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, name):
    """Return how many bytes x occupies once stored as the dtype named `name`."""
    y = x.to(getattr(t, name))
    return y.numel() * y.element_size()


example = (t.zeros((10, 10)), 'float64',)
print(solve(*example))


<!-- dd:dd-q655 -->

### Problem 655 · independent

Write a function solve(x) that takes a float tensor, converts it to 64-bit integers and then back to float32, and returns a tuple (values, dtype_name): the round-tripped values as a plain list and str() of their dtype. The fractions are gone; the type is float again.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1.0, 2.0, -3.0], 'torch.float32')
```


In [ ]:
import torch as t


def solve(x):
    """Return (x → int64 → float32 values, the result's dtype name)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.5, 2.0, -3.75]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(655)


In [ ]:
#@title 💡 Solution — Problem 655
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (x → int64 → float32 values, the result's dtype name)."""
    back = x.to(t.int64).to(t.float32)
    return (back.tolist(), str(back.dtype))


example = (t.tensor([1.5, 2.0, -3.75]),)
print(solve(*example))


<!-- dd:dd-q656 -->

### Problem 656 · independent

Write a function solve(x, name) that converts x to the dtype named by the string `name` and returns a tuple (values, dtype_name): the converted values as a plain list and str() of their dtype.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1, -1], 'torch.int64')
```


In [ ]:
import torch as t


def solve(x, name):
    """Return (x converted to the dtype named `name`, its dtype name)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.75, -1.75]), 'int64',)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(656)


In [ ]:
#@title 💡 Solution — Problem 656
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, name):
    """Return (x converted to the dtype named `name`, its dtype name)."""
    y = x.to(getattr(t, name))
    return (y.tolist(), str(y.dtype))


example = (t.tensor([1.75, -1.75]), 'int64',)
print(solve(*example))


#### Common mistakes

- **"`.to()` changes the tensor in place."** — It returns a converted copy;
  the original keeps its dtype and values. If you meant to keep it, assign it:
  `x = x.to(...)`.
- **"Float→int conversion rounds."** — It truncates toward zero: `1.9 → 1`,
  `-1.9 → -1`. If you want rounding, round first: `x.round().to(t.int64)`.
- **"`dtype='float32'` works, like in NumPy."** — PyTorch rejects the string
  with a `TypeError`. Pass `t.float32`, or `getattr(t, name)` when the name
  arrives as data.
- **"dtype names are just labels."** — The number is the bit width, which sets
  both the representable range/precision and the memory per element.
  `numel × element_size` — a 10×10 float32 tensor is exactly 400 bytes.


<!-- dd:dd-kp-numpy-reshape-flatten -->

## Reshape, flatten, and element order

`numpy.reshape-flatten`


<!-- dd:dd-seg-numpy-reshape-flatten-0 -->

### reshape re-describes the same run of numbers


A tensor's data is one flat run of numbers in memory; the shape is a note
saying how to cut that run into rows. **Reshaping rewrites the note without
touching the run** — which is why it is usually free (no copy) and why the one
hard rule is:

> the new shape must account for exactly the same number of elements
> (`2 × 6 = 12 = 3 × 4` ✓, but 12 → `(5, 3)` ✗ raises an error).

The spelling is **`x.reshape(shape)`**, and the shape can be given loose
(`x.reshape(3, 4)`) or as a tuple (`x.reshape((3, 4))`):


In [ ]:
import torch as t

x = t.arange(12)
print(x)
print(x.reshape(3, 4))
print("(2, 6) ", x.reshape(2, 6).shape)
print("(3, 4) ", x.reshape((3, 4)).shape)




The count rule is not a guideline. A mismatch raises rather than padding or
truncating:


In [ ]:
try:
    x.reshape(5, 3)                      # 15 != 12
except RuntimeError as err:
    print("RuntimeError:", err)




One dimension may be **`-1`**, meaning "work this one out for me":
`x.reshape(3, -1)` fixes three rows and lets PyTorch derive the columns. It
only ever *derives* a length — it cannot invent elements — and only one `-1`
is allowed per call.


In [ ]:
print("(3, -1) ->", tuple(x.reshape(3, -1).shape))
print("(-1, 2) ->", tuple(x.reshape(-1, 2).shape))
assert x.reshape(3, -1).shape == x.reshape(3, 4).shape


Task: build the classic "counting matrix" — an n×n tensor containing 0..n²-1
reading left-to-right, top-to-bottom.


In [ ]:
import torch as t

n = 3
# Step 1: make the flat run 0..8. It's 1-D — shape (9,).
flat = t.arange(n * n)

# Step 2: reshape to (3, 3). The first three numbers become row 0, the next
# three row 1 — exactly the "reading order" the task describes. No data
# is copied; only the note about the shape changed.
grid = flat.reshape(n, n)
assert grid.tolist() == [[0, 1, 2], [3, 4, 5], [6, 7, 8]]

# The -1 shortcut: "3 rows, you work out the columns."
assert grid.tolist() == t.arange(9).reshape(3, -1).tolist()
print(grid)




Why each step:

1. `arange` + `reshape` is the standard two-step for "matrix containing the
   numbers 0..k in reading order" — generate the flat values, then cut them.
2. `-1` earns its keep when one dimension is derived: you state the part you
   know and let PyTorch check the arithmetic.


<!-- dd:dd-q46 -->

### Problem 46 · faded — your turn

n×n matrix containing 0..n²-1 in reading order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [3, 4, 5],
        [6, 7, 8]])
```


In [ ]:
import torch as t

def solve(n):
    """Return the n x n matrix of 0..n*n-1 in row-major reading order."""
    return t.arange(_____)._____(_____, _____)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = 3
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(46)


In [ ]:
#@title 💡 Solution — Problem 46
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.arange(n * n).reshape(n, n)


example = 3
print(solve(example))


Task: build the classic "counting matrix" — an n×n tensor containing 0..n²-1
reading left-to-right, top-to-bottom.


In [ ]:
import torch as t

n = 3
# Step 1: make the flat run 0..8. It's 1-D — shape (9,).
flat = t.arange(n * n)

# Step 2: reshape to (3, 3). The first three numbers become row 0, the next
# three row 1 — exactly the "reading order" the task describes. No data
# is copied; only the note about the shape changed.
grid = flat.reshape(n, n)
assert grid.tolist() == [[0, 1, 2], [3, 4, 5], [6, 7, 8]]

# The -1 shortcut: "3 rows, you work out the columns."
assert grid.tolist() == t.arange(9).reshape(3, -1).tolist()
print(grid)




Why each step:

1. `arange` + `reshape` is the standard two-step for "matrix containing the
   numbers 0..k in reading order" — generate the flat values, then cut them.
2. `-1` earns its keep when one dimension is derived: you state the part you
   know and let PyTorch check the arithmetic.


<!-- dd:dd-q36 -->

### Problem 36 · faded — your turn

A 1-D run cut into a 3-D shape whose product matches its length.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[ 0.,  1.,  2.],
         [ 3.,  4.,  5.]],

        [[ 6.,  7.,  8.],
         [ 9., 10., 11.]]])
```


In [ ]:
import torch as t

def solve(a, shape):
    """Return a re-described with the given 3-D shape."""
    return a._____(shape)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(12.0)
print(solve(example, (2, 2, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(36)


In [ ]:
#@title 💡 Solution — Problem 36
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, shape):
    return a.reshape(shape)


example = t.arange(12.0)
print(solve(example, (2, 2, 3)))


Task: build the classic "counting matrix" — an n×n tensor containing 0..n²-1
reading left-to-right, top-to-bottom.


In [ ]:
import torch as t

n = 3
# Step 1: make the flat run 0..8. It's 1-D — shape (9,).
flat = t.arange(n * n)

# Step 2: reshape to (3, 3). The first three numbers become row 0, the next
# three row 1 — exactly the "reading order" the task describes. No data
# is copied; only the note about the shape changed.
grid = flat.reshape(n, n)
assert grid.tolist() == [[0, 1, 2], [3, 4, 5], [6, 7, 8]]

# The -1 shortcut: "3 rows, you work out the columns."
assert grid.tolist() == t.arange(9).reshape(3, -1).tolist()
print(grid)




Why each step:

1. `arange` + `reshape` is the standard two-step for "matrix containing the
   numbers 0..k in reading order" — generate the flat values, then cut them.
2. `-1` earns its keep when one dimension is derived: you state the part you
   know and let PyTorch check the arithmetic.


<!-- dd:dd-q491 -->

### Problem 491 · faded — your turn

You know the column count. Do not compute the rows — let -1 do it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [3, 4, 5]])
```


In [ ]:
import torch as t

def solve(x, cols):
    """Return x reshaped to `cols` columns, inferring the row count."""
    return x._____(_____, cols)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.arange(6), 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(491)


In [ ]:
#@title 💡 Solution — Problem 491
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, cols):
    """Return x reshaped to `cols` columns, inferring the row count."""
    return x.reshape(-1, cols)


example = (t.arange(6), 3)
print(solve(*example))


Task: build the classic "counting matrix" — an n×n tensor containing 0..n²-1
reading left-to-right, top-to-bottom.


In [ ]:
import torch as t

n = 3
# Step 1: make the flat run 0..8. It's 1-D — shape (9,).
flat = t.arange(n * n)

# Step 2: reshape to (3, 3). The first three numbers become row 0, the next
# three row 1 — exactly the "reading order" the task describes. No data
# is copied; only the note about the shape changed.
grid = flat.reshape(n, n)
assert grid.tolist() == [[0, 1, 2], [3, 4, 5], [6, 7, 8]]

# The -1 shortcut: "3 rows, you work out the columns."
assert grid.tolist() == t.arange(9).reshape(3, -1).tolist()
print(grid)




Why each step:

1. `arange` + `reshape` is the standard two-step for "matrix containing the
   numbers 0..k in reading order" — generate the flat values, then cut them.
2. `-1` earns its keep when one dimension is derived: you state the part you
   know and let PyTorch check the arithmetic.


<!-- dd:dd-q619 -->

### Problem 619 · faded — your turn

The same run of numbers, re-described as `rows` rows of `cols`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0, 1, 2], [3, 4, 5]]
```


In [ ]:
import torch as t

def solve(x, rows, cols):
    """Return x re-described as (rows, cols), as a plain nested list."""
    return x._____(rows, cols).tolist()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.arange(6), 2, 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(619)


In [ ]:
#@title 💡 Solution — Problem 619
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, rows, cols):
    """Return x re-described as (rows, cols), as a plain nested list."""
    return x.reshape(rows, cols).tolist()


example = (t.arange(6), 2, 3,)
print(solve(*example))


<!-- dd:dd-seg-numpy-reshape-flatten-1 -->

### flatten, and the order the numbers come out in


**`x.flatten()`** is the other direction: any shape back to one axis. Both
operations answer the same question — **in what order do elements fill a
shape?** — and PyTorch's answer is **row-major** ("C order"): the *last* axis
moves fastest. Reading a 2-D tensor in row-major order means walking across
row 0 left to right, then row 1, and so on.


In [ ]:
import torch as t

z = t.arange(6).reshape(2, 3)
print(z)
print("flatten ->", z.flatten())
assert z.flatten().tolist() == [0, 1, 2, 3, 4, 5]




The same rule explains higher-dimensional shapes: `reshape(2, 2, 3)` fills the
last axis (length 3) fastest and the first axis slowest.


In [ ]:
print(t.arange(12).reshape(2, 2, 3))




Read that print bottom-up: the innermost brackets (length 3) count by one, so
that axis moves fastest; the outermost changes only once.

Column-major ("Fortran") order — first axis fastest — has **no keyword** in
PyTorch. NumPy spells it `order='F'`; here you get it by swapping the axes
first and then flattening: `z.T.flatten()` reads the matrix column by column.


In [ ]:
print("row-major   ", z.flatten())
print("column-major", z.T.flatten())
assert z.T.flatten().tolist() == [0, 3, 1, 4, 2, 5]


Task: take the counting matrix apart again, both ways.


In [ ]:
import torch as t

grid = t.arange(9).reshape(3, 3)

# Step 1: flatten undoes the reshape — the row-major walk gives 0..8 back.
assert grid.flatten().tolist() == [0, 1, 2, 3, 4, 5, 6, 7, 8]

# Step 2: the column-major walk reads DOWN each column instead. No order=
# keyword exists, so transpose first and let the row-major walk do the work.
assert grid.T.flatten().tolist() == [0, 3, 6, 1, 4, 7, 2, 5, 8]
print(grid)
print("row-major   ", grid.flatten())
print("column-major", grid.T.flatten())




Why: "flatten" is not one operation until you say the order. The default
matches how the matrix prints; the column-by-column read is a transpose away.


<!-- dd:dd-q490 -->

### Problem 490 · faded — your turn

Any shape in, exactly one axis out.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 2, 3, 4, 5, 6])
```


In [ ]:
import torch as t

def solve(x):
    """Return x collapsed to a single axis."""
    return x._____()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [4, 5, 6]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(490)


In [ ]:
#@title 💡 Solution — Problem 490
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return x collapsed to a single axis."""
    return x.flatten()


example = t.tensor([[1, 2, 3], [4, 5, 6]])
print(solve(example))


Task: take the counting matrix apart again, both ways.


In [ ]:
import torch as t

grid = t.arange(9).reshape(3, 3)

# Step 1: flatten undoes the reshape — the row-major walk gives 0..8 back.
assert grid.flatten().tolist() == [0, 1, 2, 3, 4, 5, 6, 7, 8]

# Step 2: the column-major walk reads DOWN each column instead. No order=
# keyword exists, so transpose first and let the row-major walk do the work.
assert grid.T.flatten().tolist() == [0, 3, 6, 1, 4, 7, 2, 5, 8]
print(grid)
print("row-major   ", grid.flatten())
print("column-major", grid.T.flatten())




Why: "flatten" is not one operation until you say the order. The default
matches how the matrix prints; the column-by-column read is a transpose away.


<!-- dd:dd-q620 -->

### Problem 620 · faded — your turn

The k-th number in reading order — flatten, then index.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
5
```


In [ ]:
import torch as t

def solve(x, k):
    """Return the k-th number of x in reading order (row-major)."""
    return x._____()[k].item()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1, 2, 3], [4, 5, 6]]), 4,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(620)


In [ ]:
#@title 💡 Solution — Problem 620
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, k):
    """Return the k-th number of x in reading order (row-major)."""
    return x.flatten()[k].item()


example = (t.tensor([[1, 2, 3], [4, 5, 6]]), 4,)
print(solve(*example))


Task: take the counting matrix apart again, both ways.


In [ ]:
import torch as t

grid = t.arange(9).reshape(3, 3)

# Step 1: flatten undoes the reshape — the row-major walk gives 0..8 back.
assert grid.flatten().tolist() == [0, 1, 2, 3, 4, 5, 6, 7, 8]

# Step 2: the column-major walk reads DOWN each column instead. No order=
# keyword exists, so transpose first and let the row-major walk do the work.
assert grid.T.flatten().tolist() == [0, 3, 6, 1, 4, 7, 2, 5, 8]
print(grid)
print("row-major   ", grid.flatten())
print("column-major", grid.T.flatten())




Why: "flatten" is not one operation until you say the order. The default
matches how the matrix prints; the column-by-column read is a transpose away.


<!-- dd:dd-q621 -->

### Problem 621 · faded — your turn

Two reads of one grid: across the rows, then down the columns.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0, 1, 2, 3, 4, 5], [0, 3, 1, 4, 2, 5])
```


In [ ]:
import torch as t

def solve(n, rows):
    """Return (row-major read, column-major read) of the counting matrix, as lists."""
    g = t.arange(n)._____(rows, -1)
    return (g._____().tolist(), g.T._____().tolist())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (6, 2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(621)


In [ ]:
#@title 💡 Solution — Problem 621
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, rows):
    """Return (row-major read, column-major read) of the counting matrix, as lists."""
    g = t.arange(n).reshape(rows, -1)
    return (g.flatten().tolist(), g.T.flatten().tolist())


example = (6, 2,)
print(solve(*example))


<!-- dd:dd-seg-numpy-reshape-flatten-2 -->

### reshape is a view of the same memory


Since reshape only rewrites the note, the result usually shares the original's
memory — it is a **view**. Writing into the view writes the original:


In [ ]:
import torch as t

base = t.zeros(6)
grid = base.reshape(2, 3)
grid[0, 0] = 9.0
print("grid:", grid)
print("base:", base)
assert base[0].item() == 9.0




That sharing is the whole reason reshape is free, and it is also the thing to
remember when a "copy" turns out not to be one. When the run cannot be
re-described in place — a transposed tensor, say, whose numbers are not in
reading order — `reshape` quietly copies instead. You will also meet
**`x.view(shape)`**, reshape's stricter sibling: it *only* ever re-labels the
existing memory and raises if that is impossible. Prefer `reshape` unless you
specifically want the error.

A round trip changes nothing: flatten, then reshape back to the old shape, and
you have the same numbers in the same arrangement.


In [ ]:
z = t.arange(6).reshape(2, 3)
back = z.flatten().reshape(z.shape)
print(back)
assert t.equal(back, z)


Task: prove the view shares memory, then prove the round trip is exact.


In [ ]:
import torch as t

x = t.tensor([[1, 2], [3, 4]])

# Step 1: a reshaped view of a tensor in reading order shares its memory.
flat = x.reshape(-1)
flat[0] = 99
assert x.tolist() == [[99, 2], [3, 4]]      # the write landed in x

# Step 2: the round trip — flatten, reshape back — is the same tensor.
again = x.flatten().reshape(x.shape)
assert t.equal(again, x)
print("x after the write:", x.tolist())
print("round trip equal: ", t.equal(again, x))




Why: step 1 is the difference between reshape and a copy, and it is what
makes a stray write into a "temporary" reshaped tensor show up in the original.


<!-- dd:dd-q622 -->

### Problem 622 · faded — your turn

Write through the reshaped view and read the original.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[1, 2], [3, 9]]
```


In [ ]:
import torch as t

def solve(x, v):
    """Reshape x to 1-D, write v into the LAST slot of the result, return x's values."""
    flat = x._____(-1)
    flat[-1] = v
    return x.tolist()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1, 2], [3, 4]]), 9,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(622)


In [ ]:
#@title 💡 Solution — Problem 622
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, v):
    """Reshape x to 1-D, write v into the LAST slot of the result, return x's values."""
    flat = x.reshape(-1)
    flat[-1] = v
    return x.tolist()


example = (t.tensor([[1, 2], [3, 4]]), 9,)
print(solve(*example))


Task: prove the view shares memory, then prove the round trip is exact.


In [ ]:
import torch as t

x = t.tensor([[1, 2], [3, 4]])

# Step 1: a reshaped view of a tensor in reading order shares its memory.
flat = x.reshape(-1)
flat[0] = 99
assert x.tolist() == [[99, 2], [3, 4]]      # the write landed in x

# Step 2: the round trip — flatten, reshape back — is the same tensor.
again = x.flatten().reshape(x.shape)
assert t.equal(again, x)
print("x after the write:", x.tolist())
print("round trip equal: ", t.equal(again, x))




Why: step 1 is the difference between reshape and a copy, and it is what
makes a stray write into a "temporary" reshaped tensor show up in the original.


<!-- dd:dd-q623 -->

### Problem 623 · faded — your turn

Flatten, then reshape back — the round trip changes nothing.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, (2, 3))
```


In [ ]:
import torch as t

def solve(x):
    """Flatten x and reshape it back; return (equal to x?, shape of the round trip)."""
    back = x._____()._____(x.shape)
    return (t.equal(back, x), tuple(back.shape))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1, 2, 3], [4, 5, 6]]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(623)


In [ ]:
#@title 💡 Solution — Problem 623
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Flatten x and reshape it back; return (equal to x?, shape of the round trip)."""
    back = x.flatten().reshape(x.shape)
    return (t.equal(back, x), tuple(back.shape))


example = (t.tensor([[1, 2, 3], [4, 5, 6]]),)
print(solve(*example))


<!-- dd:dd-q23 -->

### Problem 23 · independent

Write a function solve(z) that takes a 2-D PyTorch tensor and returns a 1-D tensor listing all of z's entries in column-major order: read down the first column top to bottom, then the second column, and so on. Note that this is not the default row-major flattening, and unlike numpy, torch's flatten has no order argument — so change the axis order first.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0, 3, 1, 4, 2, 5])
```


In [ ]:
import torch as t

def solve(z):
    """Flatten z in column-major order."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(6).reshape(2, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(23)


In [ ]:
#@title 💡 Solution — Problem 23
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.T.flatten()


example = t.arange(6).reshape(2, 3)
print(solve(example))


<!-- dd:dd-q492 -->

### Problem 492 · independent

Write a function solve(n, rows) that returns a 2-D tensor holding the integers 0 through n-1 in order, arranged into `rows` rows. n is always divisible by rows. Build the run of values first, then re-describe its shape.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [3, 4, 5]])
```


In [ ]:
import torch as t

def solve(n, rows):
    """Return 0..n-1 laid out with `rows` rows."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (6, 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(492)


In [ ]:
#@title 💡 Solution — Problem 492
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, rows):
    """Return 0..n-1 laid out with `rows` rows."""
    return t.arange(n).reshape(rows, -1)


example = (6, 2)
print(solve(*example))


<!-- dd:dd-q493 -->

### Problem 493 · independent

Write a function solve(x) that returns a tuple (values, shape). `values` is a plain Python list of x's elements in row-major order; `shape` is x's shape as a plain tuple of ints. Together these are everything you would need to rebuild x — which is the whole point of reshape being free: the values never move.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1, 2, 3, 4, 5, 6], (2, 3))
```


In [ ]:
import torch as t

def solve(x):
    """Return (flattened copy, shape before flattening)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [4, 5, 6]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(493)


In [ ]:
#@title 💡 Solution — Problem 493
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (flattened copy, shape before flattening)."""
    return (x.flatten().tolist(), tuple(x.shape))


example = t.tensor([[1, 2, 3], [4, 5, 6]])
print(solve(example))


<!-- dd:dd-q494 -->

### Problem 494 · independent

Write a function solve(x) that takes a CONTIGUOUS 2-D integer tensor — one built directly, not a transpose or other rearranged view — reshapes it to 1-D, writes 99 into the first element of the RESULT, and returns a tuple (result_values, x_values_now) with both as plain lists. Reshaping a contiguous tensor hands back a view over the same memory, so watch what happens to x. The contiguity restriction is the point, not fine print: reshape can only give you a view when the values are already laid out in the order the new shape wants, and it silently copies when they are not.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([99, 2, 3, 4], [99, 2, 3, 4])
```


In [ ]:
import torch as t

def solve(x):
    """Return a reshaped VIEW and prove it shares storage."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2], [3, 4]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(494)


In [ ]:
#@title 💡 Solution — Problem 494
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return a reshaped VIEW and prove it shares storage."""
    v = x.reshape(-1)
    v[0] = 99
    return (v.tolist(), x.flatten().tolist())


example = t.tensor([[1, 2], [3, 4]])
print(solve(example))


<!-- dd:dd-q624 -->

### Problem 624 · independent

Write a function solve(x) that takes a tensor of ANY rank and returns a tuple (values, before, after): the values as one plain flat list in reading order, how many axes x had, and how many axes the flattened result has.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1, 2, 3, 4], 2, 1)
```


In [ ]:
import torch as t


def solve(x):
    """Return (flat values, axes before, axes after)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1, 2], [3, 4]]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(624)


In [ ]:
#@title 💡 Solution — Problem 624
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (flat values, axes before, axes after)."""
    flat = x.flatten()
    return (flat.tolist(), x.ndim, flat.ndim)


example = (t.tensor([[1, 2], [3, 4]]),)
print(solve(*example))


<!-- dd:dd-q625 -->

### Problem 625 · independent

Write a function solve(x, r, c) that takes a 2-D tensor x of shape (r, c) and returns a tuple of two plain nested lists: x re-described as (c, r), and x transposed. Both have the same shape; only one of them moved any numbers.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[1, 2], [3, 4], [5, 6]], [[1, 4], [2, 5], [3, 6]])
```


In [ ]:
import torch as t


def solve(x, r, c):
    """Return (x reshaped to (c, r), x transposed), both as nested lists."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1, 2, 3], [4, 5, 6]]), 2, 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(625)


In [ ]:
#@title 💡 Solution — Problem 625
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, r, c):
    """Return (x reshaped to (c, r), x transposed), both as nested lists."""
    return (x.reshape(c, r).tolist(), x.T.tolist())


example = (t.tensor([[1, 2, 3], [4, 5, 6]]), 2, 3,)
print(solve(*example))


<!-- dd:dd-q626 -->

### Problem 626 · independent

Write a function solve(x, a, b) that takes a 1-D tensor and two axis lengths, re-describes x as a 3-D tensor whose first two axes have lengths a and b, and returns a tuple (values, shape): the nested list and the shape as a plain tuple. Do not compute the third length — let -1 do it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[[0, 1], [2, 3], [4, 5]], [[6, 7], [8, 9], [10, 11]]], (2, 3, 2))
```


In [ ]:
import torch as t


def solve(x, a, b):
    """Return (x as an (a, b, ?) tensor, its shape) — the last axis inferred."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.arange(12), 2, 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(626)


In [ ]:
#@title 💡 Solution — Problem 626
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, a, b):
    """Return (x as an (a, b, ?) tensor, its shape) — the last axis inferred."""
    g = x.reshape(a, b, -1)
    return (g.tolist(), tuple(g.shape))


example = (t.arange(12), 2, 3,)
print(solve(*example))


#### Common mistakes

- **"Reshape can rearrange values."** — Reshape never *reorders* values: the
  flat row-major sequence of elements is identical before and after, and only
  the shape metadata changes. (It may still copy that sequence into fresh
  memory when the input is a non-contiguous view — same order, new buffer.)
  If the values need to move (transpose, sort, flip), reshape is the wrong
  tool.
- **"reshape(3, 4) on 11 elements will pad or truncate."** — It raises a
  `RuntimeError`. Counts must match exactly; `-1` only *derives* a dimension,
  it can't invent elements.
- **"Flattening reads down the columns."** — The order is row-major: across
  row 0 first. Column-major means transposing first.
- **"`order='F'` works, like in NumPy."** — PyTorch's `flatten` takes no order
  argument at all. `z.T.flatten()` is the column-major read.


<!-- dd:dd-kp-numpy-elementwise-ufuncs -->

## Elementwise math

`numpy.elementwise-ufuncs`


<!-- dd:dd-seg-numpy-elementwise-ufuncs-0 -->

### write the formula once — operators are elementwise


The core promise of tensor programming: **write the formula once, and it
applies to every element** — no loop. Operators `+ - * / ** %` between a
tensor and a scalar, or between two same-shaped tensors, work element by
element: `z * 2` doubles everything; `a * b` multiplies corresponding entries
(NOT matrix multiplication — that's `@`).

The general procedure for any "transform each entry" task:

> Express the rule for ONE element as a formula, then write that formula
> with the whole tensor in place of the element.


In [ ]:
import torch as t

z = t.tensor([1.0, 2.0, 3.0, 4.0])
print(z * 2)
print(z ** 2 - 1)
print(z % 2)




"Replace each x by x² − 1" → `z**2 - 1`. If you find yourself writing
`for i in range(len(z))`, stop — the elementwise spelling is shorter and
orders of magnitude faster (the loop happens in compiled code, and on a GPU it
happens in parallel). All of these return **new tensors** and leave the input
untouched.


In [ ]:
before = z.tolist()
squared = z ** 2 - 1
print("z after the expression:", z)
assert z.tolist() == before          # nothing was written back




And the one operator that is NOT elementwise, so the contrast lands early:


In [ ]:
a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print("a * a (elementwise):")
print(a * a)
print("a @ a (matrix product):")
print(a @ a)
assert not t.equal(a * a, a @ a)


In [ ]:
import torch as t

z = t.tensor([1.0, 2.0, 3.0])

# The per-element rule "x**2 - 1", written once for the whole tensor:
out = z**2 - 1
assert out.tolist() == [0.0, 3.0, 8.0]

# The input is untouched — the expression built a new tensor.
assert z.tolist() == [1.0, 2.0, 3.0]
print("z  ", z)
print("out", out)




Why: the expression reads exactly like the per-element rule — that
transliteration IS the method.


<!-- dd:dd-q192 -->

### Problem 192 · faded — your turn

Elementwise cube.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([  1.,   8., -27.])
```


In [ ]:
import torch as t

def solve(x):
    """Each entry raised to the third power."""
    return x _____ 3


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([1.0, 2.0, -3.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(192)


In [ ]:
#@title 💡 Solution — Problem 192
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.pow(x, 3)


print(solve(t.tensor([1.0, 2.0, -3.0])))


In [ ]:
import torch as t

z = t.tensor([1.0, 2.0, 3.0])

# The per-element rule "x**2 - 1", written once for the whole tensor:
out = z**2 - 1
assert out.tolist() == [0.0, 3.0, 8.0]

# The input is untouched — the expression built a new tensor.
assert z.tolist() == [1.0, 2.0, 3.0]
print("z  ", z)
print("out", out)




Why: the expression reads exactly like the per-element rule — that
transliteration IS the method.


<!-- dd:dd-q630 -->

### Problem 630 · faded — your turn

A line through every entry — one formula, no loop.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2.5000, 4.5000, 6.5000])
```


In [ ]:
import torch as t

def solve(x, m, b):
    """Return m * x + b for every entry."""
    return m _____ x _____ b


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.0, 2.0, 3.0]), 2.0, 0.5,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(630)


In [ ]:
#@title 💡 Solution — Problem 630
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, m, b):
    """Return m * x + b for every entry."""
    return m * x + b


example = (t.tensor([1.0, 2.0, 3.0]), 2.0, 0.5,)
print(solve(*example))


<!-- dd:dd-seg-numpy-elementwise-ufuncs-1 -->

### named math functions and the rounding family


Beyond operators, named functions map over the whole tensor: `t.sqrt`,
`t.abs`, `t.exp`, `t.log`, `t.sin`, …

Rounding is a *family*, and the members differ on negatives:

- `t.round` — nearest;
- `t.floor` — largest integer ≤ x, so −0.3 → −1.0 (away from zero);
- `t.ceil` — smallest integer ≥ x;
- `t.trunc` — toward zero, so −0.3 → −0.0 (same as `.to(t.int64)`).

Read the task's example values to see which member is being asked for — and
the fastest way to tell them apart is to run all four on the same negatives:


In [ ]:
import torch as t

v = t.tensor([1.7, -0.3, 2.5, -2.5])
print("input", v)
print("round", t.round(v))
print("floor", t.floor(v))
print("ceil ", t.ceil(v))
print("trunc", t.trunc(v))




The only column where floor and trunc disagree is the negative one, which is
exactly where a grader will look:


In [ ]:
assert t.floor(v).tolist() == [1.0, -1.0, 2.0, -3.0]
assert t.trunc(v).tolist() == [1.0, -0.0, 2.0, -2.0]
assert t.equal(t.trunc(v).to(t.int64), v.to(t.int64))
print("at -0.3: floor ->", t.floor(v)[1].item(),
      "but trunc ->", t.trunc(v)[1].item())
print("trunc == .to(t.int64):", bool(t.equal(t.trunc(v).to(t.int64),
                                             v.to(t.int64))))


In [ ]:
import torch as t

v = t.tensor([1.7, -0.3, 2.5])

assert t.floor(v).tolist() == [1.0, -1.0, 2.0]   # floor moves DOWN
assert t.trunc(v).tolist() == [1.0, -0.0, 2.0]   # trunc moves toward zero
assert t.sqrt(t.tensor([4.0, 9.0])).tolist() == [2.0, 3.0]
print("input", v)
print("floor", t.floor(v))
print("trunc", t.trunc(v), "  <- differs only at the negative")




Why: floor vs trunc only disagree on negatives — that's exactly where tasks
(and graders) check.


<!-- dd:dd-q49 -->

### Problem 49 · faded — your turn

Floor every entry (note what floor does to negatives).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 1.,  3., -1.])
```


In [ ]:
import torch as t

def solve(z):
    """Replace each entry by the largest integer value <= it."""
    return t._____(z)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.2, 3.7, -0.3])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(49)


In [ ]:
#@title 💡 Solution — Problem 49
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.floor(z)


example = t.tensor([1.2, 3.7, -0.3])
print(solve(example))


In [ ]:
import torch as t

v = t.tensor([1.7, -0.3, 2.5])

assert t.floor(v).tolist() == [1.0, -1.0, 2.0]   # floor moves DOWN
assert t.trunc(v).tolist() == [1.0, -0.0, 2.0]   # trunc moves toward zero
assert t.sqrt(t.tensor([4.0, 9.0])).tolist() == [2.0, 3.0]
print("input", v)
print("floor", t.floor(v))
print("trunc", t.trunc(v), "  <- differs only at the negative")




Why: floor vs trunc only disagree on negatives — that's exactly where tasks
(and graders) check.


<!-- dd:dd-q631 -->

### Problem 631 · faded — your turn

The fractional part, sign kept — pick the rounding family member that cuts toward zero.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 0.5000, -0.2500,  0.0000])
```


In [ ]:
import torch as t

def solve(x):
    """Return the fractional part of every entry, keeping its sign."""
    return x - t._____(x)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.5, -2.25, 3.0]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(631)


In [ ]:
#@title 💡 Solution — Problem 631
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return the fractional part of every entry, keeping its sign."""
    return x - t.trunc(x)


example = (t.tensor([1.5, -2.25, 3.0]),)
print(solve(*example))


<!-- dd:dd-seg-numpy-elementwise-ufuncs-2 -->

### elementwise choosers — maximum, minimum, clamp


`t.maximum(a, b)` / `t.minimum(a, b)` pick the larger/smaller *at each
position* (contrast with `a.max()`, which reduces the whole tensor to one
number — different KP). `z.clamp(min=lo, max=hi)` limits values to a range;
NumPy calls this `clip`, and PyTorch accepts that spelling as an alias, but
`clamp` is the name you will read in model code.


In [ ]:
import torch as t

a = t.tensor([1.0, 5.0, 2.0])
b = t.tensor([3.0, 4.0, 2.5])
print("maximum:", t.maximum(a, b))     # one answer per position
print("a.max():", a.max())             # one answer, full stop




These overlap: `clamp(max=100)` and `t.minimum(x, ...)` compute the same
thing, and `clamp(min=0)` is ReLU. Prefer `clamp` for a constant bound — it
takes a plain Python number, whereas `t.minimum` wants a second tensor, and a
tensor you construct on the spot lands on the CPU and will not match an input
living on a GPU.


In [ ]:
readings = t.tensor([-2.0, 0.5, 7.0, 12.0])
print("clamp(min=0):     ", readings.clamp(min=0.0))       # ReLU
print("clamp(max=10):    ", readings.clamp(max=10.0))
print("clamp(0, 10):     ", readings.clamp(min=0.0, max=10.0))
assert t.equal(readings.clamp(max=10.0), t.minimum(readings, t.tensor(10.0)))


In [ ]:
import torch as t

# Elementwise chooser between TWO tensors: keep the larger at each slot.
a = t.tensor([1.0, 5.0, 2.0])
b = t.tensor([3.0, 4.0, 2.5])
assert t.maximum(a, b).tolist() == [3.0, 5.0, 2.5]

# Pipeline: curve exam scores — add 5, cap at 100, floor to whole points.
scores = t.tensor([71.5, 88.25, 97.0, 99.5])
curved = t.floor((scores + 5).clamp(max=100.0))
assert curved.tolist() == [76.0, 93.0, 100.0, 100.0]
print("raw   ", scores)
print("curved", curved)

# The other bound. `min=` sets a FLOOR, `max=` sets a CEILING — and the two
# read backwards from how they sound: min= raises everything below it.
readings = t.tensor([-2.0, 0.5, 7.0])
assert readings.clamp(min=0.0).tolist() == [0.0, 0.5, 7.0]
print("readings   ", readings)
print("clamp(min=0)", readings.clamp(min=0.0), " <- this is ReLU")




Why: composing these left-to-right is normal style — each stage maps over
the whole tensor, and the pipeline reads exactly like the per-element rule:
add, cap, floor.

The last line is the one worth memorising: `clamp(min=0.0)` is ReLU, the
single most common nonlinearity in the whole field.


<!-- dd:dd-q67 -->

### Problem 67 · faded — your turn

Negatives become 0.0, non-negatives pass through (ReLU).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.0000, 0.5000, 3.0000])
```


In [ ]:
import torch as t

def solve(z):
    """Each negative entry replaced by 0.0 (new tensor; z unmodified)."""
    return z._____(_____=0.0)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([-2.0, 0.5, 3.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(67)


In [ ]:
#@title 💡 Solution — Problem 67
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.clamp(min=0.0)


example = t.tensor([-2.0, 0.5, 3.0])
print(solve(example))


In [ ]:
import torch as t

# Elementwise chooser between TWO tensors: keep the larger at each slot.
a = t.tensor([1.0, 5.0, 2.0])
b = t.tensor([3.0, 4.0, 2.5])
assert t.maximum(a, b).tolist() == [3.0, 5.0, 2.5]

# Pipeline: curve exam scores — add 5, cap at 100, floor to whole points.
scores = t.tensor([71.5, 88.25, 97.0, 99.5])
curved = t.floor((scores + 5).clamp(max=100.0))
assert curved.tolist() == [76.0, 93.0, 100.0, 100.0]
print("raw   ", scores)
print("curved", curved)

# The other bound. `min=` sets a FLOOR, `max=` sets a CEILING — and the two
# read backwards from how they sound: min= raises everything below it.
readings = t.tensor([-2.0, 0.5, 7.0])
assert readings.clamp(min=0.0).tolist() == [0.0, 0.5, 7.0]
print("readings   ", readings)
print("clamp(min=0)", readings.clamp(min=0.0), " <- this is ReLU")




Why: composing these left-to-right is normal style — each stage maps over
the whole tensor, and the pipeline reads exactly like the per-element rule:
add, cap, floor.

The last line is the one worth memorising: `clamp(min=0.0)` is ReLU, the
single most common nonlinearity in the whole field.


<!-- dd:dd-q632 -->

### Problem 632 · faded — your turn

Position by position, keep the smaller of the two.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 2., 3.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return the smaller of each pair of corresponding entries."""
    return t._____(a, b)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.0, 5.0, 3.0]), t.tensor([4.0, 2.0, 3.0]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(632)


In [ ]:
#@title 💡 Solution — Problem 632
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return the smaller of each pair of corresponding entries."""
    return t.minimum(a, b)


example = (t.tensor([1.0, 5.0, 3.0]), t.tensor([4.0, 2.0, 3.0]),)
print(solve(*example))


<!-- dd:dd-q487 -->

### Problem 487 · guided

Write a function solve(x) that takes a PyTorch tensor and returns a new tensor with every element doubled. Write the formula once for the whole tensor — there is no loop to write.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 2., -4.,  7.])
```


<details>
<summary>Hints</summary>

1. The whole tensor at once — there is no index to loop over.
2. An arithmetic operator applied to a tensor is applied to every element.
3. `x * 2`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return x with every element doubled."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.0, -2.0, 3.5])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(487)


In [ ]:
#@title 💡 Solution — Problem 487
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return x with every element doubled."""
    return x * 2


example = t.tensor([1.0, -2.0, 3.5])
print(solve(example))


<!-- dd:dd-q488 -->

### Problem 488 · guided

Write a function solve(x) that takes a PyTorch tensor of floats and returns a new tensor holding the absolute value of each element. The original must be left unchanged.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1.5000, 2.0000, 3.2500])
```


<details>
<summary>Hints</summary>

1. You need the magnitude of each element, sign discarded.
2. It is a method on the tensor, and it returns a NEW tensor rather than
   editing yours — which is what keeps the input unchanged.
3. `x.abs()`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the absolute value of every element."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([-1.5, 2.0, -3.25])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(488)


In [ ]:
#@title 💡 Solution — Problem 488
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return the absolute value of every element."""
    return x.abs()


example = t.tensor([-1.5, 2.0, -3.25])
print(solve(example))


<!-- dd:dd-q43 -->

### Problem 43 · independent

Write a function solve(a, b) that takes two 1-D PyTorch tensors of equal length and returns a new tensor of the same length, where each element is the larger of the two corresponding elements from a and b. If the two values at a position are equal, keep that shared value. Do not use Python loops or Python's built-in max.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([4., 5., 3., 7.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return the elementwise maximum of a and b."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
a = t.tensor([1.0, 5.0, 3.0, 2.0])
b = t.tensor([4.0, 2.0, 3.0, 7.0])
print(solve(a, b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(43)


In [ ]:
#@title 💡 Solution — Problem 43
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.maximum(a, b)


a = t.tensor([1.0, 5.0, 3.0, 2.0])
b = t.tensor([4.0, 2.0, 3.0, 7.0])
print(solve(a, b))


<!-- dd:dd-q489 -->

### Problem 489 · independent

Write a function solve(x, lo, hi) that takes a float tensor and two floats, and returns a new tensor in which every element below lo has been raised to lo and every element above hi lowered to hi. Values already inside the range are left alone. Remember that min= sets the FLOOR and max= the CEILING — they read backwards from how they sound.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.0000, 0.5000, 5.0000])
```


In [ ]:
import torch as t

def solve(x, lo, hi):
    """Return x with every element pulled into [lo, hi]."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([-3.0, 0.5, 9.0]), 0.0, 5.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(489)


In [ ]:
#@title 💡 Solution — Problem 489
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, lo, hi):
    """Return x with every element pulled into [lo, hi]."""
    return x.clamp(min=lo, max=hi)


example = (t.tensor([-3.0, 0.5, 9.0]), 0.0, 5.0)
print(solve(*example))


<!-- dd:dd-q63 -->

### Problem 63 · independent

Write a function solve(z) that takes a 1-D PyTorch tensor of POSITIVE floats and returns a float tensor of the same shape holding just the integer part of each entry (e.g. 3.7 becomes 3.0). Several torch idioms compute this — use whichever you like.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([3., 0., 5.])
```


In [ ]:
import torch as t

def solve(z):
    """Return the integer part of each positive entry of z, as floats."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([3.7, 0.2, 5.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(63)


In [ ]:
#@title 💡 Solution — Problem 63
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.floor(z)


example = t.tensor([3.7, 0.2, 5.0])
print(solve(example))


<!-- dd:dd-q633 -->

### Problem 633 · independent

Write a function solve(a, b) that takes two float tensors of the same shape (the two legs of a right triangle, entry by entry) and returns the hypotenuse for each: the square root of a squared plus b squared.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 5., 13.])
```


In [ ]:
import torch as t


def solve(a, b):
    """Return the hypotenuse sqrt(a² + b²) for each pair of entries."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([3.0, 5.0]), t.tensor([4.0, 12.0]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(633)


In [ ]:
#@title 💡 Solution — Problem 633
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return the hypotenuse sqrt(a² + b²) for each pair of entries."""
    return t.sqrt(a ** 2 + b ** 2)


example = (t.tensor([3.0, 5.0]), t.tensor([4.0, 12.0]),)
print(solve(*example))


<!-- dd:dd-q634 -->

### Problem 634 · independent

Write a function solve(x) that takes a float tensor and returns a tuple of two plain lists: the integer just below each entry (or the entry itself when it is already whole) and the integer just above it. Both stay floats.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1.0, -2.0, 2.0], [2.0, -1.0, 2.0])
```


In [ ]:
import torch as t


def solve(x):
    """Return (floor of each entry, ceil of each entry) as lists."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.5, -1.5, 2.0]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(634)


In [ ]:
#@title 💡 Solution — Problem 634
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (floor of each entry, ceil of each entry) as lists."""
    return (t.floor(x).tolist(), t.ceil(x).tolist())


example = (t.tensor([1.5, -1.5, 2.0]),)
print(solve(*example))


<!-- dd:dd-q635 -->

### Problem 635 · independent

Write a function solve(x, floor_value) that returns a new tensor in which every entry below floor_value has been raised to floor_value, using maximum against a tensor of that constant shaped like x (not clamp). Entries already at or above it are unchanged.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.0000, 0.5000, 3.0000])
```


In [ ]:
import torch as t


def solve(x, floor_value):
    """Raise every entry to at least floor_value, using maximum against a constant tensor."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([-2.0, 0.5, 3.0]), 0.0,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(635)


In [ ]:
#@title 💡 Solution — Problem 635
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, floor_value):
    """Raise every entry to at least floor_value, using maximum against a constant tensor."""
    return t.maximum(x, t.full_like(x, floor_value))


example = (t.tensor([-2.0, 0.5, 3.0]), 0.0,)
print(solve(*example))


#### Common mistakes

- **"`a * b` on two matrices is matrix multiplication."** — It is elementwise.
  Matrix product is `a @ b`. This distinction matters enough that it gets its
  own KP later.
- **"`t.maximum` and `t.max` are the same."** — `t.maximum(a, b)` compares
  two tensors position-by-position (returns a tensor); `t.max(a)` reduces one
  tensor to its single largest value.
- **"floor and truncate are the same."** — Only for positives. For negatives,
  floor moves AWAY from zero (−0.3 → −1.0) while trunc moves toward it
  (−0.3 → −0.0). Read the task's example values to see which is being asked.
- **"An in-place version is just a style choice."** — Methods ending in an
  underscore (`z.clamp_(min=0)`) modify the tensor you were handed. That is a
  different contract from `z.clamp(min=0)`, and it is how you accidentally
  mutate a caller's data.


<!-- dd:dd-kp-numpy-aggregations -->

## Whole-tensor aggregations and Python scalars

`numpy.aggregations`


<!-- dd:dd-seg-numpy-aggregations-0 -->

### reductions — collapsing a tensor to one number


Where an elementwise operation maps a tensor to a same-shaped tensor, an
**aggregation (reduction)** collapses a tensor down to a single number:
`x.sum()`, `x.mean()`, `x.min()`, `x.max()`, `x.std()` — the workhorses.

Called with no arguments, each of these reduces over **all** elements
regardless of shape — a 2-D tensor's `x.max()` is the max of the whole matrix.
(Reducing along just one axis is the `dim=` keyword, which gets its own KP in
the broadcasting lesson — walk before running.)


In [ ]:
import torch as t

grid = t.tensor([[3.0, 8.0, 1.0],
                 [6.0, 2.0, 9.0]])
print("sum ", grid.sum())
print("mean", grid.mean())
print("min ", grid.min())
print("max ", grid.max())




Six values in, one value out, every time — and notice the shape never came
into it:


In [ ]:
flat = grid.reshape(6)
assert t.equal(flat.max(), grid.max())
assert t.equal(flat.sum(), grid.sum())
print(grid.shape, "and", flat.shape, "→ same answers")


Task: global min and max of a matrix of sensor readings.


In [ ]:
import torch as t

readings = t.tensor([[3.5, -2.0, 7.25],
                     [0.0,  9.5, -8.75]])

# min/max with no dim argument scan the WHOLE tensor, ignoring shape.
lo, hi = readings.min(), readings.max()
assert (lo.item(), hi.item()) == (-8.75, 9.5)
print("min", lo.item(), "| max", hi.item())




Why: no dim argument = one value for the whole tensor, shape ignored. That's
the default to internalize before `dim=` complicates things.


<!-- dd:dd-q26 -->

### Problem 26 · faded — your turn

Global min and max of a 2-D tensor, returned as a (min, max) pair of plain
Python numbers.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(-8.75, 9.5)
```


In [ ]:
import torch as t

def solve(x):
    """Return (smallest, largest) element of the whole 2-D tensor."""
    return (x._____().item(), x._____().item())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[3.5, -2.0, 7.25], [0.0, 9.5, -8.75]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(26)


In [ ]:
#@title 💡 Solution — Problem 26
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return (x.min().item(), x.max().item())


example = t.tensor([[3.5, -2.0, 7.25], [0.0, 9.5, -8.75]])
print(solve(example))


<!-- dd:dd-seg-numpy-aggregations-1 -->

### 0-dimensional tensors vs plain Python numbers


One practical wrinkle, and it bites harder here than in NumPy: reductions
return a **0-dimensional tensor**, not a number. It prints as
`tensor(1.5833)` and it still carries a dtype, a device, and possibly a
gradient. Graders, JSON encoders, and f-strings care.

When a task says "return a plain Python int/float/bool", convert explicitly:

> `float(x.mean())`, `int(x.sum())`, `bool((x > 0).any())`
> — or `x.item()`, the generic "unwrap this 0-d result".


In [ ]:
import torch as t

grid = t.tensor([[3.0, 8.0, 1.0],
                 [6.0, 2.0, 9.0]])
raw = grid.mean()
print(raw, "| ndim", raw.ndim, "| type", type(raw).__name__)
print(float(raw), "| type", type(float(raw)).__name__)




The first line still says `tensor(...)`. That is the whole distinction:


In [ ]:
assert raw.ndim == 0 and isinstance(raw, t.Tensor)
assert isinstance(raw.item(), float)
assert int(grid.sum()) == 29
assert bool((grid > 0).all()) is True
print("raw.ndim", raw.ndim, "-> still a tensor:", isinstance(raw, t.Tensor))
print("int(grid.sum())", int(grid.sum()), type(int(grid.sum())).__name__)
print("bool((grid > 0).all())", bool((grid > 0).all()),
      type(bool((grid > 0).all())).__name__)




Keep tensors *inside* your computation; convert exactly at the boundary
where a plain Python value is required. Unwrapping early is how you
accidentally break the autograd chain in real model code.


In [ ]:
import torch as t

readings = t.tensor([[3.5, -2.0, 7.25],
                     [0.0,  9.5, -8.75]])

# mean returns a 0-d TENSOR. Usually fine, but when the contract says
# "a single float scalar", unwrap it explicitly.
raw = readings.mean()
assert raw.ndim == 0 and isinstance(raw, t.Tensor)

avg = float(raw)
assert isinstance(avg, float)
assert abs(avg - 1.5833333) < 1e-5
print("as a tensor:", raw, "| as a float:", avg)




Why: `float(...)` at the boundary — the computation stays in torch, only the
returned value is unwrapped.


<!-- dd:dd-q28 -->

### Problem 28 · faded — your turn

The arithmetic mean of a vector, as a plain Python float.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
2.5
```


In [ ]:
import torch as t

def solve(x):
    """Mean of x as a plain Python float."""
    return _____(x._____())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.0, 2.0, 3.0, 4.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(28)


In [ ]:
#@title 💡 Solution — Problem 28
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return float(x.mean())


example = t.tensor([1.0, 2.0, 3.0, 4.0])
print(solve(example))


<!-- dd:dd-seg-numpy-aggregations-2 -->

### whole-tensor yes/no verdicts


Yes/no questions about tensors have two standard shapes:

- **Comparison, then reduce.** A comparison builds a boolean tensor (previous
  KP); `x.any()` (is at least one entry True?) or `x.all()` (are they all
  True?) collapses it to one answer. `(x > 0).all()` asks "is everything
  positive?".
- **Whole-tensor equality.** `t.equal(a, b)` is an exact match of shape
  and values; `t.allclose(a, b)` is equality within floating-point tolerance
  — the right check after float arithmetic.


In [ ]:
import torch as t

grid = t.tensor([[3.0, -8.0, 1.0],
                 [6.0, 2.0, -9.0]])
print("negatives present?", bool((grid < 0).any()))
print("all positive?    ", bool((grid > 0).all()))




`a == b` alone is NOT a verdict — it's elementwise and yields a boolean
tensor (and `if` on it raises an error).


In [ ]:
a = t.tensor([1.0, 2.0])
b = t.tensor([1.0, 5.0])
print(a == b)                 # a tensor, not an answer
print(t.equal(a, b))          # one bool




And the reason `allclose` exists at all — float arithmetic does not land
where the arithmetic says it should:


In [ ]:
summed = t.full((10,), 0.1).sum()
print(summed.item(), "vs", 1.0)
assert not t.equal(summed, t.tensor(1.0))
assert t.allclose(summed, t.tensor(1.0))


In [ ]:
import torch as t

readings = t.tensor([[3.5, -2.0, 7.25],
                     [0.0,  9.5, -8.75]])

# Boolean pipeline: comparison (elementwise) then reduction (any).
# Read it aloud: "readings less than zero — any?"
has_negative = bool((readings < 0).any())
assert has_negative is True

# Float-safe equality: after arithmetic, prefer allclose. Ten 0.1s summed
# in float32 land just past 1.0.
a = t.full((10,), 0.1).sum()
b = t.tensor(1.0)
assert not t.equal(a, b)         # bitwise-exact? no — accumulated float error
assert t.allclose(a, b)          # equal within tolerance? yes
print("any negative?", has_negative)
print("ten 0.1s summed:", a.item(), "| equal:", bool(t.equal(a, b)),
      "| allclose:", bool(t.allclose(a, b)))




Why: exact equality is for ints/bools and provenance checks; `allclose` is
for anything that went through float arithmetic — and float32 has fewer
digits to spare than the float64 you may be used to.


<!-- dd:dd-q64 -->

### Problem 64 · faded — your turn

Tolerant closeness AND exact equality of two tensors, as two plain bools.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, True)
```


In [ ]:
import torch as t

def solve(a, b):
    """(close within float tolerance?, exactly equal?) as plain bools."""
    return (bool(t._____(a, b)), bool(t._____(a, b)))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([1.0, 2.0]), t.tensor([1.0, 2.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(64)


In [ ]:
#@title 💡 Solution — Problem 64
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return bool(t.allclose(a, b)), bool(t.equal(a, b))


print(solve(t.tensor([1.0, 2.0]), t.tensor([1.0, 2.0])))


<!-- dd:dd-q495 -->

### Problem 495 · guided

Write a function solve(x) that takes a PyTorch tensor and returns the sum of ALL its elements as a plain Python number, not a tensor. A whole-tensor reduction gives you a 0-dimensional tensor; .item() is how you get the number out of it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
10.0
```


<details>
<summary>Hints</summary>

1. Two steps: collapse the tensor to one number, then leave PyTorch behind.
2. A whole-tensor reduction returns a 0-dimensional TENSOR, not a Python
   number — the test checks which one you handed back.
3. `x.sum().item()`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the sum of every element as a Python number."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(495)


In [ ]:
#@title 💡 Solution — Problem 495
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return the sum of every element as a Python number."""
    return x.sum().item()


example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


<!-- dd:dd-q496 -->

### Problem 496 · guided

Write a function solve(x) that returns a tuple (mean, count): the mean of every element as a plain Python float, and the total number of elements as a plain int. Both are whole-tensor questions — no axis is involved.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2.5, 4)
```


<details>
<summary>Hints</summary>

1. Both halves are whole-tensor questions, so neither needs an axis.
2. The count is metadata you already know how to read — it is not a
   reduction, and it does not need .item().
3. `(x.mean().item(), x.numel())`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return (mean, count) over the whole tensor."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(496)


In [ ]:
#@title 💡 Solution — Problem 496
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (mean, count) over the whole tensor."""
    return (x.mean().item(), x.numel())


example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


<!-- dd:dd-q62 -->

### Problem 62 · independent

Write a function solve(z) that takes a 1-D PyTorch integer tensor and returns the sum of all its entries as a plain Python int — not a 0-dimensional tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
45
```


In [ ]:
import torch as t

def solve(z):
    """Return the sum of z's entries as a Python int."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(10)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(62)


In [ ]:
#@title 💡 Solution — Problem 62
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return int(z.sum())


example = t.arange(10)
print(solve(example))


<!-- dd:dd-q497 -->

### Problem 497 · independent

Write a function solve(x, threshold) that returns a tuple (any_above, how_many): a plain Python bool saying whether ANY element exceeds the threshold, and a plain int counting how many do. Both come from the same boolean tensor — build it once. True counts as 1 when you sum it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, 2)
```


In [ ]:
import torch as t

def solve(x, threshold):
    """Return (any above threshold?, how many)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.0, 5.0, 9.0]), 4.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(497)


In [ ]:
#@title 💡 Solution — Problem 497
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, threshold):
    """Return (any above threshold?, how many)."""
    mask = x > threshold
    return (bool(mask.any()), int(mask.sum()))


example = (t.tensor([1.0, 5.0, 9.0]), 4.0)
print(solve(*example))


<!-- dd:dd-q498 -->

### Problem 498 · independent

Write a function solve(x) that returns a tuple (min, max, span) of plain Python floats: the smallest element, the largest, and the distance between them. Reduce twice and do the subtraction in Python — the span is not itself a reduction.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(-2.0, 7.25, 9.25)
```


In [ ]:
import torch as t

def solve(x):
    """Return (min, max, span) as plain Python floats."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[3.5, -2.0], [7.25, 0.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(498)


In [ ]:
#@title 💡 Solution — Problem 498
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (min, max, span) as plain Python floats."""
    lo = x.min().item()
    hi = x.max().item()
    return (lo, hi, hi - lo)


example = t.tensor([[3.5, -2.0], [7.25, 0.0]])
print(solve(example))


#### Common mistakes

- **"`x.max()` on a matrix gives per-row maxima."** — With no arguments it
  reduces over everything: one value for the whole tensor. Per-row/column
  reductions need `dim=`, covered in the broadcasting lesson.
- **"Reductions return normal Python numbers."** — They return 0-dimensional
  tensors. Mostly interchangeable in arithmetic, but "return a plain
  int/float" contracts require `int(...)`/`float(...)`/`.item()`.
- **"`==` tells me whether two tensors are equal."** — `a == b` is ELEMENTWISE,
  yielding a boolean tensor (and `if` on it raises an error). Whole-tensor
  verdicts are `t.equal` (exact) or `t.allclose` (float-tolerant).
- **"`t.array_equal` is the exact check."** — That's the NumPy name; there is
  no such function here. It's `t.equal`.


<!-- dd:dd-kp-numpy-sorting -->

## Sorting tensors

`numpy.sorting`


<!-- dd:dd-seg-numpy-sorting-0 -->

### sort returns a pair, not a tensor


**`t.sort(z)` does not return a sorted tensor.** It returns a *pair* — the
sorted values and the indices that produced them — and forgetting that is the
mistake this KP exists to prevent:

```python no-run
t.sort(z)          # -> torch.return_types.sort(values=..., indices=...)
t.sort(z).values   # the sorted tensor you actually wanted
```


In [ ]:
import torch as t

z = t.tensor([0.5, 0.25, 0.75])
print(t.sort(z))




That printout is the pair, and it is what a function returns if you forget
`.values`.

You can unpack it either way: `values, indices = t.sort(z)`, or reach for
`.values` / `.indices` by name. The indices half is not a consolation prize —
it is what "sort one thing by another" tasks need, and it is the same thing
`t.argsort(z)` gives you on its own.


In [ ]:
values, indices = t.sort(z)
print("values ", values)
print("indices", indices)
assert t.equal(indices, t.argsort(z))
assert t.equal(z[indices], values)      # the indices reconstruct the values




Sorting never modifies the input: `t.sort(z)` and the method form `z.sort()`
both leave `z` alone and hand back a new pair. (There is no in-place `sort_`.)

**`descending=True`** sorts largest-first — a real keyword, unlike NumPy, where
you have to sort ascending and reverse afterwards.


In [ ]:
print("z is still", z)
print("descending", t.sort(z, descending=True).values)
assert z.tolist() == [0.5, 0.25, 0.75]


Task: produce a sorted copy of a vector, confirm the original is intact, then
get the same values descending.


In [ ]:
import torch as t

z = t.tensor([0.5, 0.25, 0.75])

# sort returns a PAIR — take .values for the sorted tensor.
result = t.sort(z)
assert result.values.tolist() == [0.25, 0.5, 0.75]
assert result.indices.tolist() == [1, 0, 2]     # where each value came from

# The input keeps its original order (the grader often checks this).
assert z.tolist() == [0.5, 0.25, 0.75]

# Descending is a keyword here — no reverse step needed.
desc = t.sort(z, descending=True).values
assert desc.tolist() == [0.75, 0.5, 0.25]

# Unpacking works too, and reads well when you want both halves.
values, indices = t.sort(z)
assert values.tolist() == [0.25, 0.5, 0.75]
print("input     ", z)
print("values    ", values)
print("indices   ", indices)
print("descending", desc)




Why each step:

1. Taking `.values` is the habit to build. Returning `t.sort(z)` straight from
   a function hands the caller a pair, and the failure looks like a type error
   far from the line that caused it.
2. The indices are the bridge to the order-statistics KP: they say *where*
   each sorted value came from, which is how you carry a second tensor along.
3. `descending=True` is one of the places PyTorch is friendlier than NumPy —
   worth knowing so you don't write the reverse-slice workaround (which
   wouldn't work here anyway, since negative slice steps are rejected).


<!-- dd:dd-q58 -->

### Problem 58 · faded — your turn

Sorted copy, smallest to largest, input left unmodified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.1000, 0.4000, 0.9000])
```


In [ ]:
import torch as t

def solve(z):
    """Return a NEW tensor with z's values in ascending order."""
    return t._____(z)._____


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([0.4, 0.1, 0.9])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(58)


In [ ]:
#@title 💡 Solution — Problem 58
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.sort(z).values


example = t.tensor([0.4, 0.1, 0.9])
print(solve(example))


Task: produce a sorted copy of a vector, confirm the original is intact, then
get the same values descending.


In [ ]:
import torch as t

z = t.tensor([0.5, 0.25, 0.75])

# sort returns a PAIR — take .values for the sorted tensor.
result = t.sort(z)
assert result.values.tolist() == [0.25, 0.5, 0.75]
assert result.indices.tolist() == [1, 0, 2]     # where each value came from

# The input keeps its original order (the grader often checks this).
assert z.tolist() == [0.5, 0.25, 0.75]

# Descending is a keyword here — no reverse step needed.
desc = t.sort(z, descending=True).values
assert desc.tolist() == [0.75, 0.5, 0.25]

# Unpacking works too, and reads well when you want both halves.
values, indices = t.sort(z)
assert values.tolist() == [0.25, 0.5, 0.75]
print("input     ", z)
print("values    ", values)
print("indices   ", indices)
print("descending", desc)




Why each step:

1. Taking `.values` is the habit to build. Returning `t.sort(z)` straight from
   a function hands the caller a pair, and the failure looks like a type error
   far from the line that caused it.
2. The indices are the bridge to the order-statistics KP: they say *where*
   each sorted value came from, which is how you carry a second tensor along.
3. `descending=True` is one of the places PyTorch is friendlier than NumPy —
   worth knowing so you don't write the reverse-slice workaround (which
   wouldn't work here anyway, since negative slice steps are rejected).


<!-- dd:dd-q660 -->

### Problem 660 · faded — your turn

Both halves of the pair that sort returns.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0.25, 0.5, 0.75], [1, 0, 2])
```


In [ ]:
import torch as t

def solve(z):
    """Return (sorted values, where each came from), both as lists."""
    s = t._____(z)
    return (s._____.tolist(), s._____.tolist())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([0.5, 0.25, 0.75]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(660)


In [ ]:
#@title 💡 Solution — Problem 660
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    """Return (sorted values, where each came from), both as lists."""
    s = t.sort(z)
    return (s.values.tolist(), s.indices.tolist())


example = (t.tensor([0.5, 0.25, 0.75]),)
print(solve(*example))


<!-- dd:dd-seg-numpy-sorting-1 -->

### argsort — the positions, and reordering by them


`t.argsort(z)` gives you the indices half on its own: `order[0]` is the position
of the smallest element, `order[1]` the next, and so on. On a single tensor that
is just a slower route to `t.sort(z).values` — you would still have to index with
it.

Its real job is **carrying a second tensor along**. When two tensors are parallel
— names and scores, boxes and confidences, tokens and logits — sorting one of
them independently destroys the correspondence. So you rank once, get the order,
and index *every* tensor with that same order. They stay lined up because they
all moved the same way.

Indexing a tensor with an index tensor is the "fancy indexing" you have already
met: `names[order]` builds a new tensor by reading `names` at each position in
`order`, in that order.


In [ ]:
import torch as t

ids = t.tensor([10, 11, 12, 13])
scores = t.tensor([0.4, 0.9, 0.1, 0.7])

order = t.argsort(scores, descending=True)
print("order  ", order)
print("ids    ", ids[order])
print("scores ", scores[order])




Both tensors moved by the SAME order, so row-by-row they still describe the
same items. Sorting them separately is the bug this pattern prevents:


In [ ]:
broken = t.sort(ids, descending=True).values
print("independently sorted ids:", broken, "— no longer paired with anything")
assert t.equal(scores[order], t.sort(scores, descending=True).values)


In [ ]:
import torch as t

names = t.tensor([10, 20, 30])
scores = t.tensor([0.5, 0.25, 0.75])

# The positions that WOULD sort scores ascending — not the values.
order = t.argsort(scores)
assert order.tolist() == [1, 0, 2]

# Index both tensors with the SAME order and they stay in correspondence.
assert scores[order].tolist() == [0.25, 0.5, 0.75]
assert names[order].tolist() == [20, 10, 30]

# argsort takes the same direction keyword sort does.
best_first = t.argsort(scores, descending=True)
assert best_first.tolist() == [2, 0, 1]
assert names[best_first].tolist() == [30, 10, 20]
print("scores       ", scores, " names", names)
print("best-first   ", best_first)
print("names ranked ", names[best_first], " scores", scores[best_first])




Read the last two lines: name 30 comes first because score 0.75 is the highest,
and nothing about `names` was sorted. That is the whole pattern — rank one
tensor, index all of them.


<!-- dd:dd-q520 -->

### Problem 520 · faded — your turn

Rank by score, then move both tensors with the same order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([20, 10, 30], [0.8999999761581421, 0.4000000059604645, 0.10000000149011612])
```


In [ ]:
import torch as t

def solve(names, scores):
    """Return (names, scores) as lists, both ordered highest score first."""
    order = t._____(scores, _____=True)
    return (names[_____].tolist(), scores[_____].tolist())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([10, 20, 30]), t.tensor([0.4, 0.9, 0.1]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(520)


In [ ]:
#@title 💡 Solution — Problem 520
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(names, scores):
    """Reorder one tensor by another's ranking."""
    order = t.argsort(scores, descending=True)
    return (names[order].tolist(), scores[order].tolist())


example = (t.tensor([10, 20, 30]), t.tensor([0.4, 0.9, 0.1]))
print(solve(*example))


In [ ]:
import torch as t

names = t.tensor([10, 20, 30])
scores = t.tensor([0.5, 0.25, 0.75])

# The positions that WOULD sort scores ascending — not the values.
order = t.argsort(scores)
assert order.tolist() == [1, 0, 2]

# Index both tensors with the SAME order and they stay in correspondence.
assert scores[order].tolist() == [0.25, 0.5, 0.75]
assert names[order].tolist() == [20, 10, 30]

# argsort takes the same direction keyword sort does.
best_first = t.argsort(scores, descending=True)
assert best_first.tolist() == [2, 0, 1]
assert names[best_first].tolist() == [30, 10, 20]
print("scores       ", scores, " names", names)
print("best-first   ", best_first)
print("names ranked ", names[best_first], " scores", scores[best_first])




Read the last two lines: name 30 comes first because score 0.75 is the highest,
and nothing about `names` was sorted. That is the whole pattern — rank one
tensor, index all of them.


<!-- dd:dd-q661 -->

### Problem 661 · faded — your turn

Reorder one tensor by another's ranking.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[20, 10, 30]
```


In [ ]:
import torch as t

def solve(ids, scores):
    """Return ids reordered by ascending score, as a list."""
    order = t._____(scores)
    return ids[order].tolist()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([10, 20, 30]), t.tensor([0.5, 0.25, 0.75]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(661)


In [ ]:
#@title 💡 Solution — Problem 661
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(ids, scores):
    """Return ids reordered by ascending score, as a list."""
    order = t.argsort(scores)
    return ids[order].tolist()


example = (t.tensor([10, 20, 30]), t.tensor([0.5, 0.25, 0.75]),)
print(solve(*example))


<!-- dd:dd-seg-numpy-sorting-2 -->

### picking an axis, and taking only the top k


On a tensor with more than one axis, **`dim=`** says which axis to sort along,
and every lane along that axis is sorted independently. `t.sort(m, dim=1)` sorts
*within* each row — which means it scrambles each row's contents and destroys any
correspondence between columns. That is usually not what you want from "sort the
matrix"; reordering whole rows is the argsort-plus-indexing pattern from the last
segment, applied to axis 0.


In [ ]:
import torch as t

m = t.tensor([[3.0, 1.0, 2.0],
              [9.0, 7.0, 8.0]])
print("dim=1 (within each row)")
print(t.sort(m, dim=1).values)
print("dim=0 (down each column)")
print(t.sort(m, dim=0).values)




**`t.topk(z, k)`** answers a narrower question: the k largest values, already
ordered largest first, plus their positions. Sorting the whole tensor to slice
off k of them does strictly more work — and on a long vector, a great deal more.
Like `sort`, it hands back a `(values, indices)` pair.


In [ ]:
v = t.tensor([5.0, 1.0, 9.0, 3.0, 7.0])
top = t.topk(v, 2)
print(top)
assert top.values.tolist() == [9.0, 7.0]
assert t.equal(top.values, t.sort(v, descending=True).values[:2])


In [ ]:
import torch as t

m = t.tensor([[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]])

# dim=1 sorts WITHIN each row, independently of the other rows.
assert t.sort(m, dim=1).values.tolist() == [[1.0, 2.0, 3.0], [7.0, 8.0, 9.0]]

# dim=0 does the same down the columns.
assert t.sort(m, dim=0).values.tolist() == [[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]]

# argsort takes the same dim= keyword, and answers with positions not values.
assert t.argsort(m, dim=1).tolist() == [[1, 2, 0], [1, 2, 0]]

# ...and the same descending= keyword you met a segment ago.
z = t.tensor([0.5, 0.25, 0.75, 0.125])
assert t.argsort(z, descending=True).tolist() == [2, 0, 1, 3]

# topk: the k largest, largest first — a pair again, so take .values.
top = t.topk(z, 2)
assert top.values.tolist() == [0.75, 0.5]
assert top.indices.tolist() == [2, 0]
print("dim=1 (within rows)")
print(t.sort(m, dim=1).values)
print("dim=0 (down columns) — unchanged, columns were already ascending")
print(t.sort(m, dim=0).values)
print("topk:", top)




The second assertion is the one to sit with: sorting down the columns left this
matrix unchanged, because every column was already ascending. Sorting along an
axis tells you nothing about the other axis.


<!-- dd:dd-q521 -->

### Problem 521 · faded — your turn

Per-row rankings, largest first — an axis and a direction at the same time.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 0, 2],
        [0, 2, 1]])
```


In [ ]:
import torch as t

def solve(x):
    """Return each row's column indices ordered by that row's values, largest first."""
    return t._____(x, _____=1, _____=True)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[0.4, 0.9, 0.1], [5.0, 1.0, 3.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(521)


In [ ]:
#@title 💡 Solution — Problem 521
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return each row's ranking positions, descending."""
    return t.argsort(x, dim=1, descending=True)


example = t.tensor([[0.4, 0.9, 0.1], [5.0, 1.0, 3.0]])
print(solve(example))


In [ ]:
import torch as t

m = t.tensor([[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]])

# dim=1 sorts WITHIN each row, independently of the other rows.
assert t.sort(m, dim=1).values.tolist() == [[1.0, 2.0, 3.0], [7.0, 8.0, 9.0]]

# dim=0 does the same down the columns.
assert t.sort(m, dim=0).values.tolist() == [[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]]

# argsort takes the same dim= keyword, and answers with positions not values.
assert t.argsort(m, dim=1).tolist() == [[1, 2, 0], [1, 2, 0]]

# ...and the same descending= keyword you met a segment ago.
z = t.tensor([0.5, 0.25, 0.75, 0.125])
assert t.argsort(z, descending=True).tolist() == [2, 0, 1, 3]

# topk: the k largest, largest first — a pair again, so take .values.
top = t.topk(z, 2)
assert top.values.tolist() == [0.75, 0.5]
assert top.indices.tolist() == [2, 0]
print("dim=1 (within rows)")
print(t.sort(m, dim=1).values)
print("dim=0 (down columns) — unchanged, columns were already ascending")
print(t.sort(m, dim=0).values)
print("topk:", top)




The second assertion is the one to sit with: sorting down the columns left this
matrix unchanged, because every column was already ascending. Sorting along an
axis tells you nothing about the other axis.


<!-- dd:dd-q662 -->

### Problem 662 · faded — your turn

The top k, with their positions.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([9.0, 7.0], [2, 4])
```


In [ ]:
import torch as t

def solve(x, k):
    """Return (the k largest values largest first, their positions), as lists."""
    top = t._____(x, k)
    return (top._____.tolist(), top._____.tolist())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([5.0, 1.0, 9.0, 3.0, 7.0]), 2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(662)


In [ ]:
#@title 💡 Solution — Problem 662
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, k):
    """Return (the k largest values largest first, their positions), as lists."""
    top = t.topk(x, k)
    return (top.values.tolist(), top.indices.tolist())


example = (t.tensor([5.0, 1.0, 9.0, 3.0, 7.0]), 2,)
print(solve(*example))


<!-- dd:dd-q516 -->

### Problem 516 · guided

Write a function solve(z) that takes a 1-D float tensor and returns a NEW tensor with the same values ordered from largest to smallest. The input must be left unchanged.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2.0000, 1.0000, 0.5000])
```


<details>
<summary>Hints</summary>

1. A NEW tensor, so the input has to survive untouched.
2. Sorting returns both values and the positions they came from; you want
   the values half, and there is a keyword for the direction.
3. `t.sort(z, descending=True).values`.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return the values of z, largest first."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([0.5, 2.0, 1.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(516)


In [ ]:
#@title 💡 Solution — Problem 516
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    """Return the values of z, largest first."""
    return t.sort(z, descending=True).values


example = t.tensor([0.5, 2.0, 1.0])
print(solve(example))


<!-- dd:dd-q517 -->

### Problem 517 · guided

Write a function solve(z) that takes a 1-D float tensor and returns a tensor of indices: the position each element would come from if z were sorted smallest to largest. Sorting gives you the values; argsort gives you where they came from — which is what you need when a second tensor has to be reordered the same way.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 0, 2])
```


<details>
<summary>Hints</summary>

1. Not the sorted values — where each sorted value WOULD have come from.
2. The result is an index tensor the same length as z, and result[0] is the
   position of the smallest element.
3. `t.argsort(z)`.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return the POSITIONS that would sort z ascending."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([0.4, 0.1, 0.9])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(517)


In [ ]:
#@title 💡 Solution — Problem 517
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    """Return the POSITIONS that would sort z ascending."""
    return t.argsort(z)


example = t.tensor([0.4, 0.1, 0.9])
print(solve(example))


<!-- dd:dd-q518 -->

### Problem 518 · independent

Write a function solve(x) that takes a 2-D float tensor and returns a tensor of the same shape in which each row has been sorted ascending on its own. Rows do not mix — name the axis you want sorted along.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 2., 3.],
        [7., 8., 9.]])
```


In [ ]:
import torch as t

def solve(x):
    """Sort each ROW of a 2-D tensor independently."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(518)


In [ ]:
#@title 💡 Solution — Problem 518
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Sort each ROW of a 2-D tensor independently."""
    return t.sort(x, dim=1).values


example = t.tensor([[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]])
print(solve(example))


<!-- dd:dd-q519 -->

### Problem 519 · independent

Write a function solve(z, k) that takes a 1-D float tensor and an int, and returns the k largest values ordered largest first. Sorting the whole tensor to take k of them does more work than the question asks for; there is an operation that answers exactly this.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.9000, 0.7000])
```


In [ ]:
import torch as t

def solve(z, k):
    """Return the k largest values, largest first."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([0.4, 0.9, 0.1, 0.7]), 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(519)


In [ ]:
#@title 💡 Solution — Problem 519
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, k):
    """Return the k largest values, largest first."""
    return t.topk(z, k).values


example = (t.tensor([0.4, 0.9, 0.1, 0.7]), 2)
print(solve(*example))


<!-- dd:dd-q522 -->

### Problem 522 · independent

Write a function solve(x, col) that returns the rows of the 2-D float tensor x reordered so that column `col` is ascending, as a plain nested list. Rows must stay intact — pull the key column out, rank it, and use that ranking to index the whole matrix along axis 0.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[1.0, 2.0], [2.0, 3.0], [3.0, 1.0]]
```


In [ ]:
import torch as t

def solve(x, col):
    """Sort the ROWS of a matrix by one column's values."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[3.0, 1.0], [1.0, 2.0], [2.0, 3.0]]), 0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(522)


In [ ]:
#@title 💡 Solution — Problem 522
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, col):
    """Sort the ROWS of a matrix by one column's values."""
    order = t.argsort(x[:, col])
    return x[order].tolist()


example = (t.tensor([[3.0, 1.0], [1.0, 2.0], [2.0, 3.0]]), 0)
print(solve(*example))


<!-- dd:dd-q663 -->

### Problem 663 · independent

Write a function solve(x) that takes a 2-D float tensor and returns it with each COLUMN sorted smallest to largest independently, as a plain nested list. Columns do not mix — name the axis that runs down a column.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[1.0, 0.0], [2.0, 1.0], [3.0, 2.0]]
```


In [ ]:
import torch as t


def solve(x):
    """Return x with each COLUMN sorted ascending on its own, as a nested list."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[3.0, 1.0], [1.0, 2.0], [2.0, 0.0]]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(663)


In [ ]:
#@title 💡 Solution — Problem 663
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return x with each COLUMN sorted ascending on its own, as a nested list."""
    return t.sort(x, dim=0).values.tolist()


example = (t.tensor([[3.0, 1.0], [1.0, 2.0], [2.0, 0.0]]),)
print(solve(*example))


<!-- dd:dd-q664 -->

### Problem 664 · independent

Write a function solve(z) that takes a 1-D float tensor and returns the POSITION of its largest entry as a plain int. Rank the positions largest-first and take the first one.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
1
```


In [ ]:
import torch as t


def solve(z):
    """Return the position of the largest entry, via a descending argsort."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([0.4, 0.9, 0.1]),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(664)


In [ ]:
#@title 💡 Solution — Problem 664
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    """Return the position of the largest entry, via a descending argsort."""
    return t.argsort(z, descending=True)[0].item()


example = (t.tensor([0.4, 0.9, 0.1]),)
print(solve(*example))


<!-- dd:dd-q665 -->

### Problem 665 · independent

Write a function solve(z, k) that takes a 1-D float tensor and an integer k, and returns the k SMALLEST values ordered smallest first, as a plain list. topk answers the opposite question — build this one from a sort.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[1.0, 3.0]
```


In [ ]:
import torch as t


def solve(z, k):
    """Return the k SMALLEST values, smallest first, as a list."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([5.0, 1.0, 9.0, 3.0, 7.0]), 2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(665)


In [ ]:
#@title 💡 Solution — Problem 665
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, k):
    """Return the k SMALLEST values, smallest first, as a list."""
    return t.sort(z).values[:k].tolist()


example = (t.tensor([5.0, 1.0, 9.0, 3.0, 7.0]), 2,)
print(solve(*example))


#### Common mistakes

- **"`t.sort(z)` returns the sorted tensor."** — It returns a
  `(values, indices)` pair. Take `.values`, or unpack both.
- **"`z.sort()` sorts in place, like the NumPy method."** — It does not. There
  is no in-place sort; the input is never modified.
- **"There's no descending option, so I'll reverse the result."** — There is:
  `descending=True`. And the NumPy reversal idiom `[::-1]` would raise here
  anyway.
- **"Sorting a 2-D tensor sorts the rows as units."** — `t.sort(z, dim=1)`
  sorts *within* each row independently, destroying row integrity. Keeping
  rows intact while reordering them is an argsort + indexing pattern (later
  KP).


<!-- dd:dd-kp-numpy-random-samplers -->

## Drawing random numbers

`numpy.random-samplers`


Four functions cover almost all the random tensors you will ever need. They are
constructors like `t.zeros` and `t.ones` — you say what SHAPE you want and get a
tensor of that shape back — except the numbers are drawn rather than fixed.


In [ ]:
import torch as t

print("rand    ", t.rand(3))            # uniform floats in [0, 1)
print("randn   ", t.randn(3))           # standard normal: mean 0, spread 1
print("randint ", t.randint(0, 10, (3,)))   # whole numbers in [0, 10)
print("randperm", t.randperm(6))        # 0..5 shuffled into a random order




The four differ in **what they draw from**, not in how you call them:

| Call | What comes out |
|---|---|
| `t.rand(shape)` | floats spread evenly across `[0, 1)` — 0 possible, 1 never |
| `t.randn(shape)` | floats from the standard normal — negatives are normal here |
| `t.randint(low, high, shape)` | whole numbers in `[low, high)` — `high` excluded |
| `t.randperm(n)` | every number `0..n-1` exactly once, in a random order |

`rand` and `randn` take the shape the way the constructors do — loose integers or
a tuple, whichever reads better:


In [ ]:
import torch as t

print(t.rand(2, 3).shape)      # loose integers
print(t.rand((2, 3)).shape)    # the same shape as a tuple
print(t.randn(4).shape)        # 1-D, four numbers




`randint` is the odd one out: its shape argument comes **third and must be a
tuple**, because the first two slots are already spoken for by the range.


In [ ]:
import torch as t

x = t.randint(0, 10, (5,))
print(x, x.dtype)              # note the dtype: whole numbers, not floats
print(t.randint(0, 10, (2, 3)))




`randperm` is not a draw from a distribution at all — it is a **shuffling**.
Every number from `0` to `n-1` comes out exactly once, which is what makes it
the tool for putting things in a random order rather than picking random things.


In [ ]:
import torch as t

p = t.randperm(5)
print("a permutation:", p)
print("sorted back  :", sorted(p.tolist()))
assert sorted(p.tolist()) == [0, 1, 2, 3, 4]




That last assert is the whole difference between `randperm` and `randint`:
`randint` draws **with replacement**, so it repeats values and skips others.


In [ ]:
import torch as t

print("randint may repeat:", t.randint(0, 5, (5,)))
print("randperm never does:", t.randperm(5))




Because a draw is different every time you run it, the things you can *assert*
about a random tensor are its structure and its bounds — shape, dtype, and the
range the values fall in — not the values themselves. That is what the drills
below ask for.


> **Watch out.** - **`t.randint`'s `high` is EXCLUSIVE** — `t.randint(0, 10, (5,))` never
  produces a 10. Same convention as `range` and as slicing.
- **`t.randint` wants a SHAPE, not a count** — the third argument is `(n,)`,
  with the trailing comma, not `n`. `(n, 1)` is a column, which is a different
  tensor.
- **`t.rand` is not `t.randn`** — one is uniform on `[0, 1)`, the other is the
  standard normal and goes negative. They have the same shape and the same
  dtype, so the shape cannot tell you which one you called.
- **`t.randperm(n)` takes a COUNT, not a shape** — it is always 1-D, and it is a
  permutation of `0..n-1`, never of your own values.


Task: draw with each of the four samplers and report the facts about the result
that are true on every run.


In [ ]:
import torch as t

# Uniform: shape you asked for, floats, every value in [0, 1).
u = t.rand(2, 3)
assert tuple(u.shape) == (2, 3)
assert str(u.dtype) == "torch.float32"
assert bool((u >= 0).all())
assert bool((u < 1).all())
print("uniform:", u)

# Normal: same shape rules, same dtype, but NOT bounded to [0, 1).
g = t.randn(2, 3)
assert tuple(g.shape) == (2, 3)
assert str(g.dtype) == "torch.float32"
print("normal :", g)

# Ints: the range comes first, the SHAPE third, and the dtype is integral.
i = t.randint(0, 10, (5,))
assert tuple(i.shape) == (5,)
assert str(i.dtype) == "torch.int64"
assert bool((i >= 0).all())
assert bool((i < 10).all())
print("ints   :", i)

# Permutation: a count, not a shape — and every value appears exactly once.
p = t.randperm(5)
assert tuple(p.shape) == (5,)
assert sorted(p.tolist()) == [0, 1, 2, 3, 4]
print("perm   :", p)




Why each step:

1. Asserting shape and dtype rather than values is the only thing that can be
   true on every run — and it is what the graders below check.
2. The bound checks are what distinguish `rand` from `randn` in code: only the
   uniform one is trapped in `[0, 1)`.
3. `sorted(p.tolist())` is the definition of a permutation, written out. If it
   ever came back with a repeat, `randperm` would not be doing its job.


<!-- dd:dd-q42 -->

### Problem 42 · faded — your turn

A uniform random tensor of a given 3-D shape.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0.4963, 0.7682, 0.0885],
         [0.1320, 0.3074, 0.6341],
         [0.4901, 0.8964, 0.4556]],

        [[0.6323, 0.3489, 0.4017],
         [0.0223, 0.1689, 0.2939],
         [0.5185, 0.6977, 0.8000]],

        [[0.1610, 0.2823, 0.6816],
         [0.9152, 0.3971, 0.8742],
         [0.4194, 0.5529, 0.9527]]])
```


In [ ]:
import torch as t

def solve(shape):
    """Return a tensor of the given shape, uniform in [0, 1)."""
    return t._____(shape)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
t.manual_seed(0)
example = (3, 3, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(42)


In [ ]:
#@title 💡 Solution — Problem 42
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(shape):
    return t.rand(shape)


t.manual_seed(0)
example = (3, 3, 3)
print(solve(example))


Task: draw with each of the four samplers and report the facts about the result
that are true on every run.


In [ ]:
import torch as t

# Uniform: shape you asked for, floats, every value in [0, 1).
u = t.rand(2, 3)
assert tuple(u.shape) == (2, 3)
assert str(u.dtype) == "torch.float32"
assert bool((u >= 0).all())
assert bool((u < 1).all())
print("uniform:", u)

# Normal: same shape rules, same dtype, but NOT bounded to [0, 1).
g = t.randn(2, 3)
assert tuple(g.shape) == (2, 3)
assert str(g.dtype) == "torch.float32"
print("normal :", g)

# Ints: the range comes first, the SHAPE third, and the dtype is integral.
i = t.randint(0, 10, (5,))
assert tuple(i.shape) == (5,)
assert str(i.dtype) == "torch.int64"
assert bool((i >= 0).all())
assert bool((i < 10).all())
print("ints   :", i)

# Permutation: a count, not a shape — and every value appears exactly once.
p = t.randperm(5)
assert tuple(p.shape) == (5,)
assert sorted(p.tolist()) == [0, 1, 2, 3, 4]
print("perm   :", p)




Why each step:

1. Asserting shape and dtype rather than values is the only thing that can be
   true on every run — and it is what the graders below check.
2. The bound checks are what distinguish `rand` from `randn` in code: only the
   uniform one is trapped in `[0, 1)`.
3. `sorted(p.tolist())` is the definition of a permutation, written out. If it
   ever came back with a repeat, `randperm` would not be doing its job.


<!-- dd:dd-q676 -->

### Problem 676 · faded — your turn

The shape, dtype and rank of a standard-normal draw.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((2, 3), 'torch.float32', 2)
```


In [ ]:
import torch as t

def solve(rows, cols):
    """Return (shape, dtype name, ndim) of a standard-normal draw."""
    x = t._____(rows, cols)
    return (tuple(x.shape), str(x.dtype), x.ndim)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(676)


In [ ]:
#@title 💡 Solution — Problem 676
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols):
    """Return (shape, dtype name, ndim) of a standard-normal draw."""
    x = t.randn(rows, cols)
    return (tuple(x.shape), str(x.dtype), x.ndim)


example = (2, 3,)
print(solve(*example))


Task: draw with each of the four samplers and report the facts about the result
that are true on every run.


In [ ]:
import torch as t

# Uniform: shape you asked for, floats, every value in [0, 1).
u = t.rand(2, 3)
assert tuple(u.shape) == (2, 3)
assert str(u.dtype) == "torch.float32"
assert bool((u >= 0).all())
assert bool((u < 1).all())
print("uniform:", u)

# Normal: same shape rules, same dtype, but NOT bounded to [0, 1).
g = t.randn(2, 3)
assert tuple(g.shape) == (2, 3)
assert str(g.dtype) == "torch.float32"
print("normal :", g)

# Ints: the range comes first, the SHAPE third, and the dtype is integral.
i = t.randint(0, 10, (5,))
assert tuple(i.shape) == (5,)
assert str(i.dtype) == "torch.int64"
assert bool((i >= 0).all())
assert bool((i < 10).all())
print("ints   :", i)

# Permutation: a count, not a shape — and every value appears exactly once.
p = t.randperm(5)
assert tuple(p.shape) == (5,)
assert sorted(p.tolist()) == [0, 1, 2, 3, 4]
print("perm   :", p)




Why each step:

1. Asserting shape and dtype rather than values is the only thing that can be
   true on every run — and it is what the graders below check.
2. The bound checks are what distinguish `rand` from `randn` in code: only the
   uniform one is trapped in `[0, 1)`.
3. `sorted(p.tolist())` is the definition of a permutation, written out. If it
   ever came back with a repeat, `randperm` would not be doing its job.


<!-- dd:dd-q677 -->

### Problem 677 · faded — your turn

Random integers, and a check that they landed inside the half-open range.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((5,), True, True)
```


In [ ]:
import torch as t

def solve(low, high, n):
    """Return (shape, every value >= low?, every value < high?) for n random ints."""
    x = t._____(low, high, (n,))
    return (tuple(x.shape), bool((x >= low).all()), bool((x < high).all()))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (0, 10, 5,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(677)


In [ ]:
#@title 💡 Solution — Problem 677
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(low, high, n):
    """Return (shape, every value >= low?, every value < high?) for n random ints."""
    x = t.randint(low, high, (n,))
    return (tuple(x.shape), bool((x >= low).all()), bool((x < high).all()))


example = (0, 10, 5,)
print(solve(*example))


<!-- dd:dd-q678 -->

### Problem 678 · independent

Write a function solve(n) that draws n uniform random floats and returns a tuple (shape, all_at_least_zero, all_below_one, dtype_name).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((5,), True, True, 'torch.float32')
```


In [ ]:
import torch as t


def solve(n):
    """Return (shape, all >= 0?, all < 1?, dtype name) for n uniform draws."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (5,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(678)


In [ ]:
#@title 💡 Solution — Problem 678
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    """Return (shape, all >= 0?, all < 1?, dtype name) for n uniform draws."""
    x = t.rand(n)
    return (tuple(x.shape), bool((x >= 0).all()), bool((x < 1).all()), str(x.dtype))


example = (5,)
print(solve(*example))


<!-- dd:dd-q679 -->

### Problem 679 · independent

Write a function solve(n) that draws a random permutation of the whole numbers 0 to n-1 and returns a tuple (shape, sorted_values): the shape as a plain tuple and the drawn values sorted back into order as a plain list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((4,), [0, 1, 2, 3])
```


In [ ]:
import torch as t


def solve(n):
    """Return (shape, the values sorted back into order) for a random permutation."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (4,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(679)


In [ ]:
#@title 💡 Solution — Problem 679
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    """Return (shape, the values sorted back into order) for a random permutation."""
    p = t.randperm(n)
    return (tuple(p.shape), sorted(p.tolist()))


example = (4,)
print(solve(*example))


<!-- dd:dd-q680 -->

### Problem 680 · independent

Write a function solve(rows) that builds a tensor from the nested list, shuffles its ROWS into a random order, and returns a tuple (shape, sorted_rows): the shuffled tensor's shape as a plain tuple and its rows as a plain nested list sorted back into a canonical order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((3, 2), [[1, 2], [3, 4], [5, 6]])
```


In [ ]:
import torch as t


def solve(rows):
    """Shuffle a's rows at random; return (shape, the rows sorted back into order)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2], [3, 4], [5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(680)


In [ ]:
#@title 💡 Solution — Problem 680
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows):
    """Shuffle a's rows at random; return (shape, the rows sorted back into order)."""
    a = t.tensor(rows)
    order = t.randperm(a.shape[0])
    shuffled = a[order]
    return (tuple(shuffled.shape), sorted(shuffled.tolist()))


example = ([[1, 2], [3, 4], [5, 6]],)
print(solve(*example))


<!-- dd:dd-q681 -->

### Problem 681 · independent

Write a function solve(rows, cols, high) that draws a 2-D tensor of random integers from [0, high) and returns a tuple (shape, all_at_least_zero, all_below_high, dtype_name).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((2, 3), True, True, 'torch.int64')
```


In [ ]:
import torch as t


def solve(rows, cols, high):
    """Return (shape, all >= 0?, all < high?, dtype name) for a 2-D random int draw."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3, 10,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(681)


In [ ]:
#@title 💡 Solution — Problem 681
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols, high):
    """Return (shape, all >= 0?, all < high?, dtype name) for a 2-D random int draw."""
    x = t.randint(0, high, (rows, cols))
    return (tuple(x.shape), bool((x >= 0).all()), bool((x < high).all()), str(x.dtype))


example = (2, 3, 10,)
print(solve(*example))


<!-- dd:dd-q682 -->

### Problem 682 · independent

Write a function solve(shape) that draws a uniform random tensor of the given shape (handed to you as a plain tuple) and returns a tuple (shape, ndim, numel, dtype_name).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((2, 3), 2, 6, 'torch.float32')
```


In [ ]:
import torch as t


def solve(shape):
    """Return (shape, ndim, numel, dtype name) for a uniform draw of the given shape."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ((2, 3),)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(682)


In [ ]:
#@title 💡 Solution — Problem 682
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(shape):
    """Return (shape, ndim, numel, dtype name) for a uniform draw of the given shape."""
    x = t.rand(shape)
    return (tuple(x.shape), x.ndim, x.numel(), str(x.dtype))


example = ((2, 3),)
print(solve(*example))


<!-- dd:dd-q683 -->

### Problem 683 · independent

Write a function solve(rows, cols) that draws a standard-normal tensor of shape (rows, cols) and returns a tuple (shape, numel, first_row_length, dtype_name), where first_row_length is how many numbers the first row holds.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((2, 3), 6, 3, 'torch.float32')
```


In [ ]:
import torch as t


def solve(rows, cols):
    """Return (shape, numel, length of row 0, dtype name) for a 2-D normal draw."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(683)


In [ ]:
#@title 💡 Solution — Problem 683
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols):
    """Return (shape, numel, length of row 0, dtype name) for a 2-D normal draw."""
    x = t.randn(rows, cols)
    return (tuple(x.shape), x.numel(), len(x[0].tolist()), str(x.dtype))


example = (2, 3,)
print(solve(*example))


<!-- dd:dd-kp-numpy-random-seeding -->

## Seeding your own reproducible stream

`numpy.random-seeding`


So far the generator has always arrived from somewhere else. Making one is two
calls, and they are almost always written as one line:


In [ ]:
import torch as t

rng = t.Generator().manual_seed(0)
print(rng)
print(t.rand(3, generator=rng))




`t.Generator()` builds an empty stream; `.manual_seed(seed)` fixes where that
stream starts and **returns the generator**, which is why the two chain onto one
line. An unseeded `t.Generator()` still works — it just starts somewhere nobody
chose, so nothing about it can be reproduced.

Every sampler takes the generator the same way — as a keyword argument named
`generator`, always last:


In [ ]:
import torch as t

rng = t.Generator().manual_seed(0)
print("rand    ", t.rand(3, generator=rng))
print("randn   ", t.randn(3, generator=rng))
print("randint ", t.randint(0, 10, (3,), generator=rng))
print("randperm", t.randperm(4, generator=rng))




Note the shape of the API. In NumPy you call **methods on** the generator —
`rng.random(5)`. In PyTorch the generator is an **argument** to an ordinary
function — `t.rand(5, generator=rng)`. Same idea, inverted spelling, and it is
the thing most often got wrong coming from NumPy. `randint` keeps its own
argument order: the shape stays third and `generator=` comes after it.

The seed is not a source of randomness. It is a **name for a whole sequence**.
Two generators given the same seed are not merely similar; they are the same
stream, and they stay in step for as long as you draw from them equally:


In [ ]:
import torch as t

a = t.Generator().manual_seed(42)
b = t.Generator().manual_seed(42)
print("a:", t.rand(3, generator=a))
print("b:", t.rand(3, generator=b))




Draws consume that sequence, so position matters as much as the seed. After the
draws above, `a` and `b` are three numbers in — and a fresh generator on the
same seed is back at the start:


In [ ]:
import torch as t

a = t.Generator().manual_seed(42)
first = t.rand(3, generator=a)
second = t.rand(3, generator=a)
fresh = t.rand(3, generator=t.Generator().manual_seed(42))

print("first :", first)
print("second:", second)
print("fresh :", fresh)
assert first.tolist() != second.tolist()   # the stream moved on
assert fresh.tolist() == first.tolist()    # the seed replayed it




That pair of asserts is the entire contract. It is also why an *extra* draw
slipped in anywhere shifts every value after it: you have not changed the
sequence, you have changed your position in it.

Reseeding an existing generator rewinds it to the same starting point:


In [ ]:
import torch as t

rng = t.Generator().manual_seed(5)
before = t.rand(2, generator=rng)
rng.manual_seed(5)                      # back to the beginning of the stream
after = t.rand(2, generator=rng)
print("before:", before)
print("after :", after)
assert before.tolist() == after.tolist()




Do that to a generator **you** made. Never to one you were handed — that is the
caller's position in their stream, and rewinding it is the bug the previous
lesson warned about.

You will also meet the **global** API in the wild: `t.manual_seed(0)` followed by
plain `t.rand(shape)` with no generator. It seeds one shared, process-level
stream. It works, and plenty of code uses it, but any function anywhere can draw
from it and move everyone else along. For anything you need to reproduce, prefer
an explicit `Generator`: no hidden global state, and no interference between
distant pieces of code.


> **Watch out.** - **`t.Generator()` on its own is not reproducible.** Without `.manual_seed(...)`
  it starts from a state nobody chose. Seeding is the whole point.
- **`.manual_seed` returns the generator**, which is what makes
  `t.Generator().manual_seed(0)` one expression. It also mutates in place, so
  `rng.manual_seed(0)` on an existing generator rewinds it.
- **A seed names a SEQUENCE, not a number.** Same seed and same draw order gives
  the same values; change either and everything after the change moves.
- **Never reseed a generator you were handed.** Your own: fine. The caller's:
  that is their stream position, and rewinding it corrupts their run.
- **`t.manual_seed(0)` and `t.Generator().manual_seed(0)` are different things.**
  The first seeds the shared global stream; the second seeds a private one.
- **`generator=` is a KEYWORD argument.** It never rides along positionally:
  `t.rand(n, rng)` is not the same call and will not do what you want.


Task: build a seeded stream, show that the same seed replays it exactly, and
show that consuming the stream moves you along it.


In [ ]:
import torch as t

# One expression: build the stream, then fix where it starts.
rng = t.Generator().manual_seed(42)

first = t.rand(3, generator=rng)
print("first draw :", first)

# The SAME generator again: the stream has advanced past the first three.
second = t.rand(3, generator=rng)
print("second draw:", second)
assert first.tolist() != second.tolist()

# A NEW generator on the same seed starts the same sequence over.
replay = t.rand(3, generator=t.Generator().manual_seed(42))
assert replay.tolist() == first.tolist()
print("replayed   :", replay)

# Reseeding in place does the same thing to a generator you own.
rng.manual_seed(42)
rewound = t.rand(3, generator=rng)
assert rewound.tolist() == first.tolist()
print("rewound    :", rewound)

# A different seed is a different sequence entirely.
other = t.rand(3, generator=t.Generator().manual_seed(43))
assert other.tolist() != first.tolist()
print("seed 43    :", other)




Why each step:

1. Building and seeding in one line is the idiom; the assert on `second` is what
   "each draw consumes the stream" means, written as code.
2. `replay` and `rewound` are the same claim reached two ways — a fresh
   generator on the seed, and rewinding the one you have. Both land at the
   start of the same sequence.
3. The last block is the check people forget: reproducibility is only useful if
   *different* seeds really do give different runs.


<!-- dd:dd-q696 -->

### Problem 696 · faded — your turn

Build your own seeded generator and take the first draw off it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[0.49625658988952637, 0.7682217955589294, 0.08847743272781372]
```


In [ ]:
import torch as t

def solve(seed, n):
    """Seed a new generator and return its first n uniform floats."""
    rng = t._________()._________(seed)
    return t.rand(n, _____=rng).tolist()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (0, 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(696)


In [ ]:
#@title 💡 Solution — Problem 696
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(seed, n):
    """Seed a new generator and return its first n uniform floats."""
    rng = t.Generator().manual_seed(seed)
    return t.rand(n, generator=rng).tolist()


example = (0, 3,)
print(solve(*example))


Task: build a seeded stream, show that the same seed replays it exactly, and
show that consuming the stream moves you along it.


In [ ]:
import torch as t

# One expression: build the stream, then fix where it starts.
rng = t.Generator().manual_seed(42)

first = t.rand(3, generator=rng)
print("first draw :", first)

# The SAME generator again: the stream has advanced past the first three.
second = t.rand(3, generator=rng)
print("second draw:", second)
assert first.tolist() != second.tolist()

# A NEW generator on the same seed starts the same sequence over.
replay = t.rand(3, generator=t.Generator().manual_seed(42))
assert replay.tolist() == first.tolist()
print("replayed   :", replay)

# Reseeding in place does the same thing to a generator you own.
rng.manual_seed(42)
rewound = t.rand(3, generator=rng)
assert rewound.tolist() == first.tolist()
print("rewound    :", rewound)

# A different seed is a different sequence entirely.
other = t.rand(3, generator=t.Generator().manual_seed(43))
assert other.tolist() != first.tolist()
print("seed 43    :", other)




Why each step:

1. Building and seeding in one line is the idiom; the assert on `second` is what
   "each draw consumes the stream" means, written as code.
2. `replay` and `rewound` are the same claim reached two ways — a fresh
   generator on the seed, and rewinding the one you have. Both land at the
   start of the same sequence.
3. The last block is the check people forget: reproducibility is only useful if
   *different* seeds really do give different runs.


<!-- dd:dd-q697 -->

### Problem 697 · faded — your turn

Two generators on one seed, and the proof they agree.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0.8822692632675171, 0.9150039553642273, 0.38286375999450684], [0.8822692632675171, 0.9150039553642273, 0.38286375999450684], True)
```


In [ ]:
import torch as t

def solve(seed, n):
    """Two generators, one seed: return (draw a, draw b, are they identical?)."""
    rng_a = t._________()._________(seed)
    rng_b = t._________()._________(seed)
    a = t.rand(n, _____=rng_a)
    b = t.rand(n, _____=rng_b)
    return (a.tolist(), b.tolist(), a.tolist() == b.tolist())


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (42, 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(697)


In [ ]:
#@title 💡 Solution — Problem 697
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(seed, n):
    """Two generators, one seed: return (draw a, draw b, are they identical?)."""
    rng_a = t.Generator().manual_seed(seed)
    rng_b = t.Generator().manual_seed(seed)
    a = t.rand(n, generator=rng_a)
    b = t.rand(n, generator=rng_b)
    return (a.tolist(), b.tolist(), a.tolist() == b.tolist())


example = (42, 3,)
print(solve(*example))


<!-- dd:dd-q694 -->

### Problem 694 · independent

Write a function solve(rng, n) that draws n standard-normal floats from rng's stream and returns a tuple (values, shape, dtype_name).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1.5409960746765137, -0.293428897857666], (2,), 'torch.float32')
```


In [ ]:
import torch as t


def solve(rng, n):
    """Return (values, shape, dtype name) for n normal draws from rng's stream."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.Generator().manual_seed(0), 2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(694)


In [ ]:
#@title 💡 Solution — Problem 694
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, n):
    """Return (values, shape, dtype name) for n normal draws from rng's stream."""
    x = t.randn(n, generator=rng)
    return (x.tolist(), tuple(x.shape), str(x.dtype))


example = (t.Generator().manual_seed(0), 2,)
print(solve(*example))


<!-- dd:dd-q698 -->

### Problem 698 · independent

Write a function solve(seed, n) that shows a seeded stream replays: build a generator from seed and draw n floats, then build a SECOND generator from the same seed and draw n floats again. Return a tuple (draw, replay, they_match).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0.49625658988952637, 0.7682217955589294], [0.49625658988952637, 0.7682217955589294], True)
```


In [ ]:
import torch as t


def solve(seed, n):
    """Show that reseeding replays: return (first draw, replayed draw, do they match?)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (0, 2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(698)


In [ ]:
#@title 💡 Solution — Problem 698
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(seed, n):
    """Show that reseeding replays: return (first draw, replayed draw, do they match?)."""
    first = t.rand(n, generator=t.Generator().manual_seed(seed))
    replay = t.rand(n, generator=t.Generator().manual_seed(seed))
    return (first.tolist(), replay.tolist(), first.tolist() == replay.tolist())


example = (0, 2,)
print(solve(*example))


<!-- dd:dd-q699 -->

### Problem 699 · independent

Write a function solve(seed_a, seed_b, n) that draws n uniform floats from a generator seeded with seed_a and n from one seeded with seed_b, returning a tuple (draw_a, draw_b, they_differ).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0.49625658988952637, 0.7682217955589294], [0.7576315999031067, 0.2793108820915222], True)
```


In [ ]:
import torch as t


def solve(seed_a, seed_b, n):
    """Two different seeds: return (draw a, draw b, do they differ?)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (0, 1, 2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(699)


In [ ]:
#@title 💡 Solution — Problem 699
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(seed_a, seed_b, n):
    """Two different seeds: return (draw a, draw b, do they differ?)."""
    a = t.rand(n, generator=t.Generator().manual_seed(seed_a))
    b = t.rand(n, generator=t.Generator().manual_seed(seed_b))
    return (a.tolist(), b.tolist(), a.tolist() != b.tolist())


example = (0, 1, 2,)
print(solve(*example))


<!-- dd:dd-q700 -->

### Problem 700 · independent

Write a function solve(seed, n) that builds one seeded generator and takes TWO consecutive draws of n uniform floats from it, returning a tuple (first, second, they_differ) — the evidence that each draw consumes the stream.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0.49625658988952637, 0.7682217955589294], [0.08847743272781372, 0.13203048706054688], True)
```


In [ ]:
import torch as t


def solve(seed, n):
    """One seeded stream, two draws: return (first, second, do they differ?)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (0, 2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(700)


In [ ]:
#@title 💡 Solution — Problem 700
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(seed, n):
    """One seeded stream, two draws: return (first, second, do they differ?)."""
    rng = t.Generator().manual_seed(seed)
    first = t.rand(n, generator=rng)
    second = t.rand(n, generator=rng)
    return (first.tolist(), second.tolist(), first.tolist() != second.tolist())


example = (0, 2,)
print(solve(*example))


<!-- dd:dd-q701 -->

### Problem 701 · independent

Write a function solve(seed, n) that draws n uniform floats from a seeded generator, then RESEEDS that same generator with the same seed and draws n again. Return a tuple (before, after, restarted) — whether reseeding put the stream back to the beginning.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0.49625658988952637, 0.7682217955589294], [0.49625658988952637, 0.7682217955589294], True)
```


In [ ]:
import torch as t


def solve(seed, n):
    """Reseed mid-stream; return (draw before, draw after, did it restart?)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (0, 2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(701)


In [ ]:
#@title 💡 Solution — Problem 701
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(seed, n):
    """Reseed mid-stream; return (draw before, draw after, did it restart?)."""
    rng = t.Generator().manual_seed(seed)
    before = t.rand(n, generator=rng)
    rng.manual_seed(seed)
    after = t.rand(n, generator=rng)
    return (before.tolist(), after.tolist(), before.tolist() == after.tolist())


example = (0, 2,)
print(solve(*example))


<!-- dd:dd-q702 -->

### Problem 702 · independent

Write a function solve(seed, low, high, n) that builds a generator from seed and returns the first n random integers from [low, high) off that stream, as a plain list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[4, 9, 3, 0]
```


In [ ]:
import torch as t


def solve(seed, low, high, n):
    """Seed a generator and return its first n random ints in [low, high)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (0, 0, 10, 4,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(702)


In [ ]:
#@title 💡 Solution — Problem 702
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(seed, low, high, n):
    """Seed a generator and return its first n random ints in [low, high)."""
    rng = t.Generator().manual_seed(seed)
    return t.randint(low, high, (n,), generator=rng).tolist()


example = (0, 0, 10, 4,)
print(solve(*example))


<!-- dd:dd-q703 -->

### Problem 703 · independent

Write a function solve(seed, n) that builds a generator from seed and returns a tuple (permutation, shape, dtype_name) for a random permutation of 0..n-1 drawn off that stream.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0, 1, 3, 2], (4,), 'torch.int64')
```


In [ ]:
import torch as t


def solve(seed, n):
    """Seed a generator; return (permutation, shape, dtype name)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (0, 4,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(703)


In [ ]:
#@title 💡 Solution — Problem 703
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(seed, n):
    """Seed a generator; return (permutation, shape, dtype name)."""
    rng = t.Generator().manual_seed(seed)
    p = t.randperm(n, generator=rng)
    return (p.tolist(), tuple(p.shape), str(p.dtype))


example = (0, 4,)
print(solve(*example))


<!-- dd:dd-kp-numpy-random-threading -->

## Using a stream you were handed

`numpy.random-threading`


You can now build a stream and thread it into any sampler. This concept is the
other half, and it is the half that appears in real code far more often: a
function is **handed** an `rng` and has to use *that one*.

Nothing new to spell here — every symbol below came from the previous two
concepts. What is new is a rule:

> Use the generator you were given. Do not make one. Do not reseed it.


In [ ]:
import torch as t

def good(rng, n):
    return t.rand(n, generator=rng)      # threads the caller's stream

def ignores(rng, n):
    return t.rand(n)                     # silently uses the GLOBAL stream

def rewinds(rng, n):
    rng.manual_seed(0)                   # silently rewinds the CALLER's stream
    return t.rand(n, generator=rng)

shared = t.Generator().manual_seed(7)
print("threaded:", good(shared, 3))
print("the caller's stream has now advanced by three draws")




Both wrong versions **run fine and return plausible numbers**. Nothing raises.
That is what makes this worth its own concept: the failure has no symptom at
the call site, only wrong values somewhere downstream.

Why it matters is ownership. A generator is a deterministic stream, and each
draw consumes the next piece of it, so whoever holds the generator controls
what every function that receives it will see:


In [ ]:
import torch as t

def draw(rng, n):
    return t.rand(n, generator=rng)

# The caller seeds once and calls twice. The two results differ because the
# stream MOVED — not because anything is random about `draw` itself.
rng = t.Generator().manual_seed(0)
first = draw(rng, 3)
second = draw(rng, 3)
print("first :", first)
print("second:", second)
assert first.tolist() != second.tolist()

# And because `draw` threaded the stream instead of making its own, the caller
# can reproduce the whole run from the seed alone.
replay = draw(t.Generator().manual_seed(0), 3)
assert replay.tolist() == first.tolist()
print("replay:", replay)




That last assert is the contract a grader checks. A drill that hands you `rng`
has already computed what your draw must be; reseeding or building your own
produces different numbers and fails, with nothing in the error message about
generators.

The generator only reaches the sampler. Everything you build from the draw is
ordinary tensor work:


In [ ]:
import torch as t

rng = t.Generator().manual_seed(1)
order = t.randperm(4, generator=rng)   # random part: needs the stream
perm = t.eye(4)[order]                 # ordinary part: does not
print("order:", order)
print(perm)


> **Watch out.** - **Dropping `generator=` is silent.** `t.rand(n)` runs fine and returns
  plausible numbers — from the wrong stream. Nothing errors; the grader just
  reports different values.
- **Do not reseed a generator you were handed.** The caller owns the stream;
  reseeding it inside your function silently rewinds their sequence.
- **Making your own generator "just to be safe" is the unsafe move.** It is the
  one thing guaranteed to produce numbers the caller cannot reproduce.


Task: draw from a generator you were handed, show that the stream advances, and
show that threading is what makes the result the caller's rather than yours.


In [ ]:
import torch as t

def next_uniforms(rng, n):
    """The next n uniform floats from rng's stream — the caller's stream."""
    return t.rand(n, generator=rng)

# The caller owns the seed, so the caller can predict what comes out.
rng = t.Generator().manual_seed(42)
first = next_uniforms(rng, 3)
print("first draw :", first)

# The same generator, a second call: the stream has MOVED ON.
second = next_uniforms(rng, 3)
print("second draw:", second)
assert first.tolist() != second.tolist()

# Every sampler takes the same keyword, off the same stream, in call order.
ints = t.randint(0, 10, (3,), generator=rng)
perm = t.randperm(4, generator=rng)
print("ints:", ints, " perm:", perm)

# And threading is what makes it reproducible for the caller: a fresh generator
# on the same seed replays the first draw exactly.
replay = next_uniforms(t.Generator().manual_seed(42), 3)
assert replay.tolist() == first.tolist()
print("replayed   :", replay)




Why each step:

1. `next_uniforms` is the shape of every rng-taking function you will write:
   the generator comes in, goes straight to the sampler, and nothing else
   touches it.
2. The second call proves the stream is consumed, which is why a drill that
   hands you `rng` has already computed what your draw must be.
3. The replay at the end is the payoff: because the function threaded the
   generator instead of making one, the caller can reproduce the result.


<!-- dd:dd-q8 -->

### Problem 8 · faded — your turn

The next n uniform floats from a generator you are handed.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.8823, 0.9150, 0.3829])
```


In [ ]:
import torch as t

def solve(rng, n):
    """Return the next n uniform [0,1) floats from rng's stream."""
    return t.rand(n, _____=rng)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example_rng = t.Generator().manual_seed(42)
print(solve(example_rng, 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(8)


In [ ]:
#@title 💡 Solution — Problem 8
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, n):
    return t.rand(n, generator=rng)


example_rng = t.Generator().manual_seed(42)
print(solve(example_rng, 3))


Task: draw from a generator you were handed, show that the stream advances, and
show that threading is what makes the result the caller's rather than yours.


In [ ]:
import torch as t

def next_uniforms(rng, n):
    """The next n uniform floats from rng's stream — the caller's stream."""
    return t.rand(n, generator=rng)

# The caller owns the seed, so the caller can predict what comes out.
rng = t.Generator().manual_seed(42)
first = next_uniforms(rng, 3)
print("first draw :", first)

# The same generator, a second call: the stream has MOVED ON.
second = next_uniforms(rng, 3)
print("second draw:", second)
assert first.tolist() != second.tolist()

# Every sampler takes the same keyword, off the same stream, in call order.
ints = t.randint(0, 10, (3,), generator=rng)
perm = t.randperm(4, generator=rng)
print("ints:", ints, " perm:", perm)

# And threading is what makes it reproducible for the caller: a fresh generator
# on the same seed replays the first draw exactly.
replay = next_uniforms(t.Generator().manual_seed(42), 3)
assert replay.tolist() == first.tolist()
print("replayed   :", replay)




Why each step:

1. `next_uniforms` is the shape of every rng-taking function you will write:
   the generator comes in, goes straight to the sampler, and nothing else
   touches it.
2. The second call proves the stream is consumed, which is why a drill that
   hands you `rng` has already computed what your draw must be.
3. The replay at the end is the payoff: because the function threaded the
   generator instead of making one, the caller can reproduce the result.


<!-- dd:dd-q687 -->

### Problem 687 · faded — your turn

The same move on the normal sampler, returned as a plain list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[1.5409960746765137, -0.293428897857666, -2.1787893772125244]
```


In [ ]:
import torch as t

def solve(rng, n):
    """Return the next n standard-normal floats from rng's stream."""
    return t.randn(n, _____=rng).tolist()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.Generator().manual_seed(0), 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(687)


In [ ]:
#@title 💡 Solution — Problem 687
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, n):
    """Return the next n standard-normal floats from rng's stream."""
    return t.randn(n, generator=rng).tolist()


example = (t.Generator().manual_seed(0), 3,)
print(solve(*example))


<!-- dd:dd-q688 -->

### Problem 688 · independent

Write a function solve(rng, low, high, n) that returns the next n random integers from [low, high) drawn from rng's stream, as a plain list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[4, 9, 3, 0]
```


In [ ]:
import torch as t


def solve(rng, low, high, n):
    """Return the next n random ints in [low, high) from rng's stream."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.Generator().manual_seed(0), 0, 10, 4,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(688)


In [ ]:
#@title 💡 Solution — Problem 688
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, low, high, n):
    """Return the next n random ints in [low, high) from rng's stream."""
    return t.randint(low, high, (n,), generator=rng).tolist()


example = (t.Generator().manual_seed(0), 0, 10, 4,)
print(solve(*example))


<!-- dd:dd-q689 -->

### Problem 689 · independent

Write a function solve(rng, n) that returns a random permutation of 0..n-1 drawn from rng's stream, as a plain list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[0, 1, 3, 2]
```


In [ ]:
import torch as t


def solve(rng, n):
    """Return a random permutation of 0..n-1 from rng's stream."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.Generator().manual_seed(0), 4,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(689)


In [ ]:
#@title 💡 Solution — Problem 689
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, n):
    """Return a random permutation of 0..n-1 from rng's stream."""
    return t.randperm(n, generator=rng).tolist()


example = (t.Generator().manual_seed(0), 4,)
print(solve(*example))


<!-- dd:dd-q690 -->

### Problem 690 · independent

Write a function solve(rng, n) that takes two consecutive draws of n uniform floats from rng's stream and returns a tuple (first, second, they_differ): both draws as plain lists and whether the second draw differs from the first.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0.49625658988952637, 0.7682217955589294], [0.08847743272781372, 0.13203048706054688], True)
```


In [ ]:
import torch as t


def solve(rng, n):
    """Two consecutive draws from one stream; return (first, second, do they differ?)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.Generator().manual_seed(0), 2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(690)


In [ ]:
#@title 💡 Solution — Problem 690
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, n):
    """Two consecutive draws from one stream; return (first, second, do they differ?)."""
    first = t.rand(n, generator=rng)
    second = t.rand(n, generator=rng)
    return (first.tolist(), second.tolist(), first.tolist() != second.tolist())


example = (t.Generator().manual_seed(0), 2,)
print(solve(*example))


<!-- dd:dd-q691 -->

### Problem 691 · independent

Write a function solve(rng, n) that returns the next n uniform floats from rng's stream as a plain list, and does not disturb the stream in any other way.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[0.49625658988952637, 0.7682217955589294, 0.08847743272781372]
```


In [ ]:
import torch as t


def solve(rng, n):
    """Return the next n uniform [0,1) floats from rng's stream."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.Generator().manual_seed(0), 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(691)


In [ ]:
#@title 💡 Solution — Problem 691
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, n):
    """Return the next n uniform [0,1) floats from rng's stream."""
    return t.rand(n, generator=rng).tolist()


example = (t.Generator().manual_seed(0), 3,)
print(solve(*example))


<!-- dd:dd-q692 -->

### Problem 692 · independent

Write a function solve(rng, n) that adds uniform noise to a row of n zeros using rng's stream and returns the noisy row as a plain list. The point is that the generator is threaded through to the draw rather than replaced.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[0.49625658988952637, 0.7682217955589294, 0.08847743272781372]
```


In [ ]:
import torch as t


def solve(rng, n):
    """Add uniform noise from rng to n zeros; return the row."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.Generator().manual_seed(0), 3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(692)


In [ ]:
#@title 💡 Solution — Problem 692
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, n):
    """Add uniform noise from rng to n zeros; return the row."""
    noise = t.rand(n, generator=rng)
    return (t.zeros(n) + noise).tolist()


example = (t.Generator().manual_seed(0), 3,)
print(solve(*example))


<!-- dd:dd-q693 -->

### Problem 693 · independent

Write a function solve(rng, rows) that builds a tensor from the nested list and shuffles its ROWS using a permutation drawn from rng's stream, returning a tuple (shuffled, order): the shuffled rows as a plain nested list and the permutation used, as a plain list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[5, 6], [1, 2], [3, 4]], [2, 0, 1])
```


In [ ]:
import torch as t


def solve(rng, rows):
    """Shuffle a's rows with a permutation from rng; return (shuffled rows, the order)."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.Generator().manual_seed(0), [[1, 2], [3, 4], [5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(693)


In [ ]:
#@title 💡 Solution — Problem 693
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, rows):
    """Shuffle a's rows with a permutation from rng; return (shuffled rows, the order)."""
    a = t.tensor(rows)
    order = t.randperm(a.shape[0], generator=rng)
    return (a[order].tolist(), order.tolist())


example = (t.Generator().manual_seed(0), [[1, 2], [3, 4], [5, 6]],)
print(solve(*example))


<!-- dd:dd-kp-numpy-linalg-basics -->

## Matrix multiply and t.linalg basics

`numpy.linalg-basics`


<!-- dd:dd-seg-numpy-linalg-basics-0 -->

### two multiplications — * vs @


Two different "multiplications" exist for matrices, and PyTorch gives each its
own operator:

- **`a * b` — elementwise**: multiplies corresponding entries; shapes must
  match (or broadcast). No summing happens.
- **`a @ b` — matrix multiplication**: row-times-column with a sum inside.
  For `a` of shape (m, k) and `b` of shape (k, n), the result is (m, n):
  entry `[i, j]` is the dot product of row i of `a` with column j of `b`.
  The inner dimensions (k) must agree, and they disappear in the output.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print("a * b (elementwise)")
print(a * b)
print("a @ b (matrix product)")
print(a @ b)




`a*b` entry [0,0] is 1·5. `a@b` entry [0,0] is 1·5 + 2·7 = 19 — the row met
the column and the k axis was summed away.

The shape rule `(m, k) @ (k, n) → (m, n)` is worth chanting: it predicts
both whether a product is legal and what comes out. It also covers
matrix–vector: `(m, k) @ (k,) → (m,)`.


In [ ]:
print((t.ones((2, 3)) @ t.ones((3, 4))).shape)     # (2,4): the 3s vanish
print((t.ones((2, 3)) @ t.ones(3)).shape)          # (2,): matrix times vector

try:
    t.ones((2, 3)) @ t.ones((2, 3))                # inner dims 3 vs 2
except RuntimeError as err:
    print("RuntimeError:", err)




`t.matmul(a, b)` is the same operation spelled as a function, and in model
code you will meet `a.T` for the transpose that so often precedes it.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
b = t.tensor([[5.0, 6.0],
              [7.0, 8.0]])

# Elementwise vs matrix product — same operands, different operations:
elem = a * b            # [[5, 12], [21, 32]] — corresponding entries
mat = a @ b             # row·column with a sum inside
assert elem.tolist() == [[5.0, 12.0], [21.0, 32.0]]
assert mat.tolist() == [[19.0, 22.0], [43.0, 50.0]]
# Check one entry by hand: mat[0,0] = 1*5 + 2*7 = 19. Row 0 · column 0.
print("a * b")
print(elem)
print("a @ b")
print(mat)

# Shape rule: (2,3) @ (3,2) -> (2,2); the inner 3s must match and vanish.
p = t.ones((2, 3)) @ t.ones((3, 2))
assert p.shape == (2, 2)




Why: computing `mat[0, 0]` by hand once (row 0 of `a` dotted with column 0
of `b`) is the fastest way to internalize what `@` does beyond the shape
rule — and predicting shapes BEFORE running makes mismatches design errors
you catch on paper.


<!-- dd:dd-q239 -->

### Problem 239 · faded — your turn

Matrix product of shapes (m, k) and (k, n).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[19., 22.],
        [43., 50.]])
```


In [ ]:
import torch as t

def solve(a, b):
    """The (m, n) matrix product of a (m, k) and b (k, n)."""
    return a _____ b


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(a, b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(239)


In [ ]:
#@title 💡 Solution — Problem 239
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return a @ b


a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(a, b))


<!-- dd:dd-seg-numpy-linalg-basics-1 -->

### t.linalg.solve — never build the inverse


The `t.linalg` submodule holds the "real linear algebra", and its names match
NumPy's `np.linalg` almost one for one:

- **`t.linalg.solve(a, b)`** — solve the system `a @ x = b` for `x`.
  This is THE way to compute "a⁻¹ b". Numerically, solving directly is both
  faster and more accurate than `t.linalg.inv(a) @ b`; computing an
  explicit inverse is almost never what you want.
- `t.linalg.inv`, `t.linalg.det`, `t.linalg.matrix_rank`,
  `t.linalg.norm`, `t.linalg.eig` — inverse, determinant, rank, norms,
  eigendecomposition, when a task genuinely asks for them.

One dtype caveat that is easy to trip over here: these routines want floats,
and the default float is 32-bit. Ill-conditioned systems lose accuracy sooner
than the float64 you may be used to from NumPy — if a solve looks wrong,
checking the dtype is a reasonable first move.


In [ ]:
import torch as t

a_sys = t.tensor([[3.0, 1.0],
                  [1.0, 2.0]])
b_vec = t.tensor([9.0, 8.0])
x = t.linalg.solve(a_sys, b_vec)
print("x =", x)




Sanity-checking a solve is one line: plug `x` back in and compare
`a @ x` with `b` using `t.allclose` (float arithmetic — never `==`).


In [ ]:
print("a @ x =", a_sys @ x, " b =", b_vec)
print("exactly equal? ", bool(t.equal(a_sys @ x, b_vec)))
print("close enough?  ", bool(t.allclose(a_sys @ x, b_vec)))
assert t.allclose(a_sys @ x, b_vec)




The inverse route reaches the same answer and does more work to get there —
run it once so the equivalence is concrete, then stop writing it:


In [ ]:
via_inverse = t.linalg.inv(a_sys) @ b_vec
print("solve:  ", x)
print("inverse:", via_inverse)
assert t.allclose(x, via_inverse)


In [ ]:
import torch as t

# Solve a @ x = b_vec — NOT by computing an inverse.
a_sys = t.tensor([[2.0, 0.0],
                  [0.0, 4.0]])
b_vec = t.tensor([6.0, 8.0])
x = t.linalg.solve(a_sys, b_vec)
assert x.tolist() == [3.0, 2.0]
print("x =", x)

# Verification pattern: substitute back, compare with float tolerance.
assert t.allclose(a_sys @ x, b_vec)
print("a @ x =", a_sys @ x, " b =", b_vec)




Why: `solve` + `allclose` verification — the pair costs one line and
catches both wrong answers and ill-conditioned systems.


<!-- dd:dd-q107 -->

### Problem 107 · faded — your turn

Solve the linear system a @ x = b (a is invertible).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([3., 2.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return x such that a @ x = b (use a solver, not an inverse)."""
    return t._____._____(a, b)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example_a = t.tensor([[2.0, 0.0], [0.0, 4.0]])
example_b = t.tensor([6.0, 8.0])
print(solve(example_a, example_b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(107)


In [ ]:
#@title 💡 Solution — Problem 107
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.linalg.solve(a, b)


example_a = t.tensor([[2.0, 0.0], [0.0, 4.0]])
example_b = t.tensor([6.0, 8.0])
print(solve(example_a, example_b))


<!-- dd:dd-q508 -->

### Problem 508 · guided

Write a function solve(a, b) that takes two float matrices of the same shape and returns their elementwise product — entry [i][j] is a[i][j] * b[i][j]. This is the multiplication that is NOT matrix multiplication, and telling the two apart is the whole point.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 5., 12.],
        [21., 32.]])
```


<details>
<summary>Hints</summary>

1. This is the multiplication that is NOT matrix multiplication.
2. Entry [i][j] depends only on the two entries at [i][j] — nothing is summed,
   so no axis is contracted.
3. `a * b`.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return the ELEMENTWISE product of two same-shaped matrices."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(508)


In [ ]:
#@title 💡 Solution — Problem 508
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return the ELEMENTWISE product of two same-shaped matrices."""
    return a * b


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]]))
print(solve(*example))


<!-- dd:dd-q509 -->

### Problem 509 · guided

Write a function solve(a, b) that takes two square float matrices and returns a tuple (elementwise, matmul) of plain nested lists: first a * b, then a @ b. Run them side by side once and the difference stops being something you have to remember.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[5.0, 12.0], [21.0, 32.0]], [[19.0, 22.0], [43.0, 50.0]])
```


<details>
<summary>Hints</summary>

1. Both answers come from the same two matrices; only the operator changes.
2. `*` pairs entries in place; `@` contracts a's columns against b's rows.
3. `((a * b).tolist(), (a @ b).tolist())`.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return (elementwise product, matrix product) for two square matrices."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(509)


In [ ]:
#@title 💡 Solution — Problem 509
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return (elementwise product, matrix product) for two square matrices."""
    return ((a * b).tolist(), (a @ b).tolist())


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]]))
print(solve(*example))


<!-- dd:dd-q510 -->

### Problem 510 · independent

Write a function solve(a) that returns a tuple (values, shape) for the transpose of the 2-D tensor a. A transpose only re-describes the block — the numbers never move — so the shape comes back with its two axes swapped.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[1.0, 4.0], [2.0, 5.0], [3.0, 6.0]], (3, 2))
```


In [ ]:
import torch as t

def solve(a):
    """Return the transpose of a 2-D tensor, and its shape."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(510)


In [ ]:
#@title 💡 Solution — Problem 510
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    """Return the transpose of a 2-D tensor, and its shape."""
    tr = a.T
    return (tr.tolist(), tuple(tr.shape))


example = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(solve(example))


<!-- dd:dd-q511 -->

### Problem 511 · independent

Write a function solve(a, x) where a has shape (m, n) and x has shape (n,), returning a tuple (values, shape) for the matrix-vector product. The shared axis vanishes, so an (m, n) against an (n,) leaves you with (m,) — one number per row.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([3.0, 7.0], (2,))
```


In [ ]:
import torch as t

def solve(a, x):
    """Return a @ x for a matrix and a vector, plus the result shape."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([1.0, 1.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(511)


In [ ]:
#@title 💡 Solution — Problem 511
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, x):
    """Return a @ x for a matrix and a vector, plus the result shape."""
    r = a @ x
    return (r.tolist(), tuple(r.shape))


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([1.0, 1.0]))
print(solve(*example))


<!-- dd:dd-q512 -->

### Problem 512 · independent

Write a function solve(a, b) that solves the linear system a @ x = b for x, and returns a tuple (x_rounded, checks_out): x's entries rounded to 4 decimal places as a plain list, and a plain bool saying whether a @ x really does reproduce b. Solve the system directly — never build the inverse and multiply, which is slower and less accurate for the same answer.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([3.0, 2.0], True)
```


In [ ]:
import torch as t

def solve(a, b):
    """Solve a @ x = b, and confirm the solution satisfies it."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[2.0, 0.0], [0.0, 4.0]]), t.tensor([6.0, 8.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(512)


In [ ]:
#@title 💡 Solution — Problem 512
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Solve a @ x = b, and confirm the solution satisfies it."""
    x = t.linalg.solve(a, b)
    return ([round(v, 4) for v in x.tolist()], bool(t.allclose(a @ x, b, atol=1e-4)))


example = (t.tensor([[2.0, 0.0], [0.0, 4.0]]), t.tensor([6.0, 8.0]))
print(solve(*example))


<!-- dd:dd-q513 -->

### Problem 513 · independent

Write a function solve(mats, x) where mats has shape (batch, m, n) and x has shape (n,), returning a tuple (values, shape) for the result of applying every matrix in the batch to x. @ treats all but the last two axes as batch dimensions, so this is one operator, not a loop.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[3.0, 4.0], [6.0, 8.0]], (2, 2))
```


In [ ]:
import torch as t

def solve(mats, x):
    """Apply a BATCH of matrices to one vector."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[[1.0, 0.0], [0.0, 1.0]], [[2.0, 0.0], [0.0, 2.0]]]), t.tensor([3.0, 4.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(513)


In [ ]:
#@title 💡 Solution — Problem 513
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(mats, x):
    """Apply a BATCH of matrices to one vector."""
    r = mats @ x
    return (r.tolist(), tuple(r.shape))


example = (t.tensor([[[1.0, 0.0], [0.0, 1.0]], [[2.0, 0.0], [0.0, 2.0]]]), t.tensor([3.0, 4.0]))
print(solve(*example))


#### Common mistakes

- **"`*` multiplies matrices."** — `*` is elementwise; `@` is the matrix
  product. Mixing them up usually *doesn't* crash (broadcasting can make `*`
  legal), it just silently computes the wrong thing — the worst kind of bug.
- **"To solve a @ x = b, compute inv(a) @ b."** — `t.linalg.solve(a, b)` is
  more accurate and faster; explicit inverses amplify rounding error and cost
  more. Reach for `inv` only when the inverse itself is the deliverable.
- **"If `@` runs, the shapes were right."** — `@` between wrong-but-compatible
  shapes (e.g. transposed operands, square matrices) runs happily and returns
  garbage. Predict `(m, k) @ (k, n) → (m, n)` on paper first.
- **"Integer tensors are fine for linalg."** — They are not; the solvers
  require floating point and will raise. Convert with `.to(t.float32)` first.
